# Phase 3 — Python Internals: Memory, Bytecode and Imports

**Outcome:** explain where an object's memory comes from and when it is freed, read
bytecode fluently, and describe exactly what happens between `import x` and the
first line of your code running.

| Unit | Topic |
| --- | --- |
| 3.1 | Memory management |
| 3.2 | Bytecode and the interpreter |
| 3.3 | Modules, packages and the import system |
| Project | Internals toolkit |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- This phase looks at **CPython implementation details**. Object sizes, reference
  counts, addresses and `dis` listings change between Python versions (and some
  between operating systems). Cells that print them also print the interpreter
  version; your numbers may differ, the ideas do not.
- Nothing here asserts an exact size, address or reference count. Checks compare
  *relationships* (bigger than, freed or not, same object or not).
- Cells that need a special build (free-threaded, JIT) check first and print a skip
  message when the feature is missing.
- *Predict the output* questions hide the answer in a collapsible block: decide
  first, then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub, then run the check
  cell. Until then the checks print FAIL; that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.14*, …).

## What you need from Phases 1 and 2

Phase 3 opens the box behind ideas you have already used. If one feels shaky,
re-read the unit before going on.

| Earlier idea | Unit | What Phase 3 adds |
| --- | --- | --- |
| Source → tokens → AST → code object → eval loop | 1.1 | Each stage in detail: code-object fields, `dis` columns, the specializing interpreter and the JIT |
| Names are references; `is` vs `==`; the small-int cache | 1.2 | References are counted; some objects are immortal; the cyclic GC handles what counting cannot |
| `sys.getsizeof`, PEP 393 strings, list over-allocation | 1.4–1.5 | The object header, the GC header, pymalloc pools and free lists |
| `dict` internals and key-sharing | 1.6.1 | Measuring memory with `tracemalloc` snapshots and diffs |
| Closures and cells (`co_freevars`, `co_cellvars`) | 1.11 | Code objects and function objects as data you can inspect and rebuild |
| `functools.lru_cache`, decorators | 1.12 | Why `lru_cache` on a method leaks |
| Generators and frames | 1.13 | Frames, the instruction pointer and inlined calls |
| Exceptions, tracebacks, context managers | 1.14 | Stored tracebacks keep frames alive; context managers beat `__del__` |
| `__slots__`, descriptors, `__weakref__` | 2.5 | The `__weakref__` slot and what weak references are for |

> **Note:** you will see `sys._is_immortal`, `sys._jit`, `sys._is_gil_enabled` and
> `sys._debugmallocstats`. A leading underscore in `sys` means *CPython-specific
> and may change*; use them to learn and debug, not in production code.

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 3.1 — Memory Management

**Objectives**

- Picture what a running program looks like in memory: frames that hold
  references, objects on a private heap, every object starting with the same
  header.
- Explain when CPython frees an object: reference counting first, the cyclic
  garbage collector for cycles, immortal objects never.
- Use `sys.getrefcount`, `gc`, `weakref` and `tracemalloc` as instruments, and read
  their output without trusting exact numbers.
- Know how the allocator (pymalloc, mimalloc) and free lists shape memory use, and
  why the process's RSS rarely shrinks.
- Find and fix the six common kinds of leak with a repeatable workflow.

**JS bridge**

| JavaScript (V8) | Python (CPython) | The difference that bites |
| --- | --- | --- |
| Tracing generational GC (Orinoco: scavenger + mark-compact) | Reference counting + a cycle collector for containers | CPython frees most objects **the moment** the last reference goes, not "some time later" |
| Objects can move (compaction) | Objects never move; `id()` is the address | Fragmentation: freed memory can stay inside the process |
| `WeakRef`, `FinalizationRegistry` | `weakref.ref`, `weakref.finalize` | CPython runs finalizers deterministically when the count hits zero (other Pythons may not) |
| `WeakMap`, `WeakSet` | `WeakKeyDictionary`, `WeakValueDictionary`, `WeakSet` | Python also has weak **values**; many built-in types cannot be weakly referenced |
| Heap snapshots in DevTools | `tracemalloc` snapshots and diffs | Both group allocations by where they were made |
| `process.memoryUsage().rss` | `/proc/self/status`, `psutil`, `resource` | RSS includes allocator slack in both |

> **Note:** sizes, reference counts and addresses printed in this unit come from the
> interpreter running the notebook. Every cell that prints them also prints the
> version and build; your values may differ.

The next cell is this unit's setup. Besides the usual error helpers it defines
`interpreter_label()`, `rss_bytes()`, `source_line(frame)` (source text of a
`tracemalloc` frame) and `find_free_threaded_python()`.

In [2]:
import ctypes
import dis
import functools
import gc
import importlib.util
import itertools
import linecache
import os
import platform
import shutil
import struct
import subprocess
import sys
import sysconfig
import tempfile
import textwrap
import traceback
import tracemalloc
import weakref
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from pathlib import Path

CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0"}  # plain output from child interpreters


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to any size or refcount."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"


def rss_bytes() -> int | None:
    """Resident set size of this process in bytes, or None when it cannot be read."""
    status_file = Path("/proc/self/status")  # Linux
    if status_file.exists():
        for line in status_file.read_text().splitlines():
            if line.startswith("VmRSS:"):
                return int(line.split()[1]) * 1024
    if importlib.util.find_spec("psutil") is not None:  # macOS / Windows fallback
        import psutil

        return psutil.Process().memory_info().rss
    return None


def source_line(frame: tracemalloc.Frame) -> str:
    """The source text of a tracemalloc frame (works for notebook cells too)."""
    return linecache.getline(frame.filename, frame.lineno).strip() or "<source unavailable>"


def find_free_threaded_python() -> str | None:
    """Path of an installed free-threaded 3.15 interpreter, or None."""
    candidates: list[str] = []
    if uv_path := shutil.which("uv"):
        found = subprocess.run(
            [uv_path, "python", "find", "3.15t"], capture_output=True, text=True, timeout=60
        )
        if found.returncode == 0:
            candidates.append(found.stdout.strip())
    if on_path := shutil.which("python3.15t"):
        candidates.append(on_path)
    probe = "import sysconfig; print(sysconfig.get_config_var('Py_GIL_DISABLED'))"
    for candidate in candidates:
        answer = subprocess.run([candidate, "-c", probe], capture_output=True, text=True)
        if answer.stdout.strip() == "1":
            return candidate
    return None


print(interpreter_label())

CPython 3.15.1, default (GIL) build, x86_64


### 3.1.1 The memory model

Every Python value is an **object on CPython's private heap**. Nothing lives
"on the stack" the way a C `int` or a JS small integer (Smi) does. What the call
stack holds are **frames**, one per running function call, and a frame's local
variables are just **references**: pointer-sized slots that point at heap objects.

```mermaid
flowchart LR
    subgraph stack["Call stack (frames)"]
        F1["checkout()<br/>cart_items, tax_rate"]
        F2["apply_tax()<br/>cart_items, tax_rate"]
    end
    subgraph heap["Private heap (objects)"]
        L["list [120.0, 80.0]"]
        T["float 0.18"]
    end
    F1 -- "cart_items" --> L
    F2 -- "cart_items" --> L
    F1 -- "tax_rate" --> T
    F2 -- "tax_rate" --> T
```

<details><summary>ASCII fallback</summary>

```text
CALL STACK (frames)                 PRIVATE HEAP (objects)
+---------------------------+
| apply_tax()               |
|   cart_items  ------------+-----> list [120.0, 80.0]
|   tax_rate    ------------+--+
+---------------------------+  |
| checkout()                |  +--> float 0.18
|   cart_items  ------------+-----> (same list)
|   tax_rate    ------------+-----> (same float)
+---------------------------+
```

</details>

Since 3.11 the frames themselves are lightweight C structs on a per-thread data
stack; a full `frame` object is only built when someone asks for it
(`sys._getframe()`, a traceback, a debugger). The cell asks for both frames and
shows that they reference **one** list.

In [3]:
def checkout(cart_items: list[float]) -> float:
    """Outer call: owns the list and passes a reference down."""
    tax_rate = 0.18
    return apply_tax(cart_items, tax_rate)


def apply_tax(cart_items: list[float], tax_rate: float) -> float:
    """Inner call: its frame holds a second reference to the caller's list."""
    caller_frame = sys._getframe(1)
    for frame in (sys._getframe(), caller_frame):
        local_names = sorted(set(frame.f_locals) - {"caller_frame", "frame"})
        print(f"frame {frame.f_code.co_name:<9} local names: {local_names}")
    same_list = caller_frame.f_locals["cart_items"] is cart_items
    print(f"both frames reference one heap object: {same_list}")
    return round(sum(cart_items) * (1 + tax_rate), 2)


print(f"{checkout([120.0, 80.0])=}")

frame apply_tax local names: ['cart_items', 'tax_rate']
frame checkout  local names: ['cart_items', 'tax_rate']
both frames reference one heap object: True
checkout([120.0, 80.0])=236.0


**References are pointer-sized.** A list stores references, not values, so each
slot costs one pointer (8 bytes on a 64-bit build) whether it points at a small
int or a megabyte string. A thousand references to one big string cost about 8 kB,
not a gigabyte.

In [4]:
pointer_bytes = struct.calcsize("P")
print(f"{pointer_bytes=}  ({interpreter_label()})")

big_text = "x" * 1_000_000
list_of_big = [big_text] * 1_000
list_of_small = [7] * 1_000
print(f"{sys.getsizeof(big_text)=:,}")
print(f"{sys.getsizeof(list_of_big)=:,}  {sys.getsizeof(list_of_small)=:,}")
per_slot = (sys.getsizeof(list_of_big) - sys.getsizeof([])) / len(list_of_big)
print(f"bytes per list slot: {per_slot}")
print(f"{all(item is big_text for item in list_of_big)=}  (1 000 references, 1 string)")
del big_text, list_of_big, list_of_small

pointer_bytes=8  (CPython 3.15.1, default (GIL) build, x86_64)
sys.getsizeof(big_text)=1,000,041
sys.getsizeof(list_of_big)=8,056  sys.getsizeof(list_of_small)=8,056
bytes per list slot: 8.0
all(item is big_text for item in list_of_big)=True  (1 000 references, 1 string)


### 3.1.2 Object layout

Every object starts with the same C header, so the interpreter can handle any
object through a `PyObject *` pointer:

```c
typedef struct _object {
    Py_ssize_t ob_refcnt;     /* reference count (a union with split fields since 3.12) */
    PyTypeObject *ob_type;    /* pointer to the type object: list, int, your class */
} PyObject;

typedef struct {
    PyObject ob_base;
    Py_ssize_t ob_size;       /* number of items in variable-size objects */
} PyVarObject;
```

Container types that can form reference cycles (lists, dicts, sets, most class
instances) carry an extra **GC header** (`PyGC_Head`, two words) placed **before**
the object, which links them into the garbage collector's lists. Instances of
ordinary classes also have pre-header slots for the managed `__dict__` and
weak-reference list (3.12+).

```text
           +----------------------------+
  -16 B    | PyGC_Head (2 words)        |  only for GC-tracked container types
           +----------------------------+  <-- id(obj) points here (CPython)
   +0      | ob_refcnt                  |
   +8      | ob_type  ------------------+--> the type object
  +16      | ob_size (variable-size)    |  list, tuple, int digits, bytes, ...
  +24 ...  | type-specific fields       |  e.g. list: pointer to the item array
           +----------------------------+
```

| Piece | Who has it | Where you can see it |
| --- | --- | --- |
| `ob_refcnt` | every object | `sys.getrefcount(obj)` (plus one) |
| `ob_type` | every object | `type(obj)` |
| `ob_size` | variable-size objects | `len(obj)` for list / tuple / bytes |
| GC header | GC-tracked types | `sys.getsizeof` includes it; `gc.is_tracked(obj)` |
| Fixed part | every type | `type.__basicsize__` |
| Per-item part | variable-size types | `type.__itemsize__` |

> **Note:** the free-threaded build (3.1.5) has a larger header: an owning-thread
> id, two reference-count fields and a small lock.

The first cell prints the fixed and per-item sizes of common types and whether the
type is GC-capable (the `Py_TPFLAGS_HAVE_GC` bit, `1 << 14`, in `__flags__`).

In [5]:
PY_TPFLAGS_HAVE_GC = 1 << 14

print(interpreter_label())
print(f"{'type':<8}{'__basicsize__':>14}{'__itemsize__':>13}  GC-capable type")
for some_type in (object, float, int, str, bytes, tuple, list, dict, set):
    gc_capable = bool(some_type.__flags__ & PY_TPFLAGS_HAVE_GC)
    print(
        f"{some_type.__name__:<8}{some_type.__basicsize__:>14}"
        f"{some_type.__itemsize__:>13}  {gc_capable}"
    )

CPython 3.15.1, default (GIL) build, x86_64
type     __basicsize__ __itemsize__  GC-capable type
object              16            0  False
float               24            0  False
int                 24            4  False
str                 64            0  False
bytes               33            1  False
tuple               32            8  True
list                40            0  True
dict                48            0  True
set                200            0  True


**The GC header shows up in `sys.getsizeof`.** For a GC-capable type,
`sys.getsizeof` adds the header to the object's own size, so an empty list reports
`list.__basicsize__ + 16` on a 64-bit build. A `float` is not a container and has no
GC header.

In [6]:
print(interpreter_label())
samples = {"float": 1.5, "empty list": [], "empty tuple": (), "empty dict": {}}
for label, sample in samples.items():
    basic_size = type(sample).__basicsize__
    extra_bytes = sys.getsizeof(sample) - basic_size
    print(
        f"{label:<12} getsizeof={sys.getsizeof(sample):>3}  "
        f"__basicsize__={basic_size:>3}  extra (GC header)={extra_bytes:>3}"
    )

CPython 3.15.1, default (GIL) build, x86_64
float        getsizeof= 24  __basicsize__= 24  extra (GC header)=  0
empty list   getsizeof= 56  __basicsize__= 40  extra (GC header)= 16
empty tuple  getsizeof= 48  __basicsize__= 32  extra (GC header)= 16
empty dict   getsizeof= 64  __basicsize__= 48  extra (GC header)= 16


**Reading the header directly (unsafe).** `ctypes` can read raw memory at
`id(obj)`. This is **CPython-only, 64-bit-only and unsafe**: a wrong address
crashes the kernel, and the layout is private and changes between versions and
builds. The cell checks the build first and only reads live objects it holds a
reference to. It confirms `ob_type` points at `list` and `ob_size` equals `len()`.

In [7]:
CAN_PEEK = (
    sys.implementation.name == "cpython"
    and struct.calcsize("P") == 8
    and not sysconfig.get_config_var("Py_GIL_DISABLED")
)


def read_word(address: int) -> int:
    """UNSAFE: read one signed machine word at address (64-bit CPython default build)."""
    return ctypes.c_ssize_t.from_address(address).value


if CAN_PEEK:
    basket = ["tea", "rice", "salt"]
    print(f"ob_type is list: {read_word(id(basket) + 8) == id(list)}")
    print(f"ob_size: {read_word(id(basket) + 16)}  (len: {len(basket)})")
    basket.append("sugar")
    print(f"ob_size after append: {read_word(id(basket) + 16)}")
else:
    print(f"Skipped: offsets assume 64-bit CPython, default build ({interpreter_label()})")

ob_type is list: True
ob_size: 3  (len: 3)
ob_size after append: 4


**Size table for common objects.** The cell prints `sys.getsizeof` for everyday
values. What explains each family:

| Family | Size is made of |
| --- | --- |
| `None`, `bool`, small objects | the 16-byte header, maybe one more field |
| `int` | header + `ob_size` + 30-bit digits (4 bytes each) |
| `float` | header + one C `double` |
| `str` | header + PEP 393 fields + 1, 2 or 4 bytes per character (widest character decides) |
| `bytes` | header + length + hash + one byte per item + a trailing NUL |
| `tuple` | header + length + cached hash (3.14+) + 8 bytes per item (+ GC header) |
| `list`, `dict`, `set` | header + bookkeeping + GC header; items live in a **separate** array that `getsizeof` adds on top (see 3.1.11) |
| class instance | header + inline values / managed `__dict__` pointers (+ GC header) |

> **Warning:** these numbers change between Python versions (3.14 added a cached
> hash to tuples, for example). Use them to compare, never hard-code them.

In [8]:
class PlainRecord:
    """An ordinary class with two attributes."""

    def __init__(self) -> None:
        self.sku = "TEA-01"
        self.quantity = 3


def sample_function() -> None:
    """A function object to measure."""


common_objects: dict[str, object] = {
    "None": None, "True": True, "0": 0, "2**30": 2**30, "2**100": 2**100,
    "1.5": 1.5, "1+2j": 1 + 2j, "'' (empty str)": "", "'a'": "a",
    "'é' (Latin-1)": "é", "'₹' (BMP)": "₹", "'😀' (astral)": "😀",
    "'a' * 100": "a" * 100, "b''": b"", "b'a' * 100": b"a" * 100,
    "()": (), "(1,)": (1,), "(1, 2, 3)": (1, 2, 3), "[]": [], "[1, 2, 3]": [1, 2, 3],
    "{}": {}, "{'a': 1}": {"a": 1}, "set()": set(), "frozenset()": frozenset(),
    "range(10)": range(10), "object()": object(), "PlainRecord()": PlainRecord(),
    "function": sample_function,
}  # fmt: skip
print(interpreter_label())
for label, sample in common_objects.items():
    print(f"{label:<16}{sys.getsizeof(sample):>6} B   {type(sample).__name__}")

CPython 3.15.1, default (GIL) build, x86_64
None                16 B   NoneType
True                28 B   bool
0                   28 B   int
2**30               32 B   int
2**100              40 B   int
1.5                 24 B   float
1+2j                32 B   complex
'' (empty str)      41 B   str
'a'                 42 B   str
'é' (Latin-1)       61 B   str
'₹' (BMP)           60 B   str
'😀' (astral)        64 B   str
'a' * 100          141 B   str
b''                 33 B   bytes
b'a' * 100         133 B   bytes
()                  48 B   tuple
(1,)                56 B   tuple
(1, 2, 3)           72 B   tuple
[]                  56 B   list
[1, 2, 3]           88 B   list
{}                  64 B   dict
{'a': 1}           184 B   dict
set()              216 B   set
frozenset()        216 B   frozenset
range(10)           48 B   range
object()            16 B   object
PlainRecord()       48 B   PlainRecord
function           168 B   function


### 3.1.3 Reference counting

Every object counts how many references point at it. When the count drops to
zero, CPython frees the object **immediately** (and decrements everything it
referenced, which can cascade).

| Increments the count | Decrements the count |
| --- | --- |
| binding a name: `alias = obj` | rebinding or `del` of that name |
| storing in a container: `items.append(obj)`, `{key: obj}` | removing it, or the container dying |
| storing as an attribute: `holder.inner = obj` | `del holder.inner`, or `holder` dying |
| pushing it on the eval stack for an operation (usually) | popping it when the operation ends |
| a frame holding it as a local or argument | the frame finishing (return or exception) |

`sys.getrefcount(obj)` reports the count **plus one**: the call's own argument is
a temporary reference. The cell watches one object at module level and prints the
change after each step.

In [9]:
class Ticket:
    """A plain object whose reference count we watch."""


ticket = Ticket()
baseline = sys.getrefcount(ticket)
print(f"{baseline=}  (the name `ticket` + getrefcount's own argument)  [{interpreter_label()}]")

alias = ticket
print(f"alias = ticket            -> {sys.getrefcount(ticket) - baseline:+d}")
queue = [ticket]
print(f"queue = [ticket]          -> {sys.getrefcount(ticket) - baseline:+d}")
by_code = {"T1": ticket}
print(f"by_code = {{'T1': ticket}}  -> {sys.getrefcount(ticket) - baseline:+d}")
holder = Ticket()
holder.inner = ticket
print(f"holder.inner = ticket     -> {sys.getrefcount(ticket) - baseline:+d}")
del alias
print(f"del alias                 -> {sys.getrefcount(ticket) - baseline:+d}")
queue.clear()
print(f"queue.clear()             -> {sys.getrefcount(ticket) - baseline:+d}")
del by_code, holder
print(f"del by_code, holder       -> {sys.getrefcount(ticket) - baseline:+d}")

baseline=2  (the name `ticket` + getrefcount's own argument)  [CPython 3.15.1, default (GIL) build, x86_64]
alias = ticket            -> +1
queue = [ticket]          -> +2
by_code = {'T1': ticket}  -> +3
holder.inner = ticket     -> +4
del alias                 -> +3
queue.clear()             -> +2
del by_code, holder       -> +0


**The "+1" is not guaranteed.** *Changed in 3.14:* inside functions the compiler
emits `LOAD_FAST_BORROW`, which pushes a **borrowed** reference (no increment)
when it can prove the local outlives the use. `sys.getrefcount(local)` can then
print `1` for an object with one real reference, and passing an object as an
argument may add nothing. Treat `getrefcount` as an approximate instrument; compare
**changes**, never absolute values.

In [10]:
def local_refcount() -> int:
    """getrefcount of an object referenced only by this function's local."""
    fresh_ticket = Ticket()
    return sys.getrefcount(fresh_ticket)


def argument_refcount(some_object: object) -> int:
    """getrefcount of an argument, seen from inside the callee."""
    return sys.getrefcount(some_object)


print(f"{sys.getrefcount(ticket)=}  (module level)")
print(f"{argument_refcount(ticket)=}  (same object, seen as an argument)")
print(f"{local_refcount()=}  (one local reference)")
load_ops = [ins.opname for ins in dis.get_instructions(local_refcount)]
print(f"loads in local_refcount: {[name for name in load_ops if name.startswith('LOAD_FAST')]}")
print(interpreter_label())

sys.getrefcount(ticket)=2  (module level)
argument_refcount(ticket)=2  (same object, seen as an argument)
local_refcount()=1  (one local reference)
loads in local_refcount: ['LOAD_FAST_BORROW']
CPython 3.15.1, default (GIL) build, x86_64


**Deterministic deallocation.** Because freeing happens when the count reaches
zero, you can predict *exactly* when an object dies. `weakref.finalize(obj, func,
*args)` registers a callback that runs when `obj` is freed, without keeping `obj`
alive. The event list shows the finalizer running **between** the two statements
around `del`, and at the moment a function returns.

In [11]:
events: list[str] = []
receipt = Ticket()
weakref.finalize(receipt, events.append, "receipt freed")
events.append("before del")
del receipt
events.append("after del")
print(events)


def issue_temporary_ticket() -> None:
    """Create a ticket that only this frame references."""
    temporary = Ticket()
    weakref.finalize(temporary, events.append, "temporary freed (frame finished)")
    events.append("returning")


events.clear()
issue_temporary_ticket()
events.append("caller continues")
print(events)

['before del', 'receipt freed', 'after del']
['returning', 'temporary freed (frame finished)', 'caller continues']


> **Note:** this is a CPython guarantee, not a language guarantee. PyPy and GraalPy
> use tracing collectors, so finalizers there run "later". Code that must release a
> resource at a known point should use a `with` block (3.1.7).

**Reading the live count with `ctypes` (unsafe).** `ob_refcnt` is the first word of
the object. Reading it directly avoids the extra argument reference. Same
warnings as in 3.1.2: CPython-only, and never read an address after the object
might have been freed.

In [12]:
if CAN_PEEK:
    watched = Ticket()
    watched_address = id(watched)  # valid only while `watched` is alive
    # read_word (3.1.2) is ctypes.c_ssize_t.from_address(address).value
    print(f"one name:           ob_refcnt={read_word(watched_address)}")
    three_more = [watched, watched, watched]
    print(f"+ list of 3 refs:   ob_refcnt={read_word(watched_address)}")
    del three_more
    print(f"list deleted:       ob_refcnt={read_word(watched_address)}")
    print(f"compare: sys.getrefcount(watched)={sys.getrefcount(watched)}")
    del watched  # from here on, watched_address must not be read
else:
    print(f"Skipped: needs 64-bit CPython, default build ({interpreter_label()})")

one name:           ob_refcnt=1
+ list of 3 refs:   ob_refcnt=4
list deleted:       ob_refcnt=1
compare: sys.getrefcount(watched)=2


### 3.1.4 Immortal objects (PEP 683)

*New in 3.12.* Some objects are **immortal**: their reference count is set to a
special value, `Py_INCREF` / `Py_DECREF` leave it alone, and they are never freed.
Immortal objects include `None`, `True`, `False`, `Ellipsis`, the small-int cache,
the empty tuple, the empty string and one-character Latin-1 strings, the strings
CPython preallocates for its own identifiers, and the built-in types.

Why bother:

- **No writes to shared objects.** Incrementing `None`'s count on every use wrote to
  one hot cache line from every thread and dirtied memory pages shared after
  `fork()` (copy-on-write).
- **A prerequisite** for the per-interpreter GIL (3.12) and the free-threaded build
  (3.1.5).

`sys._is_immortal(obj)` (*New in 3.14*, CPython-specific) answers directly. The cell
checks a mix of objects; which strings are immortal depends on how they were
created.

In [13]:
if hasattr(sys, "_is_immortal"):
    runtime_text = "".join(["warehouse", "_zone"])  # built at run time
    immortality_samples: dict[str, object] = {
        "None": None, "True": True, "Ellipsis": ..., "int (the type)": int,
        "-5": -5, "256": int("256"), "1024": int("1024"), "1025": int("1025"),
        "'' (empty str)": "", "'a' (1 char)": "a", "'order_id' (literal)": "order_id",
        "str built at run time": runtime_text,
        "sys.intern(that str)": sys.intern(runtime_text),
        "() (empty tuple)": (), "[] (new list)": [], "object()": object(),
    }  # fmt: skip
    print(interpreter_label())
    for label, sample in immortality_samples.items():
        print(f"{label:<24} immortal={sys._is_immortal(sample)}")
else:
    print("Skipped: sys._is_immortal needs Python 3.14+")

CPython 3.15.1, default (GIL) build, x86_64
None                     immortal=True
True                     immortal=True
Ellipsis                 immortal=True
int (the type)           immortal=True
-5                       immortal=True
256                      immortal=True
1024                     immortal=True
1025                     immortal=False
'' (empty str)           immortal=True
'a' (1 char)             immortal=True
'order_id' (literal)     immortal=False
str built at run time    immortal=False
sys.intern(that str)     immortal=False
() (empty tuple)         immortal=True
[] (new list)            immortal=False
object()                 immortal=False


Notice that on this build the small-int cache reaches 1024 (*Changed in 3.15*;
it ended at 256 before). A string literal from **your** code (`'order_id'`) and a
string interned at run time with `sys.intern` are interned (deduplicated) but
**not** immortal: they are freed when the code or the last user goes away.

Making a hundred thousand new references to `None` does not change its reported
count. The huge value is the immortal marker, not a real count.

In [14]:
none_count_before = sys.getrefcount(None)
many_nones = [None] * 100_000
none_count_after = sys.getrefcount(None)
print(f"{none_count_before=:,}")
print(f"{none_count_after=:,}")
print(f"unchanged: {none_count_before == none_count_after}  [{interpreter_label()}]")
del many_nones

none_count_before=3,221,225,472
none_count_after=3,221,225,472
unchanged: True  [CPython 3.15.1, default (GIL) build, x86_64]


### 3.1.5 Reference counting in the free-threaded build

The free-threaded build (PEP 703; experimental in 3.13, officially supported since
3.14 under PEP 779) removes the GIL, so threads really run Python code in parallel.
Plain `ob_refcnt += 1` would then be a data race, and making every increment atomic
would be slow. Two techniques keep it fast (overview):

| Technique | Idea | Effect |
| --- | --- | --- |
| **Biased reference counting** | Each object records an **owning thread** (`ob_tid`). The owner updates a local count (`ob_ref_local`) with plain, fast instructions; other threads update a shared count (`ob_ref_shared`) atomically. | Most objects are only touched by the thread that made them, so most updates stay cheap. |
| **Deferred reference counting** | Objects that are referenced constantly from many threads (functions, modules, code objects, classes) skip count updates when pushed on the interpreter stack; the GC accounts for those references when it runs. | No contention on the hottest objects. |
| **Immortalization** | 3.1.4: `None`, small ints, interned strings never change count. | Zero writes for the most shared objects. |

Other parts of the design: per-object locks (*critical sections*) inside
`list` / `dict` methods, mimalloc as the allocator (3.1.9), and a GC that stops all
threads briefly. The object header grows to hold `ob_tid`, both counts and a lock.

First, which build is this kernel? `sysconfig` tells you how the interpreter was
**built**; `sys._is_gil_enabled()` (*New in 3.13*) tells you whether the GIL is on
**right now** (a free-threaded build can re-enable it, e.g. for an extension that
is not marked safe).

In [15]:
gil_disabled_build = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
print(f"{gil_disabled_build=}")
if hasattr(sys, "_is_gil_enabled"):
    print(f"{sys._is_gil_enabled()=}")
else:
    print("sys._is_gil_enabled() needs Python 3.13+")
print(f"{sys.getsizeof(object())=}  (object header size on this build)")
if not gil_disabled_build:
    print("This kernel is the default build: the comparison below runs in child processes.")

gil_disabled_build=False
sys._is_gil_enabled()=True
sys.getsizeof(object())=16  (object header size on this build)
This kernel is the default build: the comparison below runs in child processes.


**Comparing the two builds.** If a free-threaded interpreter is installed
(`uv python install 3.15t`, found with `uv python find 3.15t`), the next cell runs
the same probe in this kernel's interpreter and in the free-threaded one, each in
a child process. The probe reports:

- the size of a bare `object()` (the bigger header),
- `sys.getrefcount` of a module-level function (deferred reference counting makes
  it a huge marker value on the free-threaded build),
- wall time of 4 threads doing pure-Python arithmetic, divided by the time for 1
  thread. With a GIL the threads take turns (ratio around 4 or worse); without
  it they run in parallel (ratio near 1 when you have 4+ cores).

> **Note:** timings depend on the machine and its load. The interpreter path is
> never printed.

In [16]:
FREE_THREADING_PROBE = textwrap.dedent(
    """
    import os, sys, sysconfig, threading, time

    def spin(count):
        total = 0
        for number in range(count):
            total += number * number

    def wall_time(thread_count, count=1_500_000):
        threads = [threading.Thread(target=spin, args=(count,)) for _ in range(thread_count)]
        start = time.perf_counter()
        for thread in threads:
            thread.start()
        for thread in threads:
            thread.join()
        return time.perf_counter() - start

    def module_level_function():
        pass

    print("build", "free-threaded" if sysconfig.get_config_var("Py_GIL_DISABLED") else "default")
    print("gil_enabled", sys._is_gil_enabled())
    print("object_header_bytes", sys.getsizeof(object()))
    print("function_refcount", sys.getrefcount(module_level_function))
    print("ratio_4_threads_vs_1", round(wall_time(4) / wall_time(1), 2))
    """
)


def run_probe(python_path: str) -> dict[str, str]:
    """Run FREE_THREADING_PROBE in a child interpreter and parse "key value" lines."""
    completed = subprocess.run(
        [python_path, "-c", FREE_THREADING_PROBE],
        capture_output=True, text=True, env=CHILD_ENV, timeout=180,
    )  # fmt: skip
    return dict(line.split(" ", 1) for line in completed.stdout.splitlines())


free_threaded_python = find_free_threaded_python()
if free_threaded_python is None:
    print("Skipped: no free-threaded interpreter found (try: uv python install 3.15t)")
else:
    print(f"CPU cores: {os.cpu_count()}")
    default_result = run_probe(sys.executable)
    free_threaded_result = run_probe(free_threaded_python)
    print(f"{'metric':<22}{'this kernel':>22}{'3.15t child':>22}")
    for key in default_result:
        print(f"{key:<22}{default_result[key]:>22}{free_threaded_result.get(key, '?'):>22}")

CPU cores: 16


metric                           this kernel           3.15t child
build                                default         free-threaded
gil_enabled                             True                 False
object_header_bytes                       16                    32
function_refcount                          2   1152921504606846977
ratio_4_threads_vs_1                    6.06                  1.19


### 3.1.6 Reference cycles and the cyclic GC

Reference counting alone cannot free a **cycle**: two objects that reference each
other keep each other's count at 1 or more even after every outside name is gone.

```mermaid
flowchart LR
    name["name `parent` (deleted)"] -. "x" .-> P
    P["Category 'Tea'"] -- children --> C["Category 'Green tea'"]
    C -- parent --> P
```

<details><summary>ASCII fallback</summary>

```text
parent (name deleted)  -x->  [Category 'Tea'] --children--> [Category 'Green tea']
                                    ^                               |
                                    +------------parent-------------+
each count stays at 1: refcounting alone never frees them
```

</details>

CPython's **cyclic garbage collector** handles this. It only tracks objects that
can hold references (containers), and periodically looks for groups that are
referenced *only from inside the group*. Those are unreachable and get freed.

The cell builds a parent/child cycle with the collector switched off, shows that
`del` does not free it, then runs `gc.collect()`. The `try` / `finally` restores
the collector so later cells are unaffected.

In [17]:
class Category:
    """A tree node that links to its parent and children: a natural cycle."""

    def __init__(self, name: str, parent: Category | None = None) -> None:
        self.name = name
        self.parent = parent
        self.children: list[Category] = []
        if parent is not None:
            parent.children.append(self)


freed: list[str] = []
gc_was_enabled = gc.isenabled()
gc.disable()
try:
    tea = Category("Tea")
    green_tea = Category("Green tea", parent=tea)
    weakref.finalize(tea, freed.append, "Tea")
    weakref.finalize(green_tea, freed.append, "Green tea")
    del tea, green_tea
    print(f"after del:          freed={freed}  (the cycle keeps both alive)")
    unreachable_count = gc.collect()
    print(f"after gc.collect(): freed={sorted(freed)}  ({unreachable_count} objects found)")
finally:
    if gc_was_enabled:
        gc.enable()

after del:          freed=[]  (the cycle keeps both alive)
after gc.collect(): freed=['Green tea', 'Tea']  (6 objects found)


#### The `gc` module API

| Function | What it does | Typical use |
| --- | --- | --- |
| `gc.collect(generation=2)` | Run a collection now; returns the number of unreachable objects found | Tests, benchmarks, before a memory snapshot |
| `gc.is_tracked(obj)` | Is this object in the collector's lists? | Learning which objects the GC even looks at |
| `gc.get_threshold()` | The allocation thresholds that trigger collections | Tuning (rarely needed) |
| `gc.get_count()` | Current counters per generation | Watching allocation pressure |
| `gc.get_stats()` | Per-generation totals: collections, collected, uncollectable | Monitoring |
| `gc.get_referrers(*objs)` | Objects that refer to these objects | Finding who keeps a leaked object alive |
| `gc.get_referents(*objs)` | Objects these objects refer to | Walking an object graph |
| `gc.freeze()` | Move every tracked object to a permanent generation the GC ignores | Before `fork()` in pre-fork servers |
| `gc.callbacks` | List of `callback(phase, info)` run before and after each collection | Measuring GC pauses |

One demo per function follows.

#### `gc.collect`

Returns how many unreachable objects it found. A second call right after usually
finds nothing.

In [18]:
for _ in range(3):
    first_node, second_node = {}, {}
    first_node["peer"], second_node["peer"] = second_node, first_node
del first_node, second_node
print(f"{gc.collect()=}  (three 2-dict cycles, plus anything else pending)")
print(f"{gc.collect()=}  (nothing left)")

gc.collect()=6  (three 2-dict cycles, plus anything else pending)
gc.collect()=0  (nothing left)


#### `gc.is_tracked`

Atomic objects (ints, strings, floats) cannot form cycles, so they are never
tracked. Containers are. As an optimization, CPython may **untrack** a tuple that
holds only untracked objects, since it can never be part of a cycle. *When* that
happens (at creation, or when the collector first inspects the tuple) is an
implementation detail that has changed between versions; the output shows what
this build does.

In [19]:
print(interpreter_label())
for label, sample in {
    "42": 42,
    "'text'": "text",
    "[]": [],
    "{}": {},
    "(1, 2) constant": (1, 2),
    "([],) holds a list": ([],),
    "Category('x')": Category("x"),
}.items():
    print(f"{label:<20} tracked={gc.is_tracked(sample)}")

runtime_tuple = tuple([1, 2, 3])
tuple_with_list = tuple([[]])
print(f"runtime tuple of ints:   before collect {gc.is_tracked(runtime_tuple)}", end="")
print(f", tuple holding a list: {gc.is_tracked(tuple_with_list)}")
gc.collect()
print(f"after gc.collect():      tuple of ints {gc.is_tracked(runtime_tuple)}", end="")
print(f", tuple holding a list: {gc.is_tracked(tuple_with_list)}")

CPython 3.15.1, default (GIL) build, x86_64
42                   tracked=False
'text'               tracked=False
[]                   tracked=True
{}                   tracked=True
(1, 2) constant      tracked=True
([],) holds a list   tracked=True
Category('x')        tracked=True
runtime tuple of ints:   before collect False, tuple holding a list: True
after gc.collect():      tuple of ints False, tuple holding a list: True


#### `gc.get_threshold`

A collection of the youngest generation starts when *allocations minus
deallocations* of tracked objects exceed the first threshold; the other two values
control how often older generations are examined. `gc.set_threshold` changes them;
the cell only reads them.

In [20]:
print(f"{gc.get_threshold()=}  [{interpreter_label()}]")

gc.get_threshold()=(2000, 10, 10)  [CPython 3.15.1, default (GIL) build, x86_64]


#### `gc.get_count`

The counters that are compared with the thresholds. Allocating tracked
containers raises the first one; a collection resets it.

In [21]:
gc.collect()
print(f"after collect:          {gc.get_count()=}")
new_containers = [{} for _ in range(500)]
print(f"after 500 new dicts:    {gc.get_count()=}")
del new_containers

after collect:          gc.get_count()=(37, 0, 0)
after 500 new dicts:    gc.get_count()=(559, 0, 0)


#### `gc.get_stats`

One dictionary per generation with running totals since the interpreter started.
`uncollectable` should stay at 0 (it counts objects the GC found but could not
free).

In [22]:
for generation, stats in enumerate(gc.get_stats()):
    print(f"generation {generation}: {stats}")

generation 0: {'collections': 48, 'collected': 546, 'uncollectable': 0, 'candidates': 105001, 'duration': 0.015486129000000001}
generation 1: {'collections': 4, 'collected': 178, 'uncollectable': 0, 'candidates': 62567, 'duration': 0.013757601}
generation 2: {'collections': 5, 'collected': 12, 'uncollectable': 0, 'candidates': 309393, 'duration': 0.134555863}


#### `gc.get_referrers`

Answers "who is keeping this alive?". It scans every tracked object, so it is slow
and returns frames and namespaces too; filter the result. Here a `Category` is
found inside a registry list.

In [23]:
registry: list[Category] = []
orphan = Category("Orphan")
registry.append(orphan)
referrer_types = sorted({type(referrer).__name__ for referrer in gc.get_referrers(orphan)})
print(f"{referrer_types=}")
print(f"registry is a referrer: {any(ref is registry for ref in gc.get_referrers(orphan))}")
registry.clear()

referrer_types=['dict', 'list']
registry is a referrer: True


#### `gc.get_referents`

The other direction: what an object refers to (as the C-level traversal sees it).
For an instance with a materialized `__dict__` this includes the dict and the
class.

In [24]:
parent_category = Category("Coffee")
child_category = Category("Espresso", parent=parent_category)
vars(child_category)  # materialize the instance __dict__
print(sorted(type(item).__name__ for item in gc.get_referents(child_category)))
print([type(item).__name__ for item in gc.get_referents([1, "two", parent_category])])
del parent_category, child_category

['dict', 'type']
['Category', 'str', 'int']


#### `gc.freeze`

*New in 3.7.* Moves every currently tracked object into a **permanent
generation** that future collections skip. Pre-fork servers (gunicorn, uWSGI)
call it in the parent before forking workers, so the GC in each child does not
write to (and copy) the shared pages. `gc.unfreeze()` undoes it.

In [25]:
gc.collect()
gc.freeze()
try:
    print(f"after freeze:   {gc.get_freeze_count()=:,}")
finally:
    gc.unfreeze()
print(f"after unfreeze: {gc.get_freeze_count()=}")

after freeze:   gc.get_freeze_count()=61,766
after unfreeze: gc.get_freeze_count()=0


#### `gc.callbacks`

A list of functions called as `callback(phase, info)` with `phase` `"start"` or
`"stop"` and an `info` dict (`generation`, `collected`, `uncollectable`). Useful for
timing collections in production. Always remove your callback again.

In [26]:
collection_log: list[str] = []


def record_collection(phase: str, info: dict[str, int]) -> None:
    """GC callback: log each phase with the generation and the result."""
    collected = info.get("collected", "-")
    collection_log.append(f"{phase}: generation={info['generation']} collected={collected}")


gc.callbacks.append(record_collection)
try:
    gc.collect()
finally:
    gc.callbacks.remove(record_collection)
print("\n".join(collection_log))

start: generation=2 collected=0
stop: generation=2 collected=0


#### Generations

Most objects die young. The collector therefore groups tracked objects by age:
new containers start in the **youngest generation**, which is collected often and
cheaply; survivors move to **older generations**, examined less often. A full
collection (`gc.collect()` with no argument) looks at everything.

```text
new container --> [ young ] --survives--> [ older ] --survives--> [ oldest ]
                 collected often           less often             rarely
gc.freeze()  --> [ permanent ]  never examined
```

> **Note:** the exact scheme is an implementation detail that has been reworked in
> recent releases (thresholds and how many generations are scanned per step). Read
> `gc.get_threshold()` and `gc.get_stats()` on your version instead of memorizing
> numbers.

#### Case study: `gc.disable`

Instagram's Django workers (2017) ran with `gc.disable()`: after `fork()`, every
collection wrote to the GC headers of shared objects, which forced the OS to copy
those pages into every worker. Turning the collector off saved memory because their
code created few cycles. The risk: **any cycle you do create now leaks**. That trade
led to `gc.freeze()`, which keeps the copy-on-write benefit without giving up
collection.

The cell shows the risk. With the collector off, 20 000 small cycles per round pile
up; `tracemalloc` (3.1.11) shows memory growing until `gc.collect()` runs. The
`finally` block restores everything.

In [27]:
gc_was_enabled = gc.isenabled()
gc.disable()
tracemalloc.start()
try:
    for round_number in range(1, 4):
        for _ in range(20_000):
            first_node, second_node = {}, {}
            first_node["peer"], second_node["peer"] = second_node, first_node
        traced_now, _peak = tracemalloc.get_traced_memory()
        print(f"round {round_number}: traced {traced_now / 1e6:5.1f} MB, {gc.get_count()=}")
    del first_node, second_node
    found_count = gc.collect()
    traced_now, _peak = tracemalloc.get_traced_memory()
    print(f"gc.collect() found {found_count:,} objects; traced now {traced_now / 1e6:.1f} MB")
finally:
    tracemalloc.stop()
    if gc_was_enabled:
        gc.enable()
print(f"collector restored: {gc.isenabled()=}")

round 1: traced   7.4 MB, gc.get_count()=(40628, 0, 0)


round 2: traced  14.7 MB, gc.get_count()=(80642, 0, 0)


round 3: traced  22.1 MB, gc.get_count()=(120631, 0, 0)
gc.collect() found 120,000 objects; traced now 0.0 MB
collector restored: gc.isenabled()=True


### 3.1.7 Finalizers

`__del__` runs when an object is about to be destroyed: when its count hits zero,
or when the cyclic GC frees it. Its rules surprise people:

| Rule | Consequence |
| --- | --- |
| Timing depends on references | One forgotten reference (a traceback, a cache) and it never runs |
| Not guaranteed at interpreter exit | Objects alive at shutdown may never be finalized |
| Exceptions are not raised | They go to `sys.unraisablehook` ("Exception ignored in ...") |
| It can **resurrect** the object | By storing `self` somewhere; the object survives |
| Runs at most once per object (PEP 442) | A resurrected object will not be finalized again |
| Cycles with `__del__` are collected (PEP 442, 3.4) | Before 3.4 they went to `gc.garbage` and leaked |

**When it runs.** With no cycle involved, CPython calls `__del__` right at `del`.

In [28]:
class Connection:
    """Pretends to hold a socket and logs its own finalization."""

    log: list[str] = []

    def __init__(self, name: str) -> None:
        self.name = name

    def __del__(self) -> None:
        Connection.log.append(f"__del__ {self.name}")


database_connection = Connection("db")
Connection.log.append("before del")
del database_connection
Connection.log.append("after del")
print(Connection.log)

['before del', '__del__ db', 'after del']


**Exceptions inside `__del__` are swallowed.** They are reported through
`sys.unraisablehook` (by default a message on stderr) and the program carries on.
The cell installs a temporary hook to capture the report, then restores the
original.

In [29]:
class BrokenCleanup:
    """A finalizer that fails."""

    def __del__(self) -> None:
        raise RuntimeError("cleanup failed")


captured_reports: list[str] = []


def capture_unraisable(unraisable: sys.UnraisableHookArgs) -> None:
    """Record the exception instead of printing it."""
    captured_reports.append(f"{type(unraisable.exc_value).__name__}: {unraisable.exc_value}")


original_hook = sys.unraisablehook
sys.unraisablehook = capture_unraisable
try:
    broken = BrokenCleanup()
    del broken
    print("after del: the program kept running")
finally:
    sys.unraisablehook = original_hook
print(f"{captured_reports=}")

after del: the program kept running
captured_reports=['RuntimeError: cleanup failed']


**Resurrection and PEP 442.** A finalizer that stores `self` somewhere creates a new
reference, so the object survives. PEP 442 guarantees `__del__` is called **at most
once**: when the resurrected object dies for real, it is freed without a second
call.

In [30]:
resurrected: list[object] = []


class Phoenix:
    """Resurrects itself the first time it is finalized."""

    del_calls = 0

    def __del__(self) -> None:
        Phoenix.del_calls += 1
        resurrected.append(self)  # a new strong reference


bird = Phoenix()
del bird
print(f"{Phoenix.del_calls=}  {len(resurrected)=}  (alive again)")
resurrected.clear()  # the last reference goes away a second time
gc.collect()
print(f"{Phoenix.del_calls=}  (not called again)")

Phoenix.del_calls=1  len(resurrected)=1  (alive again)
Phoenix.del_calls=1  (not called again)


**Cycles with finalizers are collected.** Two objects with `__del__` that reference
each other are freed by `gc.collect()`; both finalizers run and nothing lands in
`gc.garbage`.

In [31]:
gc_was_enabled = gc.isenabled()
gc.disable()
try:
    Connection.log.clear()
    primary, replica = Connection("primary"), Connection("replica")
    primary.peer, replica.peer = replica, primary
    del primary, replica
    print(f"before collect: {Connection.log}")
    gc.collect()
    print(f"after collect:  {sorted(Connection.log)}   gc.garbage={gc.garbage}")
finally:
    if gc_was_enabled:
        gc.enable()

before collect: []
after collect:  ['__del__ primary', '__del__ replica']   gc.garbage=[]


**Prefer context managers and `weakref.finalize`.** For a resource that must be
released, give the class a `close()` method and `__enter__` / `__exit__`, so
callers release it at a known point. Add `weakref.finalize` as a safety net for
callers who forget:

- it does not keep the object alive (pass the data it needs, never `self` or a
  bound method),
- it runs at most once, whether called explicitly or at object death,
- by default it also runs at interpreter exit (`atexit=True`), unlike `__del__`.

In [32]:
class ScratchFolder:
    """A temporary folder: removed by close()/with, or by a finalizer as a safety net."""

    def __init__(self) -> None:
        self.path = Path(tempfile.mkdtemp(prefix="unit31_"))
        self._finalizer = weakref.finalize(self, shutil.rmtree, self.path, ignore_errors=True)

    def close(self) -> None:
        """Remove the folder now (safe to call twice)."""
        self._finalizer()

    @property
    def closed(self) -> bool:
        return not self._finalizer.alive

    def __enter__(self) -> ScratchFolder:
        return self

    def __exit__(self, *exc_info: object) -> None:
        self.close()


with ScratchFolder() as folder:
    (folder.path / "notes.txt").write_text("draft")
    print(f"inside with: exists={folder.path.exists()}")
print(f"after with:  exists={folder.path.exists()}  closed={folder.closed}")

forgotten = ScratchFolder()
forgotten_path = forgotten.path
del forgotten  # the caller forgot close(): the finalizer cleans up
print(f"forgotten folder removed by the finalizer: {not forgotten_path.exists()}")

inside with: exists=True
after with:  exists=False  closed=True
forgotten folder removed by the finalizer: True


### 3.1.8 Weak references

A **weak reference** points at an object without adding to its count, so it does
not keep the object alive. Use them for caches, registries and observer lists
that should not decide an object's lifetime.

| API | What it is | JS equivalent |
| --- | --- | --- |
| `weakref.ref(obj, callback=None)` | Call it to get the object, or `None` once it died; optional callback on death | `new WeakRef(obj)`, `.deref()` |
| `weakref.WeakValueDictionary` | Dict whose **values** are weak: entries vanish when the value dies | — |
| `weakref.WeakKeyDictionary` | Dict whose **keys** are weak: attach data to objects you do not own | `WeakMap` |
| `weakref.WeakSet` | Set of weakly held objects | `WeakSet` |
| `weakref.WeakMethod` | Weak reference to a **bound method** that stays valid while the instance lives | — |
| `__weakref__` slot | Where an instance keeps its list of weak references; needed with `__slots__` | — |

Many built-in types (`int`, `str`, `list`, `dict`, `tuple`) **cannot** be weakly
referenced; instances of your own classes can.

#### `weakref.ref`

In [33]:
class Session:
    """A logged-in user session."""

    def __init__(self, user: str) -> None:
        self.user = user


death_notices: list[str] = []
session = Session("asha")
session_ref = weakref.ref(session, lambda dead_ref: death_notices.append("session gone"))
print(f"{session_ref() is session=}  {session_ref().user=}")
del session
print(f"{session_ref()=}  {death_notices=}")
for value in (42, "text", [1, 2], {"a": 1}):
    outcome = raised_message(functools.partial(weakref.ref, value))
    print(f"weakref to {type(value).__name__:<5}: {outcome}")

session_ref() is session=True  session_ref().user='asha'
session_ref()=None  death_notices=['session gone']
weakref to int  : TypeError: cannot create weak reference to 'int' object
weakref to str  : TypeError: cannot create weak reference to 'str' object
weakref to list : TypeError: cannot create weak reference to 'list' object
weakref to dict : TypeError: cannot create weak reference to 'dict' object


#### `weakref.WeakValueDictionary`

A registry of live sessions by id. The registry never keeps a session alive: when
the last strong reference goes, the entry disappears.

In [34]:
live_sessions: weakref.WeakValueDictionary[str, Session] = weakref.WeakValueDictionary()
asha_session, ravi_session = Session("asha"), Session("ravi")
live_sessions["s-1"], live_sessions["s-2"] = asha_session, ravi_session
print(f"{sorted(live_sessions)=}")
del ravi_session
print(f"{sorted(live_sessions)=}  (s-2 vanished with its session)")
print(f"{live_sessions.get('s-2')=}")

sorted(live_sessions)=['s-1', 's-2']
sorted(live_sessions)=['s-1']  (s-2 vanished with its session)
live_sessions.get('s-2')=None


#### `weakref.WeakKeyDictionary`

Attach extra data to objects you do not own (or cannot add attributes to) without
extending their life. Like a JS `WeakMap`.

In [35]:
request_counts: weakref.WeakKeyDictionary[Session, int] = weakref.WeakKeyDictionary()
meena_session = Session("meena")
request_counts[asha_session] = 3
request_counts[meena_session] = 1
print(f"{len(request_counts)=}")
del meena_session
print(f"{len(request_counts)=}  {[s.user for s in request_counts]=}")

len(request_counts)=2
len(request_counts)=1  [s.user for s in request_counts]=['asha']


#### `weakref.WeakSet`

Track "everything currently alive" of some kind, such as open connections, for a
status page or a shutdown hook.

In [36]:
open_sessions: weakref.WeakSet[Session] = weakref.WeakSet()
temporary_sessions = [Session(f"guest-{number}") for number in range(3)]
open_sessions.update(temporary_sessions)
open_sessions.add(asha_session)
print(f"{len(open_sessions)=}")
del temporary_sessions
print(f"{len(open_sessions)=}  {[s.user for s in open_sessions]=}")

len(open_sessions)=4
len(open_sessions)=1  [s.user for s in open_sessions]=['asha']


#### `weakref.WeakMethod`

`obj.method` creates a **new** bound-method object on every access. A plain
`weakref.ref` to it dies immediately, because nothing else holds that temporary
object. `WeakMethod` stores weak references to the instance and the function
separately and rebuilds the bound method on demand.

In [37]:
class PriceDisplay:
    """Something with a callback method."""

    def refresh(self, price: float) -> str:
        return f"showing {price:.2f}"


display = PriceDisplay()
plain_ref = weakref.ref(display.refresh)
method_ref = weakref.WeakMethod(display.refresh)
print(f"{plain_ref()=}  (the temporary bound method already died)")
print(f"{method_ref()(99.5)=}")
del display
print(f"{method_ref()=}  (the instance died)")

plain_ref()=None  (the temporary bound method already died)
method_ref()(99.5)='showing 99.50'
method_ref()=None  (the instance died)


#### The `__weakref__` slot

Ordinary classes get a `__weakref__` descriptor automatically. A class with
`__slots__` does not, unless you list `"__weakref__"` among the slots; it costs one
pointer per instance.

In [38]:
class SlottedPoint:
    __slots__ = ("x", "y")


class WeakableSlottedPoint:
    __slots__ = ("x", "y", "__weakref__")


print(f"{'__weakref__' in vars(Session)=}  (added automatically)")
print(f"SlottedPoint:         {raised_message(lambda: weakref.ref(SlottedPoint()))}")
print(f"WeakableSlottedPoint: {raised_message(lambda: weakref.ref(WeakableSlottedPoint()))}")
print(
    f"{sys.getsizeof(SlottedPoint())=}  {sys.getsizeof(WeakableSlottedPoint())=}"
    f"  [{interpreter_label()}]"
)
del asha_session

'__weakref__' in vars(Session)=True  (added automatically)
SlottedPoint:         TypeError: cannot create weak reference to 'SlottedPoint' object
WeakableSlottedPoint: no error
sys.getsizeof(SlottedPoint())=48  sys.getsizeof(WeakableSlottedPoint())=64  [CPython 3.15.1, default (GIL) build, x86_64]


### 3.1.9 The allocator

Python creates and frees small objects constantly, and the C `malloc` is slow for
that pattern. CPython's default build therefore uses its own small-object
allocator, **pymalloc**, for requests of **512 bytes or less**; bigger requests go
to the system `malloc`.

```mermaid
flowchart TD
    R["allocation request"] --> Q{"size <= 512 bytes?"}
    Q -- yes --> A["pymalloc"]
    Q -- no --> M["system malloc / mmap"]
    A --> AR["arena: 1 MiB from mmap"]
    AR --> P["pools: 16 KiB, each serving ONE size class"]
    P --> B["blocks: fixed size, multiple of 16 bytes"]
```

<details><summary>ASCII fallback</summary>

```text
request --(> 512 B)--> system malloc / mmap
   |
   +--(<= 512 B)--> pymalloc
                      arena (1 MiB, from mmap)
                        +-- pool (16 KiB) for 16-byte blocks
                        +-- pool (16 KiB) for 32-byte blocks
                        +-- pool (16 KiB) for 48-byte blocks ...
                              +-- block, block, block (one object each)
```

</details>

| Level | Size (64-bit, 3.10+) | Holds |
| --- | --- | --- |
| Arena | 1 MiB, requested from the OS with `mmap` | many pools |
| Pool | 16 KiB | blocks of **one** size class |
| Block | 16, 32, 48, … 512 bytes (32 classes) | one object (request rounded up) |

`sys._debugmallocstats()` prints pymalloc's tables. It writes to the C-level
stderr, which a notebook does not capture, so the helper below runs it in a child
interpreter and returns the lines.

In [39]:
def malloc_stats_lines(
    python_path: str, setup_code: str = "", extra_env: dict[str, str] | None = None
) -> list[str]:
    """Run setup_code then sys._debugmallocstats() in a child; return its report lines."""
    script = f"import sys\n{setup_code}\nsys._debugmallocstats()"
    completed = subprocess.run(
        [python_path, "-c", script],
        capture_output=True, text=True, env={**CHILD_ENV, **(extra_env or {})}, timeout=60,
    )  # fmt: skip
    return completed.stderr.splitlines()


default_stats = malloc_stats_lines(sys.executable)
print(interpreter_label())
print(default_stats[0])
summary_prefixes = ("# arenas allocated current", "# bytes in allocated blocks", "Total")
shown_prefixes: set[str] = set()
for line in default_stats:
    prefix = next((p for p in summary_prefixes if line.startswith(p)), None)
    if prefix and prefix not in shown_prefixes:
        shown_prefixes.add(prefix)
        print(line)

CPython 3.15.1, default (GIL) build, x86_64
Small block threshold = 512, in 32 size classes.
# arenas allocated current         =                    2
# bytes in allocated blocks        =            1,483,840
Total                              =            2,097,152


**Size classes in action.** A `bytes` object of 150 bytes needs about 183 bytes
(header + data), which rounds up to the **192-byte** size class. The cell asks two
child interpreters for their 192-byte row: one idle, one holding 10 000 such
objects. Columns: size-class index, block size, pools, blocks in use, free blocks.

In [40]:
def size_class_row(stats_lines: list[str], block_size: int) -> str:
    """The pymalloc table row for one block size."""
    for line in stats_lines:
        columns = line.split()
        if len(columns) == 5 and columns[1] == str(block_size):
            return line.strip()
    return "<row not found>"


print(f"{sys.getsizeof(bytes(150))=}  -> rounded up to a 192-byte block")
print("class  size  pools  in-use  free")
print(size_class_row(malloc_stats_lines(sys.executable), 192), " (idle child)")
holding_code = "keep = [bytes(150) for _ in range(10_000)]"
print(size_class_row(malloc_stats_lines(sys.executable, holding_code), 192), " (10 000 objects)")

sys.getsizeof(bytes(150))=183  -> rounded up to a 192-byte block
class  size  pools  in-use  free


11    192           1              54            31  (idle child)


11    192         119           10053            62  (10 000 objects)


**Why RSS rarely shrinks.** pymalloc returns an arena to the OS only when **every**
block in it is free. Free one million small objects but keep one in every 64
alive, and almost every pool still has a live block, so almost every arena stays.
The memory is free *inside* the process (reused for new objects) but the RSS does
not drop. Free them all and arenas go back.

> **Note:** RSS numbers depend on the OS and the allocator; read the trend.

In [41]:
class Crumb:
    """A small object (one pymalloc block each)."""


def rss_megabytes() -> str:
    """RSS formatted in MB, or 'n/a'."""
    rss_value = rss_bytes()
    return "n/a" if rss_value is None else f"{rss_value / 1e6:6.1f} MB"


print(f"start:                      RSS {rss_megabytes()}")
crumbs = [Crumb() for _ in range(1_000_000)]
print(f"1 000 000 objects:          RSS {rss_megabytes()}")
survivors = crumbs[::64]
del crumbs
print(f"freed all but 1 in 64:      RSS {rss_megabytes()}  (arenas pinned)")
del survivors
print(f"freed the survivors too:    RSS {rss_megabytes()}  [{interpreter_label()}]")

start:                      RSS   83.6 MB


1 000 000 objects:          RSS  159.3 MB
freed all but 1 in 64:      RSS  159.3 MB  (arenas pinned)
freed the survivors too:    RSS   85.1 MB  [CPython 3.15.1, default (GIL) build, x86_64]


**mimalloc in the free-threaded build.** pymalloc relies on the GIL for thread
safety. The free-threaded build uses **mimalloc** (Microsoft's allocator, *New in
3.13*) instead: it has per-thread heaps, and the free-threaded GC finds objects by
walking mimalloc's heaps rather than a linked list. The default build also ships
mimalloc (selectable with `PYTHONMALLOC=mimalloc`). Its report looks different: no
512-byte limit, no arenas/pools table.

In [42]:
free_threaded_python = find_free_threaded_python()
if free_threaded_python is not None:
    print("free-threaded 3.15t child:")
    print("\n".join(malloc_stats_lines(free_threaded_python)[:4]))
else:
    print("Skipped: no free-threaded interpreter found (uv python install 3.15t)")
if sysconfig.get_config_var("WITH_MIMALLOC"):
    print("\nthis interpreter with PYTHONMALLOC=mimalloc:")
    print("\n".join(malloc_stats_lines(sys.executable, extra_env={"PYTHONMALLOC": "mimalloc"})[:2]))
else:
    print("Skipped: this interpreter was built without mimalloc")

free-threaded 3.15t child:
Small block threshold = 16384, in 73 size classes.
Medium block threshold = 131072
Large object max size = 16777216
    Allocated Blocks: 1181

this interpreter with PYTHONMALLOC=mimalloc:
Small block threshold = 16384, in 73 size classes.
Medium block threshold = 131072


### 3.1.10 Free lists and caches

On top of pymalloc, CPython keeps a few shortcuts for the most common objects:

| Mechanism | What it does | Visible as |
| --- | --- | --- |
| Free lists (`float`, `tuple` by length, `list`, `dict`, …) | A freed object's memory is kept on a per-type list and handed to the next object of that type | The same address reused at once |
| Small-int cache | Ints from -5 up to 256 (1024 since 3.15) are preallocated, immortal singletons | `int("100") is int("100")` |
| Empty-tuple singleton | There is only one `()` | `tuple() is ()` |
| One-character Latin-1 strings, empty `str` / `bytes` | Preallocated singletons | `chr(97) is "a"` |
| Interning | One shared object per distinct identifier-like string | `sys.intern`, identity of names |

The free lists appear at the end of `sys._debugmallocstats()`.

In [43]:
stats_lines = malloc_stats_lines(sys.executable)
free_list_lines = [line.strip() for line in stats_lines if " free Py" in line]
print(interpreter_label())
print("\n".join(free_list_lines[:8]))

CPython 3.15.1, default (GIL) build, x86_64
5 free PyDictObjects * 64 bytes each =                  320
2 free PyDictKeysObjects * 32 bytes each =                   64
2 free PyFloatObjects * 24 bytes each =                   48
7 free PyListObjects * 56 bytes each =                  392


**Address reuse.** Free a float and immediately create another, and the new one
usually lands at the same address, taken straight from the free list. That is why
`id()` values are only unique **among objects alive at the same time**. Never
store an `id()` as a long-term key.

In [44]:
multiplier = len("ab")  # computed, so the floats below are new objects
first_price = 1.5 * multiplier
first_address = id(first_price)
del first_price
second_price = 2.5 * multiplier
print(f"float reused the freed address: {id(second_price) == first_address}")

first_list = [multiplier]
first_list_address = id(first_list)
del first_list
second_list = [multiplier + 1]
print(f"list reused the freed address:  {id(second_list) == first_list_address}")
print(f"(free-list behaviour, not a guarantee)  [{interpreter_label()}]")

float reused the freed address: True
list reused the freed address:  False
(free-list behaviour, not a guarantee)  [CPython 3.15.1, default (GIL) build, x86_64]


**The small-int cache and singletons.** Numbers are built with `int(str)` at run
time; literals in one cell would be folded into shared constants and hide the
effect.

In [45]:
print(interpreter_label())
for number in (-6, -5, 256, 257, 1024, 1025):
    cached = int(str(number)) is int(str(number))
    print(f"{number:>5}: two separately built ints are one object: {cached}")

empty_tuple_literal, empty_str_literal, empty_bytes_literal = (), "", b""
print(f"built empty tuple is ():  {tuple(range(0)) is empty_tuple_literal}")
print(f"built empty str is '':    {''.join([]) is empty_str_literal}")
print(f"built empty bytes is b'': {bytes(bytearray()) is empty_bytes_literal}")
print(f"chr(97) is 'a': {chr(97) is 'abc'[0]}  (one-character Latin-1 strings are cached)")

CPython 3.15.1, default (GIL) build, x86_64
   -6: two separately built ints are one object: False
   -5: two separately built ints are one object: True
  256: two separately built ints are one object: True
  257: two separately built ints are one object: True
 1024: two separately built ints are one object: True
 1025: two separately built ints are one object: False
built empty tuple is ():  True
built empty str is '':    True
built empty bytes is b'': True
chr(97) is 'a': True  (one-character Latin-1 strings are cached)


**Interning.** The compiler interns identifiers and identifier-like string
constants, so attribute names and dict keys in your code compare by pointer first.
Strings built at run time are separate objects until you call `sys.intern`, which
returns the one canonical copy.

In [46]:
built_once = "".join(["order", "_", "status"])
built_twice = "".join(["order", "_", "status"])
print(f"{built_once == built_twice=}  {built_once is built_twice=}")
canonical_once, canonical_twice = sys.intern(built_once), sys.intern(built_twice)
print(f"after sys.intern: {canonical_once is canonical_twice=}")
order_status_literal = "order_status"
print(f"same object as the literal: {canonical_once is order_status_literal}")

built_once == built_twice=True  built_once is built_twice=False
after sys.intern: canonical_once is canonical_twice=True
same object as the literal: True


### 3.1.11 Measuring memory

| Tool | Measures | Blind spots |
| --- | --- | --- |
| `sys.getsizeof(obj)` | One object, **shallow**: not the objects it references | Contents of containers, shared objects |
| Recursive size (you write it) | An object graph | Must handle cycles and sharing (Lab 3.1, exercise 1) |
| `tracemalloc` | Every allocation made through Python's allocators, with the line that made it | Memory allocated by C libraries directly |
| Process RSS | What the OS has given the process | Cannot tell you which code; includes allocator slack |
| memray (Phase 5) | Every allocation including native (C) ones, flame graphs | Separate tool, Linux/macOS |

**Shallow vs recursive.** `sys.getsizeof` of an order dict counts only the dict's
own table. A naive recursive version adds the contents, but it double-counts shared
objects and never finishes on a cycle (it hits the recursion limit).

In [47]:
def naive_total_size(obj: object) -> int:
    """Recursive size of nested lists and dicts, WITHOUT cycle or sharing handling."""
    size = sys.getsizeof(obj)
    if isinstance(obj, list):
        size += sum(naive_total_size(item) for item in obj)
    elif isinstance(obj, dict):
        size += sum(naive_total_size(key) + naive_total_size(value) for key, value in obj.items())
    return size


order = {"id": 7, "lines": [{"sku": "TEA-01", "qty": 2}, {"sku": "RICE-5", "qty": 1}]}
print(f"{sys.getsizeof(order)=}  (shallow)")
print(f"{naive_total_size(order)=}  (contents included)  [{interpreter_label()}]")
looped_list: list[object] = [1, 2]
looped_list.append(looped_list)
print(f"on a cycle: {raised_error_name(lambda: naive_total_size(looped_list))}")

sys.getsizeof(order)=184  (shallow)
naive_total_size(order)=1067  (contents included)  [CPython 3.15.1, default (GIL) build, x86_64]
on a cycle: RecursionError


**`tracemalloc` snapshots and diffs.** `tracemalloc.start()` records the
traceback of every allocation. Take a snapshot before and after some work, then
`after.compare_to(before, "lineno")` lists the source lines whose allocations grew,
biggest first. Each `StatisticDiff` has `size_diff`, `count_diff` and a
`traceback`; `source_line` (from the setup cell) shows the code instead of a file
path.

In [48]:
tracemalloc.start()
try:
    before_snapshot = tracemalloc.take_snapshot()
    price_history = [{"sku": f"SKU-{number}", "price": number * 1.5} for number in range(50_000)]
    labels = [f"label {number}" for number in range(50_000)]
    after_snapshot = tracemalloc.take_snapshot()
finally:
    tracemalloc.stop()

ignore_tracemalloc = [tracemalloc.Filter(False, tracemalloc.__file__)]
growth = after_snapshot.filter_traces(ignore_tracemalloc).compare_to(
    before_snapshot.filter_traces(ignore_tracemalloc), "lineno"
)
for stat in growth[:2]:
    frame = stat.traceback[0]
    print(f"+{stat.size_diff / 1024:7.0f} KiB  +{stat.count_diff:>7,} blocks  {source_line(frame)}")
del price_history, labels

+  13011 KiB  +199,900 blocks  price_history = [{"sku": f"SKU-{number}", "price": number * 1.5} for number in range(50_000)]
+   2962 KiB  + 50,001 blocks  labels = [f"label {number}" for number in range(50_000)]


**Statistics, current and peak.** `snapshot.statistics("lineno")` ranks lines by
the memory they hold *now*. `tracemalloc.get_traced_memory()` returns
`(current, peak)` bytes, and `tracemalloc.reset_peak()` (3.9+) starts a new peak
measurement, handy for measuring one phase of a program.

In [49]:
tracemalloc.start()
try:
    scratch_buffers = [bytes(1_000) for _ in range(5_000)]
    current_bytes, peak_bytes = tracemalloc.get_traced_memory()
    print(
        f"holding 5 000 buffers: {current_bytes / 1e6:.2f} MB now, peak {peak_bytes / 1e6:.2f} MB"
    )
    top_stat = tracemalloc.take_snapshot().statistics("lineno")[0]
    top_source = source_line(top_stat.traceback[0])
    print(f"top line: {top_stat.size / 1e6:.2f} MB in {top_stat.count:,} blocks: {top_source}")
    del scratch_buffers
    current_bytes, peak_bytes = tracemalloc.get_traced_memory()
    print(
        f"after del:             {current_bytes / 1e6:.2f} MB now, peak {peak_bytes / 1e6:.2f} MB"
    )
    tracemalloc.reset_peak()
    print(f"after reset_peak():    peak {tracemalloc.get_traced_memory()[1] / 1e6:.2f} MB")
finally:
    tracemalloc.stop()

holding 5 000 buffers: 5.21 MB now, peak 5.21 MB
top line: 5.21 MB in 5,001 blocks: scratch_buffers = [bytes(1_000) for _ in range(5_000)]
after del:             0.01 MB now, peak 5.22 MB
after reset_peak():    peak 0.01 MB


**Grouping by traceback.** `tracemalloc.start(nframes)` keeps that many frames per
allocation, and `statistics("traceback")` groups by the whole call stack, which
tells you *which caller* drove the allocations, not just the line that made them.

In [50]:
def load_row(row_number: int) -> dict[str, object]:
    """Allocate one report row."""
    return {"row": row_number, "text": f"row {row_number} " * 3}


def build_report(row_count: int) -> list[dict[str, object]]:
    """Allocate many rows through load_row."""
    return [load_row(row_number) for row_number in range(row_count)]


tracemalloc.start(3)
try:
    report_rows = build_report(20_000)
    report_snapshot = tracemalloc.take_snapshot()
finally:
    tracemalloc.stop()
largest = report_snapshot.statistics("traceback")[0]
print(f"largest group: {largest.size / 1e6:.2f} MB in {largest.count:,} blocks; its call stack:")
for frame in largest.traceback:
    print(f"    {source_line(frame)}")
del report_rows

largest group: 5.06 MB in 59,914 blocks; its call stack:
    report_rows = build_report(20_000)
    return [load_row(row_number) for row_number in range(row_count)]
    return {"row": row_number, "text": f"row {row_number} " * 3}


**Process RSS.** `rss_bytes()` (setup cell) reads `VmRSS` from `/proc/self/status`
on Linux, with `psutil` as a fallback elsewhere. A large allocation (bigger than
512 bytes, so it skips pymalloc) gets its own `mmap` and is returned to the OS as
soon as it is freed, so RSS goes up **and back down**; compare with the small
objects in 3.1.9.

In [51]:
if rss_bytes() is None:
    print("Skipped: cannot read RSS on this platform (no /proc and no psutil)")
else:
    rss_start = rss_bytes()
    big_block = b"x" * (80 * 1024 * 1024)  # written byte by byte, so the pages are real
    rss_with_block = rss_bytes()
    del big_block
    rss_after = rss_bytes()
    print(f"RSS at start:          {rss_start / 1e6:6.0f} MB")
    print(f"RSS with an 80 MiB block: {rss_with_block / 1e6:3.0f} MB")
    print(f"RSS after del:         {rss_after / 1e6:6.0f} MB")
    print(interpreter_label())

RSS at start:             144 MB
RSS with an 80 MiB block: 228 MB
RSS after del:            144 MB
CPython 3.15.1, default (GIL) build, x86_64


**memray preview.** [memray](https://bloomberg.github.io/memray/) (Bloomberg)
traces every allocation, including those made by C extensions, and renders flame
graphs. It is a Phase 5 dependency; the cell checks whether it is installed. Typical
use:

```bash
uv run memray run -o report.bin my_script.py       # record
uv run memray flamegraph report.bin                # HTML flame graph
uv run memray run --native -o report.bin my_script.py   # include C frames
```

In Jupyter: `%load_ext memray`, then `%%memray_flamegraph` at the top of a cell.

In [52]:
if importlib.util.find_spec("memray") is None:
    print("Skipped: memray is not installed (it is added in Phase 5: uv add memray)")
else:
    import memray

    print(f"memray {memray.__version__} is installed; try %load_ext memray")

Skipped: memray is not installed (it is added in Phase 5: uv add memray)


### 3.1.12 Memory leaks in practice

In a garbage-collected language a "leak" means memory that is still
**reachable** but no longer **useful**: something keeps a reference you forgot
about. Six classic sources follow, each with a demo and a fix. A weak reference
to the victim proves whether it is still alive.

| Leak | What holds the reference | Fix |
| --- | --- | --- |
| Unbounded cache | a module-level dict that only grows | `lru_cache(maxsize=...)`, a bounded `OrderedDict`, TTLs |
| `lru_cache` on a method | the cache, keyed by `self` | per-instance cache, or cache a function of plain values |
| Captured closure | a stored callback's closure cell | capture only the values you need |
| Stored traceback | exception → traceback → frames → locals | store a string or `TracebackException`, or `traceback.clear_frames` |
| Unremoved listener | the publisher's listener list | unsubscribe (context manager) or weak references |
| C-extension leak | memory `tracemalloc` cannot see | RSS trend, memray `--native`, valgrind, debug builds |

#### Unbounded caches

A dict used as a cache, keyed by user input, grows forever.

In [53]:
_quote_cache: dict[str, float] = {}


def quote_price_unbounded(sku: str) -> float:
    """Cache every quote forever."""
    if sku not in _quote_cache:
        _quote_cache[sku] = len(sku) * 9.5
    return _quote_cache[sku]


@functools.lru_cache(maxsize=1_024)
def quote_price_bounded(sku: str) -> float:
    """Keep only the 1 024 most recently used quotes."""
    return len(sku) * 9.5


for number in range(100_000):
    quote_price_unbounded(f"SKU-{number}")
    quote_price_bounded(f"SKU-{number}")
print(f"{len(_quote_cache)=:,}  (one entry per distinct key, forever)")
print(f"{quote_price_bounded.cache_info()=}")
_quote_cache.clear()
quote_price_bounded.cache_clear()

len(_quote_cache)=100,000  (one entry per distinct key, forever)
quote_price_bounded.cache_info()=CacheInfo(hits=0, misses=100000, maxsize=1024, currsize=1024)


#### `lru_cache` on methods

`@lru_cache` (or `@cache`, which is `lru_cache(maxsize=None)`) on a method caches
on the **class**, and `self` is part of every key,
so the cache keeps every instance alive (ruff rule B019 warns about this). The fix
here gives each instance its own cache: the instance and its cache form a cycle
that the GC can free.

In [54]:
class PriceEngine:
    """Holds a big rate table and caches quotes the wrong way."""

    def __init__(self, region: str) -> None:
        self.region = region
        self.rate_table = bytearray(2_000_000)

    @functools.cache  # noqa: B019 (the leak being demonstrated)
    def quote(self, sku: str) -> float:
        return len(sku) * 2.0


class FixedPriceEngine:
    """Same engine with a per-instance cache."""

    def __init__(self, region: str) -> None:
        self.region = region
        self.rate_table = bytearray(2_000_000)
        self.quote = functools.lru_cache(maxsize=256)(self._quote)

    def _quote(self, sku: str) -> float:
        return len(sku) * 2.0


engine_refs: dict[str, weakref.ref[object]] = {}
for engine_class in (PriceEngine, FixedPriceEngine):
    engine = engine_class("south")
    engine.quote("TEA-01")
    engine_refs[engine_class.__name__] = weakref.ref(engine)
    del engine
    gc.collect()
    alive = engine_refs[engine_class.__name__]() is not None
    print(f"{engine_class.__name__:<16} alive after del + collect: {alive}")
PriceEngine.quote.cache_clear()
print(f"after cache_clear(): PriceEngine alive = {engine_refs['PriceEngine']() is not None}")

PriceEngine      alive after del + collect: True
FixedPriceEngine alive after del + collect: False
after cache_clear(): PriceEngine alive = False


> **Note:** `cache_clear()` releases the leaked `PriceEngine`. For a method without
> arguments, `functools.cached_property` stores the value on the instance and does
> not leak.

#### Captured closures

A closure keeps alive **every variable it uses** from the enclosing function. A
callback that only needs a row count but mentions the whole report keeps the whole
report.

In [55]:
class Report:
    """A report with many rows."""

    def __init__(self, row_count: int) -> None:
        self.rows = [{"row": number} for number in range(row_count)]


pending_callbacks: list[Callable[[], str]] = []


def schedule_summary_leaky(report: Report) -> None:
    """The callback captures the whole report."""
    pending_callbacks.append(lambda: f"{len(report.rows)} rows")


def schedule_summary_fixed(report: Report) -> None:
    """The callback captures only an int."""
    row_count = len(report.rows)
    pending_callbacks.append(lambda: f"{row_count} rows")


for schedule in (schedule_summary_leaky, schedule_summary_fixed):
    big_report = Report(50_000)
    report_ref = weakref.ref(big_report)
    schedule(big_report)
    del big_report
    captured = type(pending_callbacks[-1].__closure__[0].cell_contents).__name__
    alive = report_ref() is not None
    print(f"{schedule.__name__:<24} captures a {captured:<6} report alive: {alive}")
pending_callbacks.clear()

schedule_summary_leaky   captures a Report report alive: True
schedule_summary_fixed   captures a int    report alive: False


#### Stored tracebacks

An exception object holds its traceback, the traceback holds the frames, and the
frames hold **all their local variables**. Keeping exceptions in a list (for a
later error report) keeps every buffer those functions had in hand.
`traceback.clear_frames(tb)` drops the locals; storing
`traceback.TracebackException.from_exception(error)` (or a formatted string) keeps
the information without the frames.

In [56]:
class UploadBuffer:
    """A large in-memory upload."""

    def __init__(self) -> None:
        self.data = bytearray(5_000_000)


buffer_refs: list[weakref.ref[UploadBuffer]] = []
stored_errors: list[BaseException] = []


def parse_upload() -> None:
    """Fails while holding a big buffer in a local variable."""
    upload_buffer = UploadBuffer()
    buffer_refs.append(weakref.ref(upload_buffer))
    raise ValueError("bad header")


try:
    parse_upload()
except ValueError as error:
    stored_errors.append(error)  # kept for a later report
print(f"buffer alive while the exception is stored: {buffer_refs[-1]() is not None}")
traceback.clear_frames(stored_errors[-1].__traceback__)
print(f"after traceback.clear_frames:              {buffer_refs[-1]() is not None}")
summary = traceback.TracebackException.from_exception(stored_errors[-1])
print(f"still reportable: {''.join(summary.format_exception_only()).strip()}")
stored_errors.clear()

buffer alive while the exception is stored: True
after traceback.clear_frames:              False
still reportable: ValueError: bad header


#### Unremoved listeners

A publisher keeps a list of callbacks. A widget subscribes with a bound method
(which references the widget) and is "closed" but never unsubscribes: the
publisher keeps every widget alive. The fix below subscribes inside a context
manager that always unsubscribes. (Lab 3.1, exercise 5, fixes the same problem
with weak references instead.)

In [57]:
class PriceTicker:
    """Publishes prices to subscribed callbacks."""

    def __init__(self) -> None:
        self.listeners: list[Callable[[float], None]] = []

    def subscribe(self, listener: Callable[[float], None]) -> None:
        self.listeners.append(listener)

    def unsubscribe(self, listener: Callable[[float], None]) -> None:
        self.listeners.remove(listener)


class PriceWidget:
    """A screen widget with a big pixel buffer."""

    def __init__(self) -> None:
        self.pixels = bytearray(1_000_000)
        self.last_price: float | None = None

    def on_price(self, price: float) -> None:
        self.last_price = price


@contextmanager
def subscribed(ticker: PriceTicker, listener: Callable[[float], None]) -> Iterator[None]:
    """Subscribe for the duration of a with block."""
    ticker.subscribe(listener)
    try:
        yield
    finally:
        ticker.unsubscribe(listener)

Five widgets with the leak, then five with the context manager:

In [58]:
leaky_ticker = PriceTicker()
widget_refs = []
for _ in range(5):
    widget = PriceWidget()
    leaky_ticker.subscribe(widget.on_price)  # never unsubscribed
    widget_refs.append(weakref.ref(widget))
del widget
alive_count = sum(ref() is not None for ref in widget_refs)
print(f"leaky:  {len(leaky_ticker.listeners)} listeners, {alive_count} widgets alive")

fixed_ticker = PriceTicker()
widget_refs.clear()
for _ in range(5):
    widget = PriceWidget()
    with subscribed(fixed_ticker, widget.on_price):
        pass  # the widget is on screen here
    widget_refs.append(weakref.ref(widget))
del widget
alive_count = sum(ref() is not None for ref in widget_refs)
print(f"fixed:  {len(fixed_ticker.listeners)} listeners, {alive_count} widgets alive")
leaky_ticker.listeners.clear()

leaky:  5 listeners, 5 widgets alive
fixed:  0 listeners, 0 widgets alive


#### C-extension leaks

Memory allocated by C code with plain `malloc` bypasses Python's allocators, so
`tracemalloc` never sees it; only RSS grows. The cell simulates a C library that
allocates 64 MiB through `ctypes` (and frees it in `finally`, unlike a real leak).
When RSS climbs while `tracemalloc` stays flat, suspect native code. Tools for that
case: memray with `--native`, valgrind / heaptrack, and a CPython debug build,
where `sys.gettotalrefcount()` exposes reference leaks in extensions.

In [59]:
try:
    libc = ctypes.CDLL(None)  # the C library already loaded into this process
    libc.malloc.restype, libc.malloc.argtypes = ctypes.c_void_p, [ctypes.c_size_t]
    libc.free.argtypes = [ctypes.c_void_p]
    libc.memset.argtypes = [ctypes.c_void_p, ctypes.c_int, ctypes.c_size_t]
except (OSError, AttributeError) as error:
    libc = None
    print(f"Skipped: cannot load the C library here ({type(error).__name__})")

if libc is not None and rss_bytes() is not None:
    block_size = 64 * 1024 * 1024
    tracemalloc.start()
    rss_before, traced_before = rss_bytes(), tracemalloc.get_traced_memory()[0]
    native_block = libc.malloc(block_size)
    try:
        libc.memset(native_block, 1, block_size)  # touch the pages
        rss_growth = (rss_bytes() - rss_before) / 1e6
        traced_growth = (tracemalloc.get_traced_memory()[0] - traced_before) / 1e6
        print(f"RSS grew {rss_growth:.0f} MB; tracemalloc saw {traced_growth:.3f} MB")
    finally:
        libc.free(native_block)
        tracemalloc.stop()
print(f"{hasattr(sys, 'gettotalrefcount')=}  (True only on debug builds)")

RSS grew 67 MB; tracemalloc saw 0.001 MB
hasattr(sys, 'gettotalrefcount')=False  (True only on debug builds)


#### A diagnosis workflow

1. **Confirm the trend.** Plot or log RSS over hours. Steady growth under steady
   load is a leak; a plateau is just a warm cache or allocator slack.
2. **Reproduce in a loop.** Find a request or job you can run N times in a test
   or a REPL.
3. **Diff `tracemalloc` snapshots** between rounds (`compare_to(..., "lineno")`).
   Lines that grow **every** round are suspects; one-time growth is warm-up.
4. **Find the holder.** `statistics("traceback")` for the call path, then
   `gc.get_referrers(suspect)` (or objgraph's back-reference chains) to see which
   container keeps the objects.
5. **Fix and lock it in** with a test: a weak reference that must die, or a
   `tracemalloc` growth limit.
6. **If `tracemalloc` is flat but RSS grows**, look at native code and
   fragmentation: memray `--native`, valgrind, `sys._debugmallocstats()`.

The cell walks steps 2–4 on a request handler with a hidden leak.

In [60]:
_audit_log: list[object] = []  # the hidden holder


class OrderSession:
    """Per-request state with a payload."""

    def __init__(self, order_id: int) -> None:
        self.order_id = order_id
        self.payload = "x" * 2_000

    def close(self) -> None:
        _audit_log.append(self)  # bug: keeps every session forever


def handle_order(order_id: int) -> None:
    OrderSession(order_id).close()


tracemalloc.start()
try:
    round_snapshots = []
    for _ in range(3):
        for order_id in range(2_000):
            handle_order(order_id)
        round_snapshots.append(tracemalloc.take_snapshot())
finally:
    tracemalloc.stop()
for earlier, later in itertools.pairwise(round_snapshots):
    top = later.compare_to(earlier, "lineno")[0]
    print(f"round growth +{top.size_diff / 1024:.0f} KiB at: {source_line(top.traceback[0])}")
suspect = _audit_log[-1]
held_by_log = any(referrer is _audit_log for referrer in gc.get_referrers(suspect))
print(f"suspect is held by _audit_log: {held_by_log}")
del suspect
_audit_log.clear()

round growth +172 KiB at: OrderSession(order_id).close()
round growth +172 KiB at: OrderSession(order_id).close()
suspect is held by _audit_log: True


### 3.1.13 JS bridge

| Topic | V8 (Node.js, Chrome) | CPython |
| --- | --- | --- |
| Main mechanism | Tracing GC: finds live objects from roots, everything else is garbage | Reference counting frees objects as soon as the count is 0 |
| Generations | Young generation (scavenger, copies survivors) + old generation (mark-sweep-compact) | Generations only for the cycle collector; refcounting has no generations |
| Cycles | No problem for a tracing GC | Need the cyclic GC (3.1.6) |
| When memory is freed | Whenever the GC decides | At the `del` / return for acyclic objects |
| Object movement | Compaction moves objects | Objects never move; fragmentation possible (3.1.9) |
| Pauses | Incremental and concurrent marking | Short GC pauses for tracked containers only |
| Weak references | `WeakRef` + `deref()` | `weakref.ref` + call it |
| Finalization | `FinalizationRegistry`: the callback *may* run later, or never | `weakref.finalize`: runs at death (deterministic in CPython) and at exit |
| Weak maps | `WeakMap` (weak keys), `WeakSet` | `WeakKeyDictionary`, `WeakSet`, plus `WeakValueDictionary` |
| Heap tools | DevTools heap snapshots, `--inspect` | `tracemalloc`, `gc`, memray |

```javascript
// JavaScript: when (and whether) the callback runs is up to the engine
const registry = new FinalizationRegistry((label) => console.log(`${label} collected`));
let session = { user: "asha" };
const sessionRef = new WeakRef(session);
registry.register(session, "session");
session = null;            // nothing printed yet; maybe later, maybe never
console.log(sessionRef.deref()); // probably still the object
```

The Python version prints at the exact line where the last reference goes.

In [61]:
finalization_events: list[str] = []
session = Session("asha")
session_ref = weakref.ref(session)
weakref.finalize(session, finalization_events.append, "session collected")
session = None  # last reference gone: finalizer runs right here
finalization_events.append("next statement")
print(f"{finalization_events=}")
print(f"{session_ref()=}")

finalization_events=['session collected', 'next statement']
session_ref()=None


**Predict the output.**

```python
import weakref

class Node:
    pass

first, second = Node(), Node()
first.peer, second.peer = second, first
probe = weakref.ref(first)
del first, second
print(probe() is None)
```

<details><summary>Answer</summary>

```text
False
```

The two nodes form a cycle, so each keeps the other's count above zero after the
names are deleted. Only the cyclic GC can free them, and it has not run yet.
(In rare runs an automatic collection could happen between the two lines; the
proof disables the collector to make the result certain.) A JS developer expects
"maybe" here too, but for a different reason: V8 frees nothing until it decides
to collect.

</details>

Run the next cell to check your prediction.

In [62]:
class Node:
    """A node that will be part of a cycle."""


gc_was_enabled = gc.isenabled()
gc.disable()
try:
    first, second = Node(), Node()
    first.peer, second.peer = second, first
    probe = weakref.ref(first)
    del first, second
    print(probe() is None)
    gc.collect()
    print(f"after gc.collect(): {probe() is None}")
finally:
    if gc_was_enabled:
        gc.enable()

False
after gc.collect(): True


### Lab 3.1

Six exercises. For each, complete the stub, then run the check cell. Until then
the checks print FAIL; that is expected. Checks never compare against fixed sizes
or counts: expected values are measured live with `sys.getsizeof` and
`sys.getrefcount` on your interpreter.

#### Exercise 1 — `deep_getsizeof(obj)` with cycle handling

Return the total `sys.getsizeof` of `obj` and every object reachable from it
through:

- items of `list`, `tuple`, `set` and `frozenset`,
- keys **and** values of `dict`,
- the instance `__dict__` (count the dict itself too) and any `__slots__`
  attributes that are set (walk the class's `__mro__` for `__slots__`).

Count every object **once** (track `id()`s you have seen), so shared objects and
cycles are handled. Do not descend into classes, functions or modules. An
iterative version with an explicit stack avoids the recursion limit.

In [63]:
def deep_getsizeof(obj: object) -> int:
    """Total size of obj and everything reachable from it, each object counted once."""
    raise NotImplementedError("Lab 3.1: implement deep_getsizeof")

Run the checks. Expected values are computed with `sys.getsizeof` at check time;
objects are built inside the check functions.

In [64]:
class StockLine:
    """An instance with a __dict__."""

    def __init__(self, sku: str, quantity: int) -> None:
        self.sku = sku
        self.quantity = quantity


class SlottedStockLine:
    """An instance with __slots__."""

    __slots__ = ("sku", "quantity")

    def __init__(self, sku: str, quantity: int) -> None:
        self.sku = sku
        self.quantity = quantity


def check_flat_list() -> bool:
    first_text, second_text = "alpha" * 3, "beta" * 3
    items = [first_text, second_text]
    expected = sys.getsizeof(items) + sys.getsizeof(first_text) + sys.getsizeof(second_text)
    return deep_getsizeof(items) == expected


def check_shared_counted_once() -> bool:
    shared_text = "shared" * 20
    items = [shared_text, shared_text, shared_text]
    return deep_getsizeof(items) == sys.getsizeof(items) + sys.getsizeof(shared_text)


def check_dict_keys_and_values() -> bool:
    key_text, value_list = "key" * 5, [10_001]
    mapping = {key_text: value_list}
    expected = (
        sys.getsizeof(mapping) + sys.getsizeof(key_text)
        + sys.getsizeof(value_list) + sys.getsizeof(value_list[0])
    )  # fmt: skip
    return deep_getsizeof(mapping) == expected


def check_self_referencing_list() -> bool:
    looped: list[object] = [10_002]
    looped.append(looped)
    return deep_getsizeof(looped) == sys.getsizeof(looped) + sys.getsizeof(looped[0])


def check_instance_dict() -> bool:
    line = StockLine("TEA-" + "01", 10_003)
    total = deep_getsizeof(line)  # may materialize __dict__ first
    expected = (
        sys.getsizeof(line) + sys.getsizeof(vars(line))
        + sys.getsizeof("sku") + sys.getsizeof("quantity")
        + sys.getsizeof(line.sku) + sys.getsizeof(line.quantity)
    )  # fmt: skip
    return total == expected


def check_slots() -> bool:
    line = SlottedStockLine("RICE-" + "05", 10_004)
    expected = sys.getsizeof(line) + sys.getsizeof(line.sku) + sys.getsizeof(line.quantity)
    return deep_getsizeof(line) == expected


def check_two_object_cycle() -> bool:
    first, second = StockLine("A", 1), StockLine("B", 2)
    first.peer, second.peer = second, first
    return deep_getsizeof(first) == deep_getsizeof(second)


run_checks(
    [
        ("int equals getsizeof", lambda: deep_getsizeof(10**20) == sys.getsizeof(10**20), True),
        ("flat list", check_flat_list, True),
        ("shared object counted once", check_shared_counted_once, True),
        ("dict keys and values", check_dict_keys_and_values, True),
        ("self-referencing list", check_self_referencing_list, True),
        ("instance __dict__", check_instance_dict, True),
        ("__slots__ attributes", check_slots, True),
        ("two-object cycle", check_two_object_cycle, True),
        ("tuple of sets", lambda: deep_getsizeof(({1}, {1})) > sys.getsizeof(({1}, {1})), True),
    ]
)

FAIL  int equals getsizeof: raised NotImplementedError: Lab 3.1: implement deep_getsizeof
FAIL  flat list: raised NotImplementedError: Lab 3.1: implement deep_getsizeof
FAIL  shared object counted once: raised NotImplementedError: Lab 3.1: implement deep_getsizeof
FAIL  dict keys and values: raised NotImplementedError: Lab 3.1: implement deep_getsizeof
FAIL  self-referencing list: raised NotImplementedError: Lab 3.1: implement deep_getsizeof
FAIL  instance __dict__: raised NotImplementedError: Lab 3.1: implement deep_getsizeof
FAIL  __slots__ attributes: raised NotImplementedError: Lab 3.1: implement deep_getsizeof
FAIL  two-object cycle: raised NotImplementedError: Lab 3.1: implement deep_getsizeof
FAIL  tuple of sets: raised NotImplementedError: Lab 3.1: implement deep_getsizeof
0/9 checks passed


#### Exercise 2 — predict six reference counts, then verify

Each scenario starts from one object referenced by one local variable and does one
thing. Predict how much `sys.getrefcount(target)` changes compared with the start
(`+1`, `+2`, `0`, …) **before** running the check. The check measures the real
changes on your interpreter.

| Key | What happens |
| --- | --- |
| `"second name"` | `alias = target` |
| `"list append"` | `holder = []; holder.append(target)` |
| `"dict key and value"` | `mapping = {target: target}` |
| `"tuple twice"` | `pair = (target, target)` |
| `"weak reference"` | `ref = weakref.ref(target)` |
| `"bind then del"` | `temporary = target; del temporary` |

Return a dict with those six keys and your predicted change for each.

In [65]:
def predict_refcount_deltas() -> dict[str, int]:
    """Your predicted change in sys.getrefcount(target) for each scenario."""
    raise NotImplementedError("Lab 3.1: fill in your six predictions")

Run the checks. `measure_refcount_deltas()` performs each scenario inside a function
and measures the change (changes are reliable even where borrowed references make
the absolute count unusual).

In [66]:
class RefcountTarget:
    """The object whose count is measured."""


def measure_refcount_deltas() -> dict[str, int]:
    """Measure the change in sys.getrefcount(target) for each scenario."""
    target = RefcountTarget()
    start = sys.getrefcount(target)
    measured: dict[str, int] = {}
    alias = target
    measured["second name"] = sys.getrefcount(target) - start
    del alias
    holder = []
    holder.append(target)
    measured["list append"] = sys.getrefcount(target) - start
    del holder
    mapping = {target: target}
    measured["dict key and value"] = sys.getrefcount(target) - start
    del mapping
    pair = (target, target)
    measured["tuple twice"] = sys.getrefcount(target) - start
    del pair
    ref = weakref.ref(target)
    measured["weak reference"] = sys.getrefcount(target) - start
    del ref
    temporary = target
    del temporary
    measured["bind then del"] = sys.getrefcount(target) - start
    return measured


def predicted_delta(scenario: str) -> int:
    """The learner's prediction for one scenario."""
    return predict_refcount_deltas()[scenario]


measured_deltas = measure_refcount_deltas()
print(f"[{interpreter_label()}]")
run_checks(
    [("all six keys answered", lambda: sorted(predict_refcount_deltas()), sorted(measured_deltas))]
    + [
        (f"{scenario}: measured {delta:+d}", functools.partial(predicted_delta, scenario), delta)
        for scenario, delta in measured_deltas.items()
    ]
)

[CPython 3.15.1, default (GIL) build, x86_64]
FAIL  all six keys answered: raised NotImplementedError: Lab 3.1: fill in your six predictions
FAIL  second name: measured +1: raised NotImplementedError: Lab 3.1: fill in your six predictions
FAIL  list append: measured +1: raised NotImplementedError: Lab 3.1: fill in your six predictions
FAIL  dict key and value: measured +2: raised NotImplementedError: Lab 3.1: fill in your six predictions
FAIL  tuple twice: measured +2: raised NotImplementedError: Lab 3.1: fill in your six predictions
FAIL  weak reference: measured +0: raised NotImplementedError: Lab 3.1: fill in your six predictions
FAIL  bind then del: measured +0: raised NotImplementedError: Lab 3.1: fill in your six predictions
0/7 checks passed


#### Exercise 3 — prove a cycle is freed only by `gc.collect`

Write `cycle_lifecycle()`:

1. Remember whether the collector is enabled, then `gc.disable()`.
2. Build two `CycleNode` objects that reference each other, and keep only a
   `weakref.ref` to one of them.
3. Delete both names and record whether the object is still alive.
4. Call `gc.collect()` and record again.
5. In a `finally` block, re-enable the collector **only if it was enabled before**.

Return `(alive_after_del, alive_after_collect)`. Also write
`acyclic_lifecycle()`, which does steps 1–5 with two nodes that do **not** form a
cycle (only `first.peer = second`), and returns the same pair.

In [67]:
class CycleNode:
    """A node with an optional peer."""

    def __init__(self, name: str) -> None:
        self.name = name
        self.peer: CycleNode | None = None


def cycle_lifecycle() -> tuple[bool, bool]:
    """(alive after del, alive after gc.collect) for a two-node cycle."""
    raise NotImplementedError("Lab 3.1: implement cycle_lifecycle")


def acyclic_lifecycle() -> tuple[bool, bool]:
    """The same measurements for two nodes without a cycle."""
    raise NotImplementedError("Lab 3.1: implement acyclic_lifecycle")

Run the checks. The last two confirm the collector's state is restored either way.

In [68]:
def set_collector(enabled: bool) -> None:
    """Switch the cyclic GC on or off."""
    if enabled:
        gc.enable()
    else:
        gc.disable()


def state_restored_after(function: Callable[[], object], start_enabled: bool) -> bool:
    """Run function with the collector on or off and check the state is unchanged."""
    was_enabled = gc.isenabled()
    set_collector(start_enabled)
    try:
        function()
        return gc.isenabled() == start_enabled
    finally:
        set_collector(was_enabled)


run_checks(
    [
        ("cycle survives del, freed by collect", cycle_lifecycle, (True, False)),
        ("no cycle: freed at del", acyclic_lifecycle, (False, False)),
        ("collector stays enabled", lambda: state_restored_after(cycle_lifecycle, True), True),
        ("collector stays disabled", lambda: state_restored_after(cycle_lifecycle, False), True),
    ]
)

FAIL  cycle survives del, freed by collect: raised NotImplementedError: Lab 3.1: implement cycle_lifecycle
FAIL  no cycle: freed at del: raised NotImplementedError: Lab 3.1: implement acyclic_lifecycle
FAIL  collector stays enabled: raised NotImplementedError: Lab 3.1: implement cycle_lifecycle
FAIL  collector stays disabled: raised NotImplementedError: Lab 3.1: implement cycle_lifecycle
0/4 checks passed


#### Exercise 4 — an `ImageCache` on `WeakValueDictionary`

Decoding images is slow, so the app caches them, but the cache must not keep
images alive on its own: an image should be freed as soon as no screen uses it.

`ImageCache(loader)`:

- `get(path)` returns the cached `Image` if one is still alive, otherwise calls
  `loader(path)` once, stores the result **weakly** and returns it.
- `load_count` (attribute or property): how many times `loader` was called.
- `len(cache)`: number of **live** entries; `path in cache`: is a live image
  cached for that path.

`Image` and `decode_image` are given.

In [69]:
class Image:
    """A decoded image (given)."""

    def __init__(self, path: str, pixels: bytes) -> None:
        self.path = path
        self.pixels = pixels


def decode_image(path: str) -> Image:
    """Slow decoder stand-in (given)."""
    return Image(path, bytes(10_000))


class ImageCache:
    """Caches decoded images without keeping them alive."""

    def __init__(self, loader: Callable[[str], Image]) -> None:
        raise NotImplementedError("Lab 3.1: implement ImageCache")

Run the checks. Each check builds its own cache and images inside a function.

In [70]:
def cache_reuses_live_image() -> tuple[bool, int]:
    cache = ImageCache(decode_image)
    first = cache.get("logo.png")
    second = cache.get("logo.png")
    return first is second, cache.load_count


def cache_forgets_dead_image() -> tuple[int, bool, int]:
    cache = ImageCache(decode_image)
    logo = cache.get("logo.png")
    del logo
    return len(cache), "logo.png" in cache, cache.load_count


def cache_reloads_after_death() -> int:
    cache = ImageCache(decode_image)
    cache.get("logo.png")  # result not kept: dies at once
    cache.get("logo.png")
    return cache.load_count


def cache_does_not_keep_alive() -> bool:
    cache = ImageCache(decode_image)
    banner = cache.get("banner.png")
    banner_ref = weakref.ref(banner)
    del banner
    return banner_ref() is None


def cache_counts_live_entries() -> tuple[int, bool]:
    cache = ImageCache(decode_image)
    kept = [cache.get(name) for name in ("a.png", "b.png", "c.png")]
    cache.get("d.png")
    return len(cache), all(name in cache for name in ("a.png", "b.png", "c.png")) and len(kept) == 3


run_checks(
    [
        ("same object while alive, one load", cache_reuses_live_image, (True, 1)),
        ("entry vanishes when the image dies", cache_forgets_dead_image, (0, False, 1)),
        ("reloads after the image died", cache_reloads_after_death, 2),
        ("cache does not keep images alive", cache_does_not_keep_alive, True),
        ("len counts only live entries", cache_counts_live_entries, (3, True)),
    ]
)

FAIL  same object while alive, one load: raised NotImplementedError: Lab 3.1: implement ImageCache
FAIL  entry vanishes when the image dies: raised NotImplementedError: Lab 3.1: implement ImageCache
FAIL  reloads after the image died: raised NotImplementedError: Lab 3.1: implement ImageCache
FAIL  cache does not keep images alive: raised NotImplementedError: Lab 3.1: implement ImageCache
FAIL  len counts only live entries: raised NotImplementedError: Lab 3.1: implement ImageCache
0/5 checks passed


#### Exercise 5 — find and fix a leaking `EventEmitter`

`LeakyEventEmitter` (given) stores every callback strongly. Subscribers that
register a bound method can never be freed, as the demo below shows.

Write `EventEmitter` that fixes it:

- `on(event, callback)` subscribes and returns a function that unsubscribes.
  **Bound methods** are stored as `weakref.WeakMethod`, so the emitter does not
  keep their instance alive. Other callables (plain functions, lambdas) are stored
  strongly, otherwise a lambda would die immediately.
- `emit(event, *args)` calls the live listeners in subscription order, drops dead
  ones, and returns how many were called.
- `listener_count(event)` returns the number of **live** listeners.

In [71]:
class LeakyEventEmitter:
    """Stores every callback strongly (given)."""

    def __init__(self) -> None:
        self.listeners: dict[str, list[Callable[..., object]]] = {}

    def on(self, event: str, callback: Callable[..., object]) -> None:
        self.listeners.setdefault(event, []).append(callback)

    def emit(self, event: str, *args: object) -> int:
        for callback in self.listeners.get(event, []):
            callback(*args)
        return len(self.listeners.get(event, []))


class StockAlertPanel:
    """A subscriber with a bound-method callback (given)."""

    def __init__(self) -> None:
        self.alerts: list[str] = []
        self.screen_buffer = bytearray(500_000)

    def on_low_stock(self, sku: str) -> None:
        self.alerts.append(sku)


leaky_emitter = LeakyEventEmitter()
panel = StockAlertPanel()
leaky_emitter.on("low_stock", panel.on_low_stock)
panel_ref = weakref.ref(panel)
del panel  # the screen closed...
print(f"panel still alive: {panel_ref() is not None}  (the emitter holds it)")
print(f"emit still reaches it: {leaky_emitter.emit('low_stock', 'TEA-01')} listener(s)")
leaky_emitter.listeners.clear()

panel still alive: True  (the emitter holds it)
emit still reaches it: 1 listener(s)


Now the stub:

In [72]:
class EventEmitter:
    """An emitter that does not keep bound-method subscribers alive."""

    def __init__(self) -> None:
        raise NotImplementedError("Lab 3.1: implement EventEmitter")

Run the checks.

In [73]:
def emitter_calls_functions_and_methods() -> tuple[int, list[str], list[str]]:
    emitter = EventEmitter()
    seen: list[str] = []
    panel = StockAlertPanel()
    emitter.on("low_stock", seen.append)
    emitter.on("low_stock", panel.on_low_stock)
    called = emitter.emit("low_stock", "TEA-01")
    return called, seen, panel.alerts


def emitter_lets_subscriber_die() -> tuple[bool, int, int]:
    emitter = EventEmitter()
    panel = StockAlertPanel()
    emitter.on("low_stock", panel.on_low_stock)
    panel_ref = weakref.ref(panel)
    del panel
    return panel_ref() is None, emitter.listener_count("low_stock"), emitter.emit("low_stock", "X")


def emitter_keeps_lambdas() -> tuple[int, int]:
    emitter = EventEmitter()
    seen: list[str] = []
    emitter.on("restock", lambda sku: seen.append(sku))
    return emitter.listener_count("restock"), emitter.emit("restock", "RICE-5")


def emitter_unsubscribe() -> tuple[int, int]:
    emitter = EventEmitter()
    unsubscribe = emitter.on("restock", print)
    before = emitter.listener_count("restock")
    unsubscribe()
    return before, emitter.listener_count("restock")


def emitter_order() -> list[str]:
    emitter = EventEmitter()
    calls: list[str] = []
    emitter.on("tick", lambda: calls.append("first"))
    emitter.on("tick", lambda: calls.append("second"))
    emitter.emit("tick")
    return calls


run_checks(
    [
        (
            "functions and bound methods are called",
            emitter_calls_functions_and_methods,
            (2, ["TEA-01"], ["TEA-01"]),
        ),
        ("subscriber freed, listener dropped", emitter_lets_subscriber_die, (True, 0, 0)),
        ("lambdas are kept", emitter_keeps_lambdas, (1, 1)),
        ("on() returns an unsubscribe function", emitter_unsubscribe, (1, 0)),
        ("subscription order", emitter_order, ["first", "second"]),
        ("unknown event", lambda: EventEmitter().emit("nothing"), 0),
        ("unknown event count", lambda: EventEmitter().listener_count("nothing"), 0),
    ]
)

FAIL  functions and bound methods are called: raised NotImplementedError: Lab 3.1: implement EventEmitter
FAIL  subscriber freed, listener dropped: raised NotImplementedError: Lab 3.1: implement EventEmitter
FAIL  lambdas are kept: raised NotImplementedError: Lab 3.1: implement EventEmitter
FAIL  on() returns an unsubscribe function: raised NotImplementedError: Lab 3.1: implement EventEmitter
FAIL  subscription order: raised NotImplementedError: Lab 3.1: implement EventEmitter
FAIL  unknown event: raised NotImplementedError: Lab 3.1: implement EventEmitter
FAIL  unknown event count: raised NotImplementedError: Lab 3.1: implement EventEmitter
0/7 checks passed


#### Exercise 6 — diagnose a leaking long-running worker with `tracemalloc`

`LeakyWorker` (given) handles requests forever, and its memory grows. Find the
line with `tracemalloc`, then fix it.

1. `top_growth_line(workload, *, repeat=3)`:
   - start `tracemalloc` if it is not already tracing (and stop it at the end
     only if you started it),
   - run `workload()` once as a warm-up, take a snapshot, run it `repeat` more
     times, take another snapshot,
   - compare with `"lineno"`, ignoring frames from the `tracemalloc` module
     itself (`tracemalloc.Filter(False, tracemalloc.__file__)`),
   - return `(source_text, size_diff)` for the line with the largest positive
     growth, where `source_text` is the stripped source line (`source_line` from
     the setup cell does this).
2. `BoundedWorker(max_audit=100)`: a subclass of `LeakyWorker` whose `audit_trail`
   keeps only the newest `max_audit` entries (hint: `deque(maxlen=...)`), without
   changing `handle`.

In [74]:
class LeakyWorker:
    """A long-running request handler (given). One line in it leaks."""

    def __init__(self) -> None:
        self.processed_count = 0
        self.recent_ids: list[int] = []
        self.audit_trail: list[str] = []

    def handle(self, request_id: int) -> str:
        payload = f"request {request_id}: " + "x" * 200
        checksum = sum(payload.encode()) % 997
        self.recent_ids = self.recent_ids[-9:] + [request_id]
        self.audit_trail.append(f"{request_id}:{checksum}:" + payload)
        self.processed_count += 1
        return f"ok {checksum}"


def top_growth_line(workload: Callable[[], object], *, repeat: int = 3) -> tuple[str, int]:
    """(source line, bytes) of the line whose allocations grew most across repeats."""
    raise NotImplementedError("Lab 3.1: implement top_growth_line")


class BoundedWorker(LeakyWorker):
    """LeakyWorker whose audit trail keeps only the newest entries."""

    def __init__(self, max_audit: int = 100) -> None:
        raise NotImplementedError("Lab 3.1: implement BoundedWorker")

Run the checks. The growth checks compare the leaky and bounded workers with each
other, not with fixed byte counts.

In [75]:
def worker_workload(worker: LeakyWorker) -> Callable[[], None]:
    """A workload that sends 2 000 requests to worker."""
    request_ids = itertools.count()

    def run_batch() -> None:
        for _ in range(2_000):
            worker.handle(next(request_ids))

    return run_batch


def leaky_line_found() -> bool:
    line_text, _size = top_growth_line(worker_workload(LeakyWorker()))
    return "self.audit_trail.append" in line_text


def leaky_growth_is_large() -> bool:
    _line, size_diff = top_growth_line(worker_workload(LeakyWorker()))
    return size_diff > 500_000


def tracing_left_as_found() -> bool:
    top_growth_line(worker_workload(LeakyWorker()), repeat=1)
    return tracemalloc.is_tracing()


def bounded_keeps_newest() -> tuple[int, str, int]:
    worker = BoundedWorker(max_audit=100)
    for request_id in range(5_000):
        worker.handle(request_id)
    return len(worker.audit_trail), worker.audit_trail[-1].split(":")[0], worker.processed_count


def bounded_growth_is_small() -> bool:
    _line, leaky_diff = top_growth_line(worker_workload(LeakyWorker()))
    _line, bounded_diff = top_growth_line(worker_workload(BoundedWorker()))
    return bounded_diff * 10 < leaky_diff


run_checks(
    [
        ("finds the leaking line", leaky_line_found, True),
        ("growth over 500 kB for 6 000 requests", leaky_growth_is_large, True),
        ("tracemalloc stopped again", tracing_left_as_found, False),
        ("bounded audit trail", bounded_keeps_newest, (100, "4999", 5_000)),
        ("bounded worker grows 10x less", bounded_growth_is_small, True),
    ]
)

FAIL  finds the leaking line: raised NotImplementedError: Lab 3.1: implement top_growth_line
FAIL  growth over 500 kB for 6 000 requests: raised NotImplementedError: Lab 3.1: implement top_growth_line
FAIL  tracemalloc stopped again: raised NotImplementedError: Lab 3.1: implement top_growth_line
FAIL  bounded audit trail: raised NotImplementedError: Lab 3.1: implement BoundedWorker
FAIL  bounded worker grows 10x less: raised NotImplementedError: Lab 3.1: implement top_growth_line
0/5 checks passed


### Recap — Unit 3.1

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Memory model | Frames hold pointer-sized references to objects on a private heap | Same idea, minus V8's unboxed Smis |
| Object header | `ob_refcnt` + `ob_type` (+ `ob_size`), GC header before containers | Hidden-class pointer + fields |
| Reference counting | Freed the moment the count hits 0; `getrefcount` is approximate (+1, borrowing in 3.14+) | — (tracing only) |
| Immortal objects | `None`, small ints, interned literals never change count (3.12+) | Read-only space roots |
| Free-threaded refcounting | Biased (owner thread vs shared) + deferred counts, mimalloc | — |
| Cyclic GC | Finds groups referenced only from inside; generations; `gc.freeze` before fork | The whole GC |
| Finalizers | `__del__` runs once, swallows errors, can resurrect; prefer `with` + `weakref.finalize` | `FinalizationRegistry` (non-deterministic) |
| Weak references | `ref`, `WeakValueDictionary`, `WeakKeyDictionary`, `WeakSet`, `WeakMethod` | `WeakRef`, `WeakMap`, `WeakSet` |
| Allocator | pymalloc arenas → pools → blocks for ≤ 512 B; RSS stays while any block lives | Heap spaces, compaction |
| Free lists and caches | Reused memory, small ints, singletons, interning; `id()` is reused | — |
| Measuring | `getsizeof` is shallow; `tracemalloc` diffs; RSS for native memory; memray | DevTools heap snapshots |
| Leaks | Something reachable you forgot: caches, `lru_cache` on methods, closures, tracebacks, listeners, C code | Detached DOM nodes, listeners, closures |

**Further reading**

- [`gc`](https://docs.python.org/3/library/gc.html),
  [`weakref`](https://docs.python.org/3/library/weakref.html),
  [`tracemalloc`](https://docs.python.org/3/library/tracemalloc.html),
  [`sys.getsizeof`](https://docs.python.org/3/library/sys.html#sys.getsizeof),
  [`sys._is_immortal`](https://docs.python.org/3/library/sys.html#sys._is_immortal)
- [Memory management (C API)](https://docs.python.org/3/c-api/memory.html),
  [Python support for free threading](https://docs.python.org/3/howto/free-threading-python.html),
  [Garbage collector design (CPython internals)](https://github.com/python/cpython/blob/main/InternalDocs/garbage_collector.md)
- [PEP 442 — Safe object finalization](https://peps.python.org/pep-0442/),
  [PEP 445 — APIs to customize memory allocators](https://peps.python.org/pep-0445/),
  [PEP 454 — tracemalloc](https://peps.python.org/pep-0454/),
  [PEP 683 — Immortal objects](https://peps.python.org/pep-0683/),
  [PEP 703 — Making the GIL optional](https://peps.python.org/pep-0703/),
  [PEP 779 — Supported status for free-threaded Python](https://peps.python.org/pep-0779/)

## Unit 3.2 — Bytecode and the Interpreter

**Objectives**

- Name the parts of CPython and follow source code through tokens, the AST, the
  compiler and the evaluation loop.
- Inspect and rebuild **code objects** and **function objects**.
- Read `dis` output fluently: columns, the value stack, loads and stores, calls,
  jumps and loops.
- Recognize compiler optimizations, the specializing adaptive interpreter
  (PEP 659) and the JIT (PEP 744) in real output.
- Understand what the GIL and the eval breaker do between instructions.
- Use `compile`, `exec`, `eval` and `ast` safely, and transform code with
  `NodeTransformer`.
- Read a `.pyc` file byte by byte.

**JS bridge**

| JavaScript (V8) | Python (CPython) | The difference that bites |
| --- | --- | --- |
| Parser → AST → Ignition bytecode | Tokenizer + PEG parser → AST → compiler → bytecode | Python's bytecode is easy to see: `dis` is in the standard library |
| Ignition (register-based interpreter) | Stack-based evaluation loop (`Python/ceval.c`) | Instructions push and pop a value stack instead of naming registers |
| Inline caches + hidden classes (maps) | Specializing adaptive interpreter: inline caches keyed on type versions | Same idea (guess types from history, guard, fall back), different names |
| Sparkplug, Maglev, TurboFan (optimizing JITs) | Tier-2 micro-ops + copy-and-patch JIT (experimental, off by default) | CPython's JIT is young; most speed comes from specialization |
| `eval`, `new Function(...)` | `eval`, `exec`, `compile` | Same danger: never on untrusted input |
| Babel / ESLint plugins (AST transforms) | `ast.NodeVisitor`, `ast.NodeTransformer` | AST tools are built in; pytest rewrites your asserts this way |
| V8 code cache | `.pyc` files in `__pycache__` | `.pyc` holds marshalled code objects, not machine code |

> **Note:** `dis` output changes in **every** Python release: instruction names,
> arguments, caches and offsets. Each listing below prints the interpreter version
> above it; on another version your listing will differ, the concepts will not.

The next cell is this unit's setup. It defines `show_dis(target, **options)`, which
prints `dis.dis` output under a version banner with code-object addresses removed,
and `opnames(function, adaptive=False)`.

In [76]:
import ast
import compileall
import contextlib
import dis
import functools
import importlib.machinery
import importlib.util
import inspect
import io
import marshal
import os
import platform
import py_compile
import re
import shutil
import signal
import struct
import subprocess
import sys
import sysconfig
import tempfile
import textwrap
import threading
import time
import tokenize
import types
from collections import Counter
from collections.abc import Callable
from pathlib import Path

CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0"}  # plain output from child interpreters


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to any dis listing."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"


_CODE_OBJECT_REPR = re.compile(r'<code object (\S+) at 0x[0-9a-f]+, file "[^"]*", line \d+>')


def show_dis(target: object, **options: object) -> None:
    """dis.dis(target) under a version banner, without code-object addresses or file paths."""
    buffer = io.StringIO()
    dis.dis(target, file=buffer, **options)
    print(f"# dis on {interpreter_label()} (instruction names change between versions)")
    print(_CODE_OBJECT_REPR.sub(r"<code object \1>", buffer.getvalue()).rstrip())


def opnames(function: Callable[..., object], *, adaptive: bool = False) -> list[str]:
    """Instruction names of a function (adaptive=True shows specialized forms)."""
    return [instruction.opname for instruction in dis.get_instructions(function, adaptive=adaptive)]


print(interpreter_label())

CPython 3.15.1, default (GIL) build, x86_64


### 3.2.1 CPython architecture tour

CPython is a C program. Its source tree maps onto the pipeline you met in
Unit 1.1:

```mermaid
flowchart LR
    S["source text"] --> T["tokenizer<br/>Parser/lexer"]
    T --> P["PEG parser<br/>Parser/parser.c"]
    P --> A["AST<br/>Python/ast.c"]
    A --> C["symbol table + compiler<br/>Python/symtable.c, compile.c,<br/>flowgraph.c, assemble.c"]
    C --> K["code object"]
    K --> E["eval loop<br/>Python/ceval.c + bytecodes.c"]
    E --> O["objects<br/>Objects/*.c"]
```

<details><summary>ASCII fallback</summary>

```text
source -> tokenizer (Parser/lexer) -> PEG parser (Parser/parser.c) -> AST (Python/ast.c)
       -> symtable + compiler (Python/symtable.c, compile.c, flowgraph.c, assemble.c)
       -> code object -> eval loop (Python/ceval.c, bytecodes.c) -> objects (Objects/*.c)
```

</details>

| Directory / file | What lives there |
| --- | --- |
| `Grammar/python.gram` | The PEG grammar; `Parser/parser.c` is **generated** from it |
| `Parser/` | Tokenizer and parser |
| `Python/compile.c`, `flowgraph.c`, `assemble.c` | AST → control-flow graph → optimized bytecode |
| `Python/bytecodes.c` | Every instruction, written in a small DSL; generates `generated_cases.c.h` for the eval loop |
| `Python/ceval.c` | The evaluation loop: runs frames instruction by instruction |
| `Python/specialize.c` | The specializing adaptive interpreter (3.2.8) |
| `Python/optimizer*.c`, `Python/jit.c` | Tier-2 micro-ops and the JIT (3.2.9) |
| `Python/gc.c`, `Objects/obmalloc.c` | Cyclic GC and pymalloc (Unit 3.1) |
| `Objects/` | Built-in types: `listobject.c`, `dictobject.c`, `longobject.c`, `unicodeobject.c`, `typeobject.c` |
| `Modules/` | C extension modules: `mathmodule.c`, `_json.c`, `_asynciomodule.c` |
| `Lib/` | The standard library written in Python (`json/`, `dis.py`, `ast.py`) |
| `Include/` | Public and internal C headers (`object.h` defines `PyObject`) |

The cell traces where some familiar names come from: compiled into the interpreter
(`built-in`), a C extension file, or Python source in `Lib/`. Only file **names**
are printed.

In [77]:
def module_origin(module_name: str) -> str:
    """Where a module's code comes from: built in, C extension, frozen, or Python source."""
    spec = importlib.util.find_spec(module_name)
    if spec is None or spec.origin is None:
        return "not found"
    if spec.origin in {"built-in", "frozen"}:
        return f"{spec.origin} (compiled into the interpreter)"
    file_name = Path(spec.origin).name
    if spec.origin.endswith(tuple(importlib.machinery.EXTENSION_SUFFIXES)):
        return f"C extension file {file_name}"
    return f"Python source {Path(spec.origin).parent.name}/{file_name}"


print(f"{sys.version=}")
print(f"{platform.python_compiler()=}")
for module_name in ("sys", "builtins", "gc", "_opcode", "math", "_json", "json", "dis", "ast"):
    print(f"{module_name:<9} {module_origin(module_name)}")
print(f"{type(len).__name__=}  (Python/bltinmodule.c)")
print(f"{type(list.append).__name__=}  (Objects/listobject.c)")

sys.version='3.15.1 (main, Oct 21 2026, 17:48:50) [Clang 23.1.3 ]'
platform.python_compiler()='Clang 23.1.3 '
sys       built-in (compiled into the interpreter)
builtins  built-in (compiled into the interpreter)
gc        built-in (compiled into the interpreter)
_opcode   built-in (compiled into the interpreter)
math      built-in (compiled into the interpreter)
_json     built-in (compiled into the interpreter)
json      Python source json/__init__.py
dis       Python source python3.15/dis.py
ast       Python source python3.15/ast.py
type(len).__name__='builtin_function_or_method'  (Python/bltinmodule.c)
type(list.append).__name__='method_descriptor'  (Objects/listobject.c)


### 3.2.2 Tokens and the AST

**Tokens.** The tokenizer turns text into a stream of typed tokens. The
`tokenize` module exposes the same tokenizer CPython uses (including comments,
`NEWLINE`, `INDENT` and `DEDENT`, which the parser needs because indentation is
syntax).

In [78]:
invoice_source = textwrap.dedent(
    """
    if quantity > 10:  # bulk order
        total = price * quantity * 0.9
    """
).lstrip()
for token in tokenize.generate_tokens(io.StringIO(invoice_source).readline):
    print(f"{tokenize.tok_name[token.type]:<9} {token.string!r}")

NAME      'if'
NAME      'quantity'
OP        '>'
NUMBER    '10'
OP        ':'
COMMENT   '# bulk order'
NEWLINE   '\n'
INDENT    '    '
NAME      'total'
OP        '='
NAME      'price'
OP        '*'
NAME      'quantity'
OP        '*'
NUMBER    '0.9'
NEWLINE   '\n'
DEDENT    ''
ENDMARKER ''


**`ast.parse` and `ast.dump`.** The parser builds an **abstract syntax tree**:
comments and most punctuation are gone, structure remains. `ast.dump(tree,
indent=2)` prints it.

In [79]:
tree = ast.parse("total = price * quantity")
print(ast.dump(tree, indent=2))

Module(
  body=[
    Assign(
      targets=[
        Name(id='total', ctx=Store())],
      value=BinOp(
        left=Name(id='price'),
        op=Mult(),
        right=Name(id='quantity')))])


**Node types.** Every node is an instance of a class in `ast`. The common ones:

| Node | Example source | Key fields |
| --- | --- | --- |
| `Module` | a whole file | `body` |
| `FunctionDef` | `def f(x): ...` | `name`, `args`, `body`, `decorator_list` |
| `Assign` | `x = 1` | `targets`, `value` |
| `BinOp` | `a * b` | `left`, `op`, `right` |
| `Compare` | `a > b` | `left`, `ops`, `comparators` |
| `Call` | `f(x, key=1)` | `func`, `args`, `keywords` |
| `Name` | `price` | `id`, `ctx` (`Load` / `Store` / `Del`) |
| `Constant` | `0.9`, `"INR"` | `value` |
| `Attribute` | `order.total` | `value`, `attr` |
| `If`, `For`, `While`, `Return`, `Try` | statements | `test` / `target` / `iter` / `body` / `handlers` |

Every node also records where it came from (`lineno`, `col_offset`,
`end_lineno`, `end_col_offset`). `ast.walk` visits every node.

In [80]:
pricing_source = textwrap.dedent(
    """
    def order_total(lines, tax_rate=0.18):
        subtotal = sum(line.price * line.quantity for line in lines)
        if subtotal > 1000:
            subtotal = subtotal * 0.95
        return round(subtotal * (1 + tax_rate), 2)
    """
)
pricing_tree = ast.parse(pricing_source)
node_counts = Counter(type(node).__name__ for node in ast.walk(pricing_tree))
print(dict(node_counts.most_common(8)))
function_node = pricing_tree.body[0]
line_span = f"{function_node.lineno}-{function_node.end_lineno}"
print(f"{type(function_node).__name__} {function_node.name!r} spans lines {line_span}")
print(f"{[argument.arg for argument in function_node.args.args]=}")
print(f"{ast.dump(function_node.args.defaults[0])=}")

{'Name': 12, 'Load': 11, 'Constant': 5, 'BinOp': 4, 'Store': 3, 'Mult': 3, 'Assign': 2, 'arg': 2}
FunctionDef 'order_total' spans lines 2-6
[argument.arg for argument in function_node.args.args]=['lines', 'tax_rate']
ast.dump(function_node.args.defaults[0])='Constant(value=0.18)'


**`ast.unparse`** (3.9+) turns a tree back into source. The round trip keeps the
meaning but not the formatting: comments and redundant parentheses are gone.

In [81]:
original = "total  =  (price * quantity)   # with comment"
print(f"{original!r}")
print(f"{ast.unparse(ast.parse(original))!r}")
print(ast.unparse(pricing_tree))

'total  =  (price * quantity)   # with comment'
'total = price * quantity'
def order_total(lines, tax_rate=0.18):
    subtotal = sum((line.price * line.quantity for line in lines))
    if subtotal > 1000:
        subtotal = subtotal * 0.95
    return round(subtotal * (1 + tax_rate), 2)


### 3.2.3 Code objects

The compiler produces a **code object** for every module, class body, function,
lambda and generator expression. It is immutable and holds everything needed to
run the code, except the run-time context (globals, defaults, closure cells),
which lives on the function object (3.2.4).

| Attribute | Holds |
| --- | --- |
| `co_code` | The bytecode: 2 bytes per instruction (opcode, argument) plus inline cache slots |
| `co_consts`, `co_names`, `co_varnames` | Constants (numbers, strings, nested code objects); global and attribute names (`LOAD_GLOBAL`, `LOAD_ATTR` index into it); local variable names, arguments first |
| `co_freevars`, `co_cellvars` | Variables read from an enclosing function (closure, 1.11); this function's variables that inner functions capture |
| `co_flags` | Bit flags: optimized, generator, coroutine, has `*args`, … |
| `co_qualname` | Qualified name (*New in 3.11*) |
| `co_exceptiontable` | Where to jump when an exception occurs (*New in 3.11*, replaces setup/pop block instructions) |
| `code.replace(...)` | A copy with some fields changed |

One cell per row follows. They all use this closure-based formatter:

In [82]:
def make_price_formatter(currency: str) -> Callable[..., str]:
    """Return a formatter that remembers the currency (a closure)."""
    symbol_width = 3

    def format_price(amount: float, *, decimals: int = 2) -> str:
        """Format an amount with the currency."""
        try:
            return f"{currency:>{symbol_width}} {amount:,.{decimals}f}"
        except ValueError:
            return f"{currency} ?"

    return format_price


format_inr = make_price_formatter("INR")
price_code = format_inr.__code__
print(f"{format_inr(1234.5)=}  {type(price_code).__name__=}")

format_inr(1234.5)='INR 1,234.50'  type(price_code).__name__='code'


#### `co_code`

Raw bytes. In 3.11+ some instructions are followed by **inline cache** entries
(`CACHE` slots, 3.2.8), so the byte count is larger than two bytes per visible
instruction.

In [83]:
visible_instructions = list(dis.get_instructions(price_code))
print(f"{len(price_code.co_code)=} bytes for {len(visible_instructions)} visible instructions")
print(f"first 12 bytes: {price_code.co_code[:12].hex(' ')}")
first = visible_instructions[0]
print(f"first instruction: {first.opname} (opcode {first.opcode})  [{interpreter_label()}]")

len(price_code.co_code)=84 bytes for 36 visible instructions
first 12 bytes: 3a 02 80 00 00 00 19 00 52 02 51 01
first instruction: COPY_FREE_VARS (opcode 58)  [CPython 3.15.1, default (GIL) build, x86_64]


#### `co_consts`, `co_names`, `co_varnames`

Instructions refer to these tuples by index: `LOAD_CONST 2` loads
`co_consts[2]`; `STORE_FAST 1` stores into local slot 1, named `co_varnames[1]`.

In [84]:
print(f"{price_code.co_consts=}")
print(f"{price_code.co_names=}")
print(f"{price_code.co_varnames=}  (arguments first: amount, decimals)")
print(f"{price_code.co_argcount=}  {price_code.co_kwonlyargcount=}  {price_code.co_nlocals=}")

price_code.co_consts=('Format an amount with the currency.', '>', ' ', ',.', 'f', ' ?')
price_code.co_names=('ValueError',)
price_code.co_varnames=('amount', 'decimals')  (arguments first: amount, decimals)
price_code.co_argcount=1  price_code.co_kwonlyargcount=1  price_code.co_nlocals=2


#### `co_freevars` and `co_cellvars`

The outer function lists the captured variables in `co_cellvars`; the inner one
lists them in `co_freevars`. At run time each is a **cell** object shared by both.

In [85]:
outer_code = make_price_formatter.__code__
print(f"{outer_code.co_cellvars=}  (captured by the inner function)")
print(f"{price_code.co_freevars=}  (read from the outer function)")
print(f"{[cell.cell_contents for cell in format_inr.__closure__]=}")

outer_code.co_cellvars=('currency', 'symbol_width')  (captured by the inner function)
price_code.co_freevars=('currency', 'symbol_width')  (read from the outer function)
[cell.cell_contents for cell in format_inr.__closure__]=['INR', 3]


#### `co_flags`

A bit field. `dis.COMPILER_FLAG_NAMES` maps bits to names; `inspect` has
constants such as `inspect.CO_GENERATOR`.

In [86]:
def flag_names(code_object: types.CodeType) -> list[str]:
    """Names of the flags set in a code object."""
    return [name for bit, name in dis.COMPILER_FLAG_NAMES.items() if code_object.co_flags & bit]


def countdown(start: int):
    """A generator."""
    yield from range(start, 0, -1)


print(f"format_price:    {flag_names(price_code)}")
print(f"countdown:       {flag_names(countdown.__code__)}")
print(f"make_price_formatter: {flag_names(outer_code)}")
print(f"{bool(countdown.__code__.co_flags & inspect.CO_GENERATOR)=}")

format_price:    ['OPTIMIZED', 'NEWLOCALS', 'NESTED', 'HAS_DOCSTRING']
countdown:       ['OPTIMIZED', 'NEWLOCALS', 'GENERATOR', 'HAS_DOCSTRING']
make_price_formatter: ['OPTIMIZED', 'NEWLOCALS', 'HAS_DOCSTRING']
bool(countdown.__code__.co_flags & inspect.CO_GENERATOR)=True


#### `co_qualname`

*New in 3.11.* The qualified name, which shows nesting, like `__qualname__` on
functions.

In [87]:
print(f"{price_code.co_name=}")
print(f"{price_code.co_qualname=}")

price_code.co_name='format_price'
price_code.co_qualname='make_price_formatter.<locals>.format_price'


#### The exception table

*New in 3.11.* `try` costs nothing when no exception happens: instead of
"setup block" instructions, the code object carries a table of
`start–end → target` ranges, consulted only when an exception is raised. `dis`
prints it at the end as `ExceptionTable:`, and `dis.Bytecode(...).exception_entries`
returns it as data.

In [88]:
show_dis(format_inr)
for entry in dis.Bytecode(format_inr).exception_entries:
    print(f"offsets {entry.start}-{entry.end} -> handler at {entry.target}, depth {entry.depth}")

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  --           COPY_FREE_VARS           2

   5           RESUME                   0

   7           NOP

   8   L1:     LOAD_DEREF               2 (currency)
               LOAD_CONST               1 ('>')
               LOAD_DEREF               3 (symbol_width)
               FORMAT_SIMPLE
               BUILD_STRING             2
               FORMAT_WITH_SPEC
               LOAD_CONST               2 (' ')
               LOAD_FAST_BORROW         0 (amount)
               LOAD_CONST               3 (',.')
               LOAD_FAST_BORROW         1 (decimals)
               FORMAT_SIMPLE
               LOAD_CONST               4 ('f')
               BUILD_STRING             3
               FORMAT_WITH_SPEC
               BUILD_STRING             3
       L2:     RETURN_VALUE

  --   L3:     PUSH_EXC_INFO

   9           LOAD_GLOBAL              0 (ValueError)
               CHECK_EXC_MAT

#### `code.replace`

Code objects are immutable; `replace(**changes)` returns a modified copy. Here a
constant is swapped and the result wrapped in a new function (`types.FunctionType`,
3.2.4). Tools like coverage and debuggers use the same technique.

In [89]:
def greet(name: str) -> str:
    """Greet in Hindi."""
    return "Namaste, " + name


greet_code = greet.__code__
english_consts = tuple(
    "Hello, " if value == "Namaste, " else value for value in greet_code.co_consts
)
english_code = greet_code.replace(
    co_consts=english_consts, co_name="greet_english", co_qualname="greet_english"
)
greet_english = types.FunctionType(english_code, globals(), "greet_english")
print(f"{greet('Asha')=}  {greet_english('Asha')=}")
print(f"{greet_code is english_code=}  {greet_code.co_consts=}")

greet('Asha')='Namaste, Asha'  greet_english('Asha')='Hello, Asha'
greet_code is english_code=False  greet_code.co_consts=('Greet in Hindi.', 'Namaste, ')


### 3.2.4 Function objects

Running a `def` statement builds a **function object** around a code object
compiled earlier. The function adds the run-time context:

| Function attribute | What it supplies |
| --- | --- |
| `__code__` | The code object (shared by every function made from the same `def`) |
| `__globals__` | The module namespace used by `LOAD_GLOBAL` |
| `__defaults__`, `__kwdefaults__` | Default values, evaluated once when `def` ran |
| `__closure__` | A tuple of cells for `co_freevars` |
| `__name__`, `__qualname__`, `__module__`, `__doc__`, `__dict__` | Metadata |

`types.FunctionType(code, globals, name=None, argdefs=None, closure=None)` builds
one by hand, which proves that a function is just these pieces.

In [90]:
print(f"{format_inr.__code__ is price_code=}")
print(f"{format_inr.__globals__ is globals()=}")
print(f"{format_inr.__defaults__=}  {format_inr.__kwdefaults__=}")
print(f"{format_inr.__name__=}  {format_inr.__qualname__=}")
format_usd = make_price_formatter("USD")
print(f"{format_usd.__code__ is format_inr.__code__=}  (one code object, two functions)")
print(f"{format_usd.__closure__ is format_inr.__closure__=}  (different cells)")

format_inr.__code__ is price_code=True
format_inr.__globals__ is globals()=True
format_inr.__defaults__=None  format_inr.__kwdefaults__={'decimals': 2}
format_inr.__name__='format_price'  format_inr.__qualname__='make_price_formatter.<locals>.format_price'
format_usd.__code__ is format_inr.__code__=True  (one code object, two functions)
format_usd.__closure__ is format_inr.__closure__=False  (different cells)


**Same code, different globals.** `LOAD_GLOBAL` looks names up in the function's
`__globals__`, so the same code object gives different answers in a different
namespace.

In [91]:
GST_RATE = 0.18


def with_tax(amount: float) -> float:
    """Add GST using the module-level rate."""
    return round(amount * (1 + GST_RATE), 2)


with_low_tax = types.FunctionType(
    with_tax.__code__, {"GST_RATE": 0.05, "round": round}, "with_low_tax"
)
print(f"{with_tax(100)=}  {with_low_tax(100)=}")

with_tax(100)=118.0  with_low_tax(100)=105.0


**Defaults and closures by hand.** `argdefs` fills `__defaults__`; `closure` must
be a tuple of `types.CellType` objects in the order of `co_freevars`.

In [92]:
cells_by_name = {"currency": types.CellType("EUR"), "symbol_width": types.CellType(5)}
closure = tuple(cells_by_name[name] for name in price_code.co_freevars)
format_eur = types.FunctionType(price_code, globals(), "format_eur", None, closure)
format_eur.__kwdefaults__ = {"decimals": 1}
print(f"{format_eur(99.25)=}")
print(f"{format_inr(99.25)=}  (unchanged)")

format_eur(99.25)='  EUR 99.2'
format_inr(99.25)='INR 99.25'  (unchanged)


### 3.2.5 Reading `dis`

**Columns.** Each line of `dis.dis` output is one instruction:

| Column | Example | Meaning |
| --- | --- | --- |
| Source line | `3` | Line number, printed when it changes |
| Label | `L1:` | Jump target (*Changed in 3.13*: labels replace offsets by default) |
| Offset | `8` | Byte offset in `co_code` (shown with `show_offsets=True`, 3.13+) |
| Opname | `BINARY_OP` | The instruction |
| Oparg | `5` | Its integer argument |
| Argrepr | `(*)` | What the argument means: a name, a constant, an operator, a jump target |

In [93]:
def discounted_price(price: float, percent: float) -> float:
    """Price after a percentage discount."""
    return price - price * percent / 100


show_dis(discounted_price)
if sys.version_info >= (3, 13):
    print()
    show_dis(discounted_price, show_offsets=True)

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  1           RESUME                   0

  3           LOAD_FAST_BORROW_LOAD_FAST_BORROW 0 (price, price)
              LOAD_FAST_BORROW         1 (percent)
              BINARY_OP                5 (*)
              LOAD_SMALL_INT         100
              BINARY_OP               11 (/)
              BINARY_OP               10 (-)
              RETURN_VALUE

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  1          0       RESUME                   0

  3          4       LOAD_FAST_BORROW_LOAD_FAST_BORROW 0 (price, price)
             6       LOAD_FAST_BORROW         1 (percent)
             8       BINARY_OP                5 (*)
            20       LOAD_SMALL_INT         100
            22       BINARY_OP               11 (/)
            34       BINARY_OP               10 (-)
            46       RETURN_VALUE


**A stack machine.** CPython's evaluation loop keeps a **value stack** per frame.
Loads push, operators pop their operands and push the result, `RETURN_VALUE` pops
the result and hands it to the caller. The walkthrough for `discounted_price` (on
3.15; `LOAD_FAST_BORROW_LOAD_FAST_BORROW` is a *superinstruction* that loads two
locals at once):

| Instruction | Pops | Pushes | Stack after (bottom → top) |
| --- | --- | --- | --- |
| `LOAD_FAST_BORROW_LOAD_FAST_BORROW (price, price)` | — | 2 | `price`, `price` |
| `LOAD_FAST_BORROW (percent)` | — | 1 | `price`, `price`, `percent` |
| `BINARY_OP (*)` | 2 | 1 | `price`, `price*percent` |
| `LOAD_SMALL_INT 100` | — | 1 | `price`, `price*percent`, `100` |
| `BINARY_OP (/)` | 2 | 1 | `price`, `price*percent/100` |
| `BINARY_OP (-)` | 2 | 1 | `price - price*percent/100` |
| `RETURN_VALUE` | 1 | — | *(empty: the value goes to the caller)* |

`dis.stack_effect(opcode, oparg)` gives the net change for each instruction; the
cell replays it to confirm the depth column. The compiler stores the maximum in
`co_stacksize`.

In [94]:
depth = 0
for instruction in dis.get_instructions(discounted_price):
    if instruction.opname == "RETURN_VALUE":
        depth -= 1  # pops the return value
    elif instruction.arg is None:
        depth += dis.stack_effect(instruction.opcode)
    else:
        depth += dis.stack_effect(instruction.opcode, instruction.arg)
    print(f"{instruction.opname:<34} {instruction.argrepr:<16} depth after: {depth}")
print(f"{discounted_price.__code__.co_stacksize=}  [{interpreter_label()}]")

RESUME                                              depth after: 0
LOAD_FAST_BORROW_LOAD_FAST_BORROW  price, price     depth after: 2
LOAD_FAST_BORROW                   percent          depth after: 3
BINARY_OP                          *                depth after: 2
LOAD_SMALL_INT                                      depth after: 3
BINARY_OP                          /                depth after: 2
BINARY_OP                          -                depth after: 1
RETURN_VALUE                                        depth after: 0
discounted_price.__code__.co_stacksize=3  [CPython 3.15.1, default (GIL) build, x86_64]


**Load and store families.** The opname tells you **where** a name lives:

| Family | Load / store | Used for |
| --- | --- | --- |
| `*_FAST` | `LOAD_FAST`, `LOAD_FAST_BORROW` (3.14+), `STORE_FAST` | Function locals: an array slot, the fastest |
| `*_DEREF` | `LOAD_DEREF`, `STORE_DEREF` | Closure cells (`nonlocal`, captured variables) |
| `*_GLOBAL` | `LOAD_GLOBAL`, `STORE_GLOBAL` | Module globals, then builtins |
| `*_NAME` | `LOAD_NAME`, `STORE_NAME` | Module and class bodies (dict lookups) |
| `*_ATTR` | `LOAD_ATTR`, `STORE_ATTR` | `obj.attr` |
| Subscript | `BINARY_OP ([])` (3.14+), `STORE_SUBSCR` | `obj[key]` |
| Constants | `LOAD_CONST`, `LOAD_SMALL_INT` (3.14+) | Literals |

One function that touches every family, then its instructions grouped:

In [95]:
RESTOCK_LEVEL = 5


def make_restocker(warehouse: dict[str, int]) -> Callable[[object], int]:
    """Inner function uses locals, a closure cell, a global, attributes and subscripts."""
    restocked = 0

    def restock(item: object) -> int:
        nonlocal restocked  # *_DEREF
        global RESTOCK_LEVEL  # *_GLOBAL
        current = warehouse[item.sku]  # subscript + attribute
        if current < RESTOCK_LEVEL:
            warehouse[item.sku] = RESTOCK_LEVEL  # STORE_SUBSCR
            item.restocked = True  # STORE_ATTR
            restocked += 1
        RESTOCK_LEVEL = 5
        return restocked

    return restock


restock_names = opnames(make_restocker({}))
for family in ("FAST", "DEREF", "GLOBAL", "ATTR", "SUBSCR", "BINARY_OP", "SMALL_INT"):
    print(f"{family:<10} {sorted({name for name in restock_names if family in name})}")

FAST       ['LOAD_FAST_BORROW', 'STORE_FAST']
DEREF      ['LOAD_DEREF', 'STORE_DEREF']
GLOBAL     ['LOAD_GLOBAL', 'STORE_GLOBAL']
ATTR       ['LOAD_ATTR', 'STORE_ATTR']
SUBSCR     ['STORE_SUBSCR']
BINARY_OP  ['BINARY_OP']
SMALL_INT  ['LOAD_SMALL_INT']


**Calls.** A call pushes the callable (and a `NULL` or `self` slot used for method
calls), then the arguments, then runs `CALL n`. Keyword arguments use `CALL_KW`
with a tuple of names; `f(*args)` uses `CALL_FUNCTION_EX`. In the listing,
`LOAD_ATTR (total + NULL|self)` loads a method in a way that avoids creating a
bound-method object.

In [96]:
def checkout(cart: object, *extra: object) -> str:
    """A method call, a keyword call and a star call."""
    total = cart.total()
    label = format(total, ",.2f")
    print(label, end="\n")
    print(*extra)
    return label


show_dis(checkout)

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  1           RESUME                   0

  3           LOAD_FAST_BORROW         0 (cart)
              LOAD_ATTR                1 (total + NULL|self)
              CALL                     0
              STORE_FAST               2 (total)

  4           LOAD_GLOBAL              3 (format + NULL)
              LOAD_FAST_BORROW         2 (total)
              LOAD_CONST               1 (',.2f')
              CALL                     2
              STORE_FAST               3 (label)

  5           LOAD_GLOBAL              5 (print + NULL)
              LOAD_FAST_BORROW         3 (label)
              LOAD_CONST               2 ('\n')
              LOAD_CONST               3 (('end',))
              CALL_KW                  2
              POP_TOP

  6           LOAD_GLOBAL              5 (print + NULL)
              LOAD_FAST_BORROW         1 (extra)
              PUSH_NULL
              CA

**Jumps.** `if` compiles to a comparison and a conditional jump. 3.13+ inserts
`TO_BOOL` (or a comparison that already yields a bool), and 3.14 adds
`NOT_TAKEN` markers used to record branch history.

In [97]:
def shipping_band(weight_kg: float) -> str:
    """One if/else."""
    if weight_kg > 20:
        return "freight"
    return "parcel"


show_dis(shipping_band)

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  1           RESUME                   0

  3           LOAD_FAST_BORROW         0 (weight_kg)
              LOAD_SMALL_INT          20
              COMPARE_OP             148 (bool(>))
              POP_JUMP_IF_FALSE        3 (to L1)
              NOT_TAKEN

  4           LOAD_CONST               1 ('freight')
              RETURN_VALUE

  5   L1:     LOAD_CONST               2 ('parcel')
              RETURN_VALUE


**Loops.** A `for` loop is `GET_ITER`, then `FOR_ITER` (which jumps out when the
iterator is exhausted), the body, and `JUMP_BACKWARD` to the top. A `while` loop
is a test, a conditional jump out, the body and `JUMP_BACKWARD`. Backward jumps
are where the interpreter checks the eval breaker (3.2.10) and where hot loops are
detected for the JIT (3.2.9).

In [98]:
def drain_stock(quantities: list[int]) -> int:
    """A for loop and a while loop."""
    total = 0
    for quantity in quantities:
        total += quantity
    while total > 100:
        total -= 40
    return total


show_dis(drain_stock)

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  1           RESUME                   0

  3           LOAD_SMALL_INT           0
              STORE_FAST               1 (total)

  4           LOAD_FAST                0 (quantities)
              GET_ITER                 0
      L1:     FOR_ITER                11 (to L2)
              STORE_FAST               2 (quantity)

  5           LOAD_FAST_BORROW_LOAD_FAST_BORROW 18 (total, quantity)
              BINARY_OP               13 (+=)
              STORE_FAST               1 (total)
              JUMP_BACKWARD           13 (to L1)

  4   L2:     END_FOR
              POP_ITER

  6   L3:     LOAD_FAST_BORROW         1 (total)
              LOAD_SMALL_INT         100
              COMPARE_OP             148 (bool(>))
              POP_JUMP_IF_FALSE       12 (to L4)
              NOT_TAKEN

  7           LOAD_FAST_BORROW         1 (total)
              LOAD_SMALL_INT          40
        

**`dis.get_instructions`** returns `Instruction` named tuples instead of text, so
tools can analyse bytecode. Useful fields: `opname`, `opcode`, `arg`, `argval`,
`argrepr`, `offset`, `line_number` (3.13+), `label`, `positions` (exact source
columns, 3.11+). The cell lists the jumps in `drain_stock` and where they go.

In [99]:
drain_instructions = list(dis.get_instructions(drain_stock))
labels_by_offset = {ins.offset: ins.label for ins in drain_instructions if ins.label}
for instruction in drain_instructions:
    if "JUMP" in instruction.opname or instruction.opname == "FOR_ITER":
        target = labels_by_offset.get(instruction.argval, instruction.argval)
        print(f"line {instruction.line_number}: {instruction.opname:<18} -> L{target}")
print(f"{drain_instructions[3].positions=}")

line 4: FOR_ITER           -> L2
line 5: JUMP_BACKWARD      -> L1
line 6: POP_JUMP_IF_FALSE  -> L4
line 7: JUMP_BACKWARD      -> L3
drain_instructions[3].positions=Positions(lineno=4, end_lineno=4, col_offset=20, end_col_offset=30)


### 3.2.6 Compiler optimizations

CPython's compiler is deliberately simple, but it does a few things before your
code runs:

| Optimization | Example | Evidence |
| --- | --- | --- |
| Constant folding | `24 * 60 * 60` → `86400` | the folded value in `co_consts` |
| Dead-code removal | code after `return`, `if False:` blocks | no instructions for them |
| Constant containers for `in` | `x in [1, 2, 3]` → tuple; `x in {1, 2, 3}` → `frozenset` | a `LOAD_CONST` of the container |
| Comprehension inlining (PEP 709, 3.12) | `[x * 2 for x in xs]` runs in the enclosing frame | no nested `<listcomp>` code object |

**Constant folding.**

In [100]:
def seconds_per_day() -> tuple[int, str]:
    """Arithmetic on literals is folded at compile time."""
    return 24 * 60 * 60, "ab" * 3


print(f"{seconds_per_day.__code__.co_consts=}")
print(f"{opnames(seconds_per_day)=}")

seconds_per_day.__code__.co_consts=('Arithmetic on literals is folded at compile time.', (86400, 'ababab'))
opnames(seconds_per_day)=['RESUME', 'LOAD_CONST', 'RETURN_VALUE']


**Dead-code removal.** No instructions are emitted for the `if False:` block or the
line after `return`. The name `print` may still appear in `co_names` (names are
collected before the dead code is dropped), but no instruction uses it.

In [101]:
def dead_code_example() -> int:
    """Contains unreachable code."""
    if False:
        print("never compiled")
    return 1
    print("unreachable")


print(f"{opnames(dead_code_example)=}")
used_names = {ins.argval for ins in dis.get_instructions(dead_code_example)}
print(f"{'print' in dead_code_example.__code__.co_names=}  {'print' in used_names=}")

opnames(dead_code_example)=['RESUME', 'NOP', 'LOAD_SMALL_INT', 'RETURN_VALUE']
'print' in dead_code_example.__code__.co_names=True  'print' in used_names=False


**Constant containers for membership tests.** A list or set literal used only for
`in` becomes an immutable constant built once at compile time, instead of a new
list or set on every call.

In [102]:
def is_metro(city: str) -> tuple[bool, bool]:
    """Membership tests against literal containers."""
    return city in ["Delhi", "Mumbai", "Chennai"], city in {"Kolkata", "Bengaluru"}


print(f"{is_metro.__code__.co_consts=}")

is_metro.__code__.co_consts=('Membership tests against literal containers.', ('Delhi', 'Mumbai', 'Chennai'), frozenset({'Kolkata', 'Bengaluru'}))


**Comprehension inlining (PEP 709, *New in 3.12*).** List, dict and set
comprehensions used to compile into a hidden nested function (`<listcomp>`) that
was called every time. Since 3.12 they are inlined into the enclosing code object,
which saves a function call and a frame. Generator expressions still get their
own code object, because they must be able to pause.

In [103]:
def comprehension_kinds(prices: list[float]) -> tuple[object, ...]:
    """A list comprehension, a dict comprehension and a generator expression."""
    doubled = [price * 2 for price in prices]
    labels = {price: f"{price:.2f}" for price in prices}
    lazy_total = sum(price for price in prices)
    return doubled, labels, lazy_total


nested_code_objects = [
    constant.co_name
    for constant in comprehension_kinds.__code__.co_consts
    if isinstance(constant, types.CodeType)
]
print(f"{nested_code_objects=}  (only the generator expression is separate)")
saves_loop_variable = "LOAD_FAST_AND_CLEAR" in opnames(comprehension_kinds)
print(f"{saves_loop_variable=}  (an inlined comprehension saves and restores its variable)")

nested_code_objects=['<genexpr>']  (only the generator expression is separate)
saves_loop_variable=True  (an inlined comprehension saves and restores its variable)


### 3.2.7 The evaluation loop

`_PyEval_EvalFrameDefault` in `Python/ceval.c` runs a frame:

```text
loop:
    instruction = code[instruction_pointer]      # fetch
    instruction_pointer += 1 + number of cache entries
    dispatch on instruction.opcode               # switch / computed goto / tail call
        execute the body from bytecodes.c        # push, pop, call into Objects/*.c
    if the eval breaker is set: handle it        # at calls and backward jumps (3.2.10)
```

- **Frames.** Each running call has an interpreter frame: the code object,
  globals, builtins, the local variable array, the value stack and the
  **instruction pointer**. Since 3.11 these frames are plain C structs pushed on a
  per-thread "data stack"; a Python `frame` object is created only when you ask
  for one.
- **Inlined Python-to-Python calls (3.11).** When Python code calls a Python
  function, the loop pushes a new interpreter frame and **keeps going in the same
  C function**, instead of recursing into a new C call. Deep Python recursion no
  longer eats the C stack, and calls got much cheaper.
- **Dispatch.** Builds with clang 19+ can use a *tail-calling* interpreter
  (*New in 3.14*): each instruction is a small C function that tail-calls the next.

The frame object exposes the instruction pointer as `f_lasti` (byte offset of
the current instruction) and the line as `f_lineno`.

In [104]:
def where_am_i() -> list[str]:
    """Describe the current frame and its callers."""
    described = []
    frame = sys._getframe()
    while frame is not None and len(described) < 2:
        described.append(
            f"{frame.f_code.co_qualname}: f_lasti={frame.f_lasti} f_lineno={frame.f_lineno}"
        )
        frame = frame.f_back
    return described


def caller_of_where() -> list[str]:
    return where_am_i()


print("\n".join(caller_of_where()))
tail_calls = bool(sysconfig.get_config_var("_Py_TAIL_CALL_INTERP"))
print(f"tail-calling interpreter in this build: {tail_calls}")

where_am_i: f_lasti=158 f_lineno=7
caller_of_where: f_lasti=14 f_lineno=14
tail-calling interpreter in this build: True


**The instruction pointer survives a pause.** A generator keeps its frame
between `next()` calls; `gi_frame.f_lasti` shows the loop resuming exactly where
it stopped.

In [105]:
def ticket_numbers():
    """Yield three ticket numbers."""
    yield 101
    yield 102
    yield 103


ticket_generator = ticket_numbers()
for _ in range(3):
    ticket = next(ticket_generator)
    print(f"got {ticket}, paused at f_lasti={ticket_generator.gi_frame.f_lasti}")

got 101, paused at f_lasti=12
got 102, paused at f_lasti=22
got 103, paused at f_lasti=32


**Inlined calls in action.** After warming up, the `CALL` in the caller becomes
`CALL_PY_EXACT_ARGS`: the specialized form that sets up the callee's frame
directly in the eval loop. And because Python-to-Python calls do not recurse in C,
a 20 000-deep Python recursion works once the recursion limit allows it (the cell
raises the limit temporarily and restores it).

In [106]:
def add_one(value: int) -> int:
    return value + 1


def call_add_one(value: int) -> int:
    return add_one(value)


for _ in range(1_000):
    call_add_one(1)
print(f"{[name for name in opnames(call_add_one, adaptive=True) if name.startswith('CALL')]=}")


def depth_of(remaining: int) -> int:
    """Recurse remaining levels deep."""
    return 0 if remaining == 0 else 1 + depth_of(remaining - 1)


original_limit = sys.getrecursionlimit()
sys.setrecursionlimit(25_000)
try:
    print(f"{depth_of(20_000)=}  (default limit is {original_limit})")
finally:
    sys.setrecursionlimit(original_limit)

[name for name in opnames(call_add_one, adaptive=True) if name.startswith('CALL')]=['CALL_PY_EXACT_ARGS']
depth_of(20_000)=20000  (default limit is 1000)


### 3.2.8 Specializing adaptive interpreter (PEP 659)

*New in 3.11.* CPython watches the types each instruction actually sees and
rewrites the instruction **in place** with a specialized version:

| Term | Meaning |
| --- | --- |
| **Quickening** | After a function warms up, its generic instructions become *adaptive* ones with counters |
| **Inline cache** | `CACHE` slots after an instruction in `co_code` store what was learned: a type version, a dict key index, a function version |
| **Specialized instruction** | e.g. `LOAD_ATTR_INSTANCE_VALUE`, `BINARY_OP_ADD_INT`, `CALL_PY_EXACT_ARGS`, `LOAD_GLOBAL_MODULE` |
| **Guard** | Each specialized instruction first checks its assumption (same type version, same dict keys) |
| **Deoptimization** | A failed guard runs the generic path; after repeated misses the instruction re-specializes for the new types (with back-off) |

`dis.dis(func, adaptive=True)` shows the current, specialized instructions.
**Before** warming:

In [107]:
class CartLine:
    """A cart line with plain instance attributes."""

    def __init__(self, price: int, quantity: int) -> None:
        self.price = price
        self.quantity = quantity


def line_total(line: object) -> object:
    """price * quantity of any object with those attributes."""
    return line.price * line.quantity


show_dis(line_total, adaptive=True)

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  9           RESUME                   0

 11           LOAD_FAST_BORROW         0 (line)
              LOAD_ATTR                0 (price)
              LOAD_FAST_BORROW         0 (line)
              LOAD_ATTR                2 (quantity)
              BINARY_OP                5 (*)
              RETURN_VALUE


**After** 1 000 calls with `CartLine` objects holding ints. `RESUME` becomes
`RESUME_CHECK`, both attribute loads become `LOAD_ATTR_INSTANCE_VALUE` (read the
value straight from the object's inline values after checking the type version)
and the multiplication becomes `BINARY_OP_MULTIPLY_INT`.

In [108]:
for _ in range(1_000):
    line_total(CartLine(25, 4))
show_dis(line_total, adaptive=True)

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  9           RESUME_CHECK             0

 11           LOAD_FAST_BORROW         0 (line)
              LOAD_ATTR_INSTANCE_VALUE 0 (price)
              LOAD_FAST_BORROW         0 (line)
              LOAD_ATTR_INSTANCE_VALUE 2 (quantity)
              BINARY_OP_MULTIPLY_INT   5 (*)
              RETURN_VALUE


**Inline caches.** `show_caches=True` reveals the `CACHE` slots that hold the
learned facts (type version, index of the value). They are why `co_code` is longer
than the visible instructions (3.2.3).

In [109]:
cache_buffer = io.StringIO()
dis.dis(line_total, adaptive=True, show_caches=True, file=cache_buffer)
cache_slot_count = cache_buffer.getvalue().count("CACHE")
print(f"visible instructions: {len(opnames(line_total))}, CACHE slots: {cache_slot_count}")
print(f"{len(line_total.__code__.co_code)=} bytes  [{interpreter_label()}]")

visible instructions: 7, CACHE slots: 24
len(line_total.__code__.co_code)=62 bytes  [CPython 3.15.1, default (GIL) build, x86_64]


**Deoptimization.** Now feed it an object whose `price` is a **property** and whose
values are floats. The guards fail, the instructions fall back, and after enough
misses they re-specialize: `LOAD_ATTR_PROPERTY` for the first load, and a different
`BINARY_OP` form for float × int.

In [110]:
class PromoLine:
    """price is a property, and a float."""

    def __init__(self, base_price: float, quantity: int) -> None:
        self._base_price = base_price
        self.quantity = quantity

    @property
    def price(self) -> float:
        return self._base_price * 0.9


warm_forms = opnames(line_total, adaptive=True)
for _ in range(1_000):
    line_total(PromoLine(25.0, 4))
for before, after in zip(warm_forms, opnames(line_total, adaptive=True), strict=True):
    marker = "   <- re-specialized" if before != after else ""
    print(f"{before:<26} -> {after:<26}{marker}")

RESUME_CHECK               -> RESUME_CHECK              
LOAD_FAST_BORROW           -> LOAD_FAST_BORROW          
LOAD_ATTR_INSTANCE_VALUE   -> LOAD_ATTR_PROPERTY           <- re-specialized
LOAD_FAST_BORROW           -> LOAD_FAST_BORROW          
LOAD_ATTR_INSTANCE_VALUE   -> LOAD_ATTR_INSTANCE_VALUE  
BINARY_OP_MULTIPLY_INT     -> BINARY_OP_EXTEND             <- re-specialized
RETURN_VALUE               -> RETURN_VALUE              


#### JS bridge: hidden classes vs type versions

| V8 | CPython |
| --- | --- |
| Every object has a hidden class (map); adding a property moves it to a new map | Every type has a **version tag**; changing the class (adding a method, assigning a class attribute) gives it a new version |
| Inline cache at a property access remembers *map → offset* | `LOAD_ATTR_INSTANCE_VALUE` remembers *type version → slot index* |
| Monomorphic / polymorphic / megamorphic sites | One specialization per instruction; on repeated misses it re-specializes or stays generic |
| Deopt bails out of optimized machine code | A failed guard runs the generic instruction body (no machine code involved, unless the JIT is on) |

The practical lesson is the same in both: code that sees **consistent types** at a
given line runs faster.

### 3.2.9 The JIT

*New in 3.13, experimental, off by default.* PEP 744 adds a second tier:

1. **Hot code detection.** Backward jumps count iterations; a loop that runs often
   enough is handed to the tier-2 optimizer.
2. **Micro-op traces.** The optimizer records a *trace* along the path the loop
   actually takes and translates each specialized instruction into smaller
   **micro-ops** (uops) such as `_GUARD_TYPE_VERSION` or `_BINARY_OP_ADD_INT`, then
   removes redundant guards.
3. **Copy-and-patch code generation.** At **build** time, LLVM compiles every uop
   into a machine-code template ("stencil") with holes. At **run** time the JIT
   copies the templates for a trace into executable memory and patches the holes
   (constants, jump targets). No compiler runs inside your process.

| Check | How |
| --- | --- |
| Was this interpreter built with the JIT? | `sys._jit.is_available()` (*New in 3.14*) |
| Is it switched on? | `sys._jit.is_enabled()`; start Python with `PYTHON_JIT=1` |
| Is the current frame running JIT code? | `sys._jit.is_active()` |
| Build-time option | `./configure --enable-experimental-jit` (`=yes-off` builds it but leaves it off) |

In [111]:
if hasattr(sys, "_jit"):
    print(f"{sys._jit.is_available()=}  (built with the JIT)")
    print(f"{sys._jit.is_enabled()=}  (switched on in this kernel)")
    print(f"{sys._jit.is_active()=}  (this cell is not running JIT code)")
    print(f"{os.environ.get('PYTHON_JIT')=}")
else:
    print("Skipped: sys._jit needs Python 3.14+ (3.13: look for --enable-experimental-jit)")

sys._jit.is_available()=True  (built with the JIT)
sys._jit.is_enabled()=False  (switched on in this kernel)
sys._jit.is_active()=False  (this cell is not running JIT code)
os.environ.get('PYTHON_JIT')=None


**Watching it work.** The kernel runs without the JIT, so the cell starts two child
interpreters, with `PYTHON_JIT=0` and `PYTHON_JIT=1`, warms a hot loop in each and
looks for **executors** (compiled traces) attached to its code object with
`_opcode.get_executor`, a private testing hook. For the first executor it prints the
number of micro-ops, the first few uop names and the size of the machine code. Only
names are printed; uop operands contain addresses.

In [112]:
JIT_PROBE = textwrap.dedent(
    """
    import sys

    import _opcode

    def hot_loop(count):
        total = 0
        for number in range(count):
            total += number * 2
        return total

    for _ in range(50):
        hot_loop(100_000)
    loop_code = hot_loop.__code__
    executors = []
    for offset in range(0, len(loop_code.co_code), 2):
        try:
            executor = _opcode.get_executor(loop_code, offset)
        except Exception:  # no executor at this offset
            continue
        if executor is not None:
            executors.append(executor)
    print("jit_enabled", sys._jit.is_enabled())
    print("executors", len(executors))
    if executors:
        uop_names = [uop[0] for uop in executors[0]]
        print("uop_count", len(uop_names))
        print("first_uops", " ".join(uop_names[3:9]))
        machine_code = executors[0].get_jit_code()
        print("machine_code_bytes", len(machine_code) if machine_code else 0)
    """
)

if hasattr(sys, "_jit") and sys._jit.is_available():
    for jit_setting in ("0", "1"):
        child_env = {**CHILD_ENV, "PYTHON_JIT": jit_setting}
        completed = subprocess.run(
            [sys.executable, "-c", JIT_PROBE],
            capture_output=True, text=True, timeout=120, env=child_env,
        )  # fmt: skip
        print(f"--- PYTHON_JIT={jit_setting} ---")
        print(completed.stdout.strip() or completed.stderr.strip()[-300:] or "(no output)")
else:
    print("Skipped: this interpreter was built without the JIT")

--- PYTHON_JIT=0 ---
jit_enabled False
executors 0


--- PYTHON_JIT=1 ---
jit_enabled True
executors 1
uop_count 39
first_uops _CHECK_PERIODIC _CHECK_VALIDITY _ITER_CHECK_RANGE _GUARD_NOT_EXHAUSTED_RANGE _ITER_NEXT_RANGE _SET_IP
machine_code_bytes 4096


> **Note:** on 3.15 the JIT gives modest, workload-dependent speedups (and can be
> slower); it is a foundation for future releases. Measure before enabling it.
> `_opcode.get_executor` and the uop names are internal and change between
> releases.

#### JS bridge: Ignition and TurboFan vs bytecode and the JIT

| Stage | V8 | CPython 3.15 |
| --- | --- | --- |
| Bytecode | Ignition: register-based bytecode interpreter | Tier 1: stack-based bytecode interpreter |
| Type feedback | Feedback vectors fed by inline caches | Inline caches in `co_code` (3.2.8) |
| Cheap speed-up | Sparkplug: baseline compiler, bytecode → machine code without optimization | Specialized instructions (still interpreted) |
| Optimizing tier | Maglev, then TurboFan (sea-of-nodes, speculative inlining) | Tier 2 uop traces + copy-and-patch JIT |
| Deoptimization | Back to Ignition frames | Exit the trace back to tier 1 |
| Default | Always on | JIT off by default; specialization always on |

### 3.2.10 The GIL and the eval breaker

In the default build, a thread must hold the **global interpreter lock** to run
bytecode, so only one thread executes Python code at a time. The eval loop gives
other threads a turn through the **eval breaker**: a word of flags that the loop
checks cheaply at specific instructions (function entry `RESUME`, backward jumps,
some calls). When a flag is set, the loop stops to handle it:

| Flag | Who sets it | What the loop does |
| --- | --- | --- |
| GIL drop request | another thread waiting longer than `sys.getswitchinterval()` (5 ms) | releases the GIL, lets the other thread run, waits to re-acquire |
| Signal pending | the C signal handler | runs the Python-level signal handler (`signal.signal`) |
| Pending calls | C code (`Py_AddPendingCall`) | runs those callbacks |
| Async exception | `PyThreadState_SetAsyncExc` | raises it in this thread |
| GC scheduled | allocation counters (3.12+) | runs a cyclic GC collection |

So **signal handlers and thread switches happen only between bytecode
instructions**, never in the middle of one, and a long-running C call (a huge
`sorted`, a regex) delays them.

In [113]:
def count_thread_switches(switch_interval: float, duration: float = 0.2) -> int:
    """Two CPU-bound threads; count how often the running thread changes."""
    state = {"owner": "", "switches": 0}
    deadline = time.perf_counter() + duration

    def spin(name: str) -> None:
        while time.perf_counter() < deadline:
            if state["owner"] != name:
                state["owner"] = name
                state["switches"] += 1

    original_interval = sys.getswitchinterval()
    sys.setswitchinterval(switch_interval)
    try:
        threads = [threading.Thread(target=spin, args=(name,)) for name in ("A", "B")]
        for thread in threads:
            thread.start()
        for thread in threads:
            thread.join()
    finally:
        sys.setswitchinterval(original_interval)
    return state["switches"]


print(f"{sys.getswitchinterval()=}  [{interpreter_label()}]")
for interval in (0.005, 0.0005):
    switches = count_thread_switches(interval)
    print(f"switch interval {interval * 1000:>4} ms -> {switches:>5} switches in 0.2 s")

sys.getswitchinterval()=0.005  [CPython 3.15.1, default (GIL) build, x86_64]


switch interval  5.0 ms ->    37 switches in 0.2 s


switch interval  0.5 ms ->   346 switches in 0.2 s


Ten times shorter interval, roughly ten times more switches: the GIL handoffs
are driven by the eval breaker. (On a free-threaded build both threads run at
once, so the count is noisy and much higher.)

**Signals arrive between instructions.** A timer thread sends `SIGUSR1` to this
process while the main thread spins in a pure-Python loop. The C-level handler
only sets a flag; the Python handler runs at the next eval-breaker check, in the
main thread. (Unix only; the old handler is restored.)

In [114]:
if hasattr(signal, "SIGUSR1") and threading.current_thread() is threading.main_thread():
    progress = {"iterations": 0, "handled_at": None}

    def on_user_signal(signal_number: int, frame: types.FrameType | None) -> None:
        """Python-level handler: runs between two bytecode instructions."""
        progress["handled_at"] = progress["iterations"]

    previous_handler = signal.signal(signal.SIGUSR1, on_user_signal)
    try:
        threading.Timer(0.05, os.kill, (os.getpid(), signal.SIGUSR1)).start()
        give_up_at = time.perf_counter() + 2
        while progress["handled_at"] is None and time.perf_counter() < give_up_at:
            progress["iterations"] += 1
    finally:
        signal.signal(signal.SIGUSR1, previous_handler)
    handled = progress["handled_at"] is not None
    iterations = progress["iterations"]
    print(f"handler ran in the main thread: {handled}, after {iterations:,} loop iterations")
else:
    print("Skipped: needs SIGUSR1 (Unix) and the main thread")

handler ran in the main thread: True, after 298,384 loop iterations


**The free-threaded design (overview).** Without a GIL (3.1.5) the eval breaker
still exists, but for different jobs: signals, pending calls and **stop-the-world**
pauses, where all threads park at an eval-breaker check so the GC (or a few other
operations) can run safely. Thread safety moves into the objects themselves
(per-object locks, biased reference counting, mimalloc), and the specializing
interpreter uses thread-safe variants. Phase 5 measures what this means for
threads, processes and subinterpreters.

### 3.2.11 `compile`, `exec`, `eval`

| Function | Input | Returns |
| --- | --- | --- |
| `compile(source, filename, mode)` | text or an AST | a code object |
| `eval(source_or_code, globals=None, locals=None)` | **one expression** | its value |
| `exec(source_or_code, globals=None, locals=None)` | any statements | `None` |

| `compile` mode | Accepts | Like |
| --- | --- | --- |
| `"exec"` | a module: any statements | running a file |
| `"eval"` | a single expression | `eval` |
| `"single"` | one interactive statement; expression values are **printed** | the REPL |

In [115]:
expression_code = compile("price * quantity", "<pricing>", "eval")
module_code = compile(
    "subtotal = price * quantity\ntotal = round(subtotal * 1.18, 2)", "<pricing>", "exec"
)
single_code = compile("price * quantity", "<repl>", "single")

namespace = {"price": 250.0, "quantity": 3}
print(f"{eval(expression_code, namespace)=}")
print(f"{exec(module_code, namespace)=}  -> namespace['total']={namespace['total']}")
repl_output = io.StringIO()
jupyter_displayhook = sys.displayhook
sys.displayhook = sys.__displayhook__  # the plain REPL hook (Jupyter installs its own)
try:
    with contextlib.redirect_stdout(repl_output):
        exec(single_code, namespace)
finally:
    sys.displayhook = jupyter_displayhook
print(f"'single' mode printed: {repl_output.getvalue().strip()!r}")
print(f"{raised_message(lambda: compile('x = 1', '<e>', 'eval'))=}")

eval(expression_code, namespace)=750.0
exec(module_code, namespace)=None  -> namespace['total']=885.0
'single' mode printed: '750.0'
raised_message(lambda: compile('x = 1', '<e>', 'eval'))='SyntaxError: invalid syntax (<e>, line 1)'


**Namespaces.** `globals` is the dict used for global lookups and stores; if it has
no `__builtins__` key, Python inserts one. With a separate `locals` mapping,
top-level assignments go to `locals`, as in a class body.

In [116]:
sandbox_globals: dict[str, object] = {"rate": 0.18}
sandbox_locals: dict[str, object] = {}
exec("tax = 100 * rate\nlabel = f'{tax:.2f}'", sandbox_globals, sandbox_locals)
print(f"{sorted(sandbox_locals)=}")
print(f"{sorted(sandbox_globals)=}  (__builtins__ was added)")
print(f"{type(sandbox_globals['__builtins__']).__name__=}")

sorted(sandbox_locals)=['label', 'tax']
sorted(sandbox_globals)=['__builtins__', 'rate']  (__builtins__ was added)
type(sandbox_globals['__builtins__']).__name__='dict'


> **Warning:** `eval` or `exec` on user input is **remote code execution**. Any
> expression can import modules and call them. Removing `__builtins__` does not help:
> every object leads back to `object.__subclasses__()`, which reaches dangerous
> classes. The cell runs two "inputs" that a calculator endpoint might receive; the
> escape list is only measured, never used.

In [117]:
user_input = "__import__('platform').system()"
print(f"eval of user input ran an import: {eval(user_input)!r}")  # deliberate: never do this

escape_input = "().__class__.__base__.__subclasses__()"
reachable_classes = eval(escape_input, {"__builtins__": {}})  # deliberate: never do this
print(f"with __builtins__ removed, still reachable classes: {len(reachable_classes) > 100}")

eval of user input ran an import: 'Linux'
with __builtins__ removed, still reachable classes: True


**`ast.literal_eval`** accepts only literals (numbers, strings, bytes, tuples,
lists, dicts, sets, `True`/`False`/`None`, and `+`/`-` on numbers). Use it to read
Python-literal data; anything else raises `ValueError`. For arithmetic, write an
evaluator that walks the AST (Lab 3.2, exercise 5).

In [118]:
catalog_text = '{"sku": "TEA-01", "sizes": [250, 500], "active": True}'
print(f"{ast.literal_eval(catalog_text)=}")
for unsafe_text in ("__import__('os')", "2 ** 1000", "price * 2"):
    outcome = raised_message(functools.partial(ast.literal_eval, unsafe_text))
    print(f"{unsafe_text!r:<20} {outcome[:48]}...")

ast.literal_eval(catalog_text)={'sku': 'TEA-01', 'sizes': [250, 500], 'active': True}
"__import__('os')"   ValueError: malformed node or string on line 1: ...
'2 ** 1000'          ValueError: malformed node or string on line 1: ...
'price * 2'          ValueError: malformed node or string on line 1: ...


### 3.2.12 AST transformation

| Class / function | Use |
| --- | --- |
| `ast.NodeVisitor` | Walk a tree; define `visit_<NodeType>` methods; call `self.generic_visit(node)` to continue into children |
| `ast.NodeTransformer` | Like a visitor, but the return value **replaces** the node (`None` deletes it) |
| `ast.fix_missing_locations(tree)` | Fill in `lineno` / `col_offset` for nodes you created, so `compile` accepts the tree |
| `ast.copy_location(new, old)` | Give a new node the old node's position |

**`NodeVisitor`:** collect which functions a piece of code calls.

In [119]:
class CallCollector(ast.NodeVisitor):
    """Count calls by the called name."""

    def __init__(self) -> None:
        self.calls: Counter[str] = Counter()

    def visit_Call(self, node: ast.Call) -> None:
        self.calls[ast.unparse(node.func)] += 1
        self.generic_visit(node)  # calls inside the arguments too


collector = CallCollector()
collector.visit(pricing_tree)
print(f"{dict(collector.calls)=}")

dict(collector.calls)={'sum': 1, 'round': 1}


**`NodeTransformer`:** a tiny version of what pytest does. It rewrites
`assert left <op> right` into an `if` that raises an `AssertionError` showing both
values. New nodes have no positions, so `compile` rejects the tree until
`ast.fix_missing_locations` fills them in. (This simple version evaluates each
side twice; pytest stores them in temporaries.)

In [120]:
OPERATOR_TEXT = {
    ast.Eq: "==",
    ast.NotEq: "!=",
    ast.Lt: "<",
    ast.LtE: "<=",
    ast.Gt: ">",
    ast.GtE: ">=",
}


class AssertExplainer(ast.NodeTransformer):
    """Rewrite `assert a <op> b` so the failure message shows both values."""

    def visit_Assert(self, node: ast.Assert) -> ast.AST:
        test = node.test
        if not (
            isinstance(test, ast.Compare)
            and len(test.ops) == 1
            and type(test.ops[0]) in OPERATOR_TEXT
        ):
            return node
        message = ast.JoinedStr(
            values=[
                ast.Constant("assert "),
                ast.FormattedValue(test.left, conversion=ord("r")),
                ast.Constant(f" {OPERATOR_TEXT[type(test.ops[0])]} "),
                ast.FormattedValue(test.comparators[0], conversion=ord("r")),
            ]
        )
        error_call = ast.Call(
            func=ast.Name("AssertionError", ast.Load()), args=[message], keywords=[]
        )
        replacement = ast.If(
            test=ast.UnaryOp(ast.Not(), test), body=[ast.Raise(exc=error_call)], orelse=[]
        )
        return ast.copy_location(replacement, node)

Apply it to a small test, compile before and after `fix_missing_locations`, and run it:

In [121]:
test_source = textwrap.dedent(
    """
    expected_total = 590.0
    actual_total = round(2 * 250 * 1.18, 2) + 0.5
    assert actual_total == expected_total
    """
)
rewritten_tree = AssertExplainer().visit(ast.parse(test_source))
print(ast.unparse(rewritten_tree).splitlines()[-2:])
print(f"compile before fix: {raised_error_name(lambda: compile(rewritten_tree, '<test>', 'exec'))}")
ast.fix_missing_locations(rewritten_tree)
print(
    f"run after fix: {raised_message(lambda: exec(compile(rewritten_tree, '<test>', 'exec'), {}))}"
)

['if not actual_total == expected_total:', "    raise AssertionError(f'assert {actual_total!r} == {expected_total!r}')"]
compile before fix: TypeError
run after fix: AssertionError: assert 590.5 == 590.0


**How pytest rewrites asserts.** pytest installs an import hook
(`_pytest.assertion.rewrite.AssertionRewritingHook`) for test modules and
`conftest.py`. When one is imported, the hook parses it, rewrites every `assert`
into code that saves each sub-expression in a temporary and builds a detailed
message, compiles the new tree and caches it as a `.pyc` with a pytest-specific
tag. That is why a plain `assert a == b` in a test prints both values and a diff.
pytest is a Phase 4 dependency; the cell checks for it.

In [122]:
if importlib.util.find_spec("pytest") is None:
    print("Skipped: pytest is not installed yet (added in Phase 4: uv add --dev pytest)")
else:
    from _pytest.assertion import rewrite

    print(f"{rewrite.AssertionRewritingHook.__name__} is in {rewrite.__name__}")

Skipped: pytest is not installed yet (added in Phase 4: uv add --dev pytest)


### 3.2.13 `.pyc` files

Importing a module compiles it, then caches the code object in
`__pycache__/<module>.<cache_tag>.pyc` (for example `pricing.cpython-315.pyc`), so
the next import can skip compilation. A `.pyc` is a 16-byte header plus the module's
code object serialized with `marshal`:

| Bytes | Field | Meaning |
| --- | --- | --- |
| 0–3 | magic number | changes whenever the bytecode format changes; must equal `importlib.util.MAGIC_NUMBER` |
| 4–7 | flags (little-endian) | `0` = timestamp-based; bit 0 = hash-based; bit 1 = check the source hash |
| 8–15 | source mtime + size (timestamp mode) **or** 8-byte SipHash of the source (hash modes) | how the cache is validated |
| 16– | `marshal.dumps(code_object)` | the module's code object |

The cell compiles a module with `py_compile`, decodes the header by hand, then
loads and runs the code object with `marshal`.

In [123]:
PRICING_SOURCE = (
    "GST_RATE = 0.18\n\ndef with_gst(amount):\n    return round(amount * (1 + GST_RATE), 2)\n"
)

with tempfile.TemporaryDirectory() as folder:
    source_path = Path(folder) / "pricing.py"
    source_path.write_text(PRICING_SOURCE)
    pyc_path = Path(py_compile.compile(str(source_path), doraise=True))
    pyc_bytes = pyc_path.read_bytes()
    source_stat = source_path.stat()

print(f"cache file: {pyc_path.parent.name}/{pyc_path.name}  ({len(pyc_bytes)} bytes)")
magic_ok = pyc_bytes[:4] == importlib.util.MAGIC_NUMBER
print(f"magic matches this interpreter: {magic_ok}  ({pyc_bytes[:4].hex(' ')})")
flags = int.from_bytes(pyc_bytes[4:8], "little")
stored_mtime, stored_size = struct.unpack("<II", pyc_bytes[8:16])
mtime_ok = stored_mtime == int(source_stat.st_mtime)
print(f"{flags=}  mtime matches: {mtime_ok}  size matches: {stored_size == source_stat.st_size}")
module_code = marshal.loads(pyc_bytes[16:])
pricing_namespace: dict[str, object] = {}
exec(module_code, pricing_namespace)
print(f"{type(module_code).__name__}  -> with_gst(100) = {pricing_namespace['with_gst'](100)}")
print(f"{sys.implementation.cache_tag=}")

cache file: __pycache__/pricing.cpython-315.pyc  (353 bytes)
magic matches this interpreter: True  (52 0e 0d 0a)
flags=0  mtime matches: True  size matches: True
code  -> with_gst(100) = 118.0
sys.implementation.cache_tag='cpython-315'


**Invalidation modes** (PEP 552, 3.7). Timestamps break reproducible builds and
some deployment tools, so a `.pyc` can store a hash of the source instead:

| `py_compile.PycInvalidationMode` | Flags | On import |
| --- | --- | --- |
| `TIMESTAMP` (default) | `0` | Recompile if the source mtime or size changed |
| `CHECKED_HASH` | `0b11` | Hash the source and compare (slower import, always correct) |
| `UNCHECKED_HASH` | `0b01` | Trust the `.pyc` without looking at the source (for read-only installs) |

In [124]:
with tempfile.TemporaryDirectory() as folder:
    source_path = Path(folder) / "pricing.py"
    source_path.write_text(PRICING_SOURCE)
    source_hash = importlib.util.source_hash(source_path.read_bytes())
    for mode in py_compile.PycInvalidationMode:
        target = Path(folder) / f"{mode.name.lower()}.pyc"
        header = Path(
            py_compile.compile(str(source_path), cfile=str(target), invalidation_mode=mode)
        ).read_bytes()[:16]
        flags = int.from_bytes(header[4:8], "little")
        stores_hash = header[8:16] == source_hash
        print(f"{mode.name:<15} flags={flags:#04b}  bytes 8-15 hold the source hash: {stores_hash}")

TIMESTAMP       flags=0b00  bytes 8-15 hold the source hash: False
CHECKED_HASH    flags=0b11  bytes 8-15 hold the source hash: True
UNCHECKED_HASH  flags=0b01  bytes 8-15 hold the source hash: True


**`compileall`** compiles a whole tree ahead of time (used by installers and in
Docker images so the first import is fast). `python -m compileall -q src/` is the
command-line form.

In [125]:
with tempfile.TemporaryDirectory() as folder:
    package = Path(folder) / "shop"
    package.mkdir()
    (package / "__init__.py").write_text("")
    (package / "cart.py").write_text("ITEMS = []\n")
    (package / "pricing.py").write_text(PRICING_SOURCE)
    succeeded = compileall.compile_dir(package, quiet=1)
    cached = sorted(path.name for path in (package / "__pycache__").iterdir())
print(f"{succeeded=}")
print(cached)

succeeded=True
['__init__.cpython-315.pyc', 'cart.cpython-315.pyc', 'pricing.cpython-315.pyc']


**`marshal`** is the serializer behind `.pyc` files. It handles code objects and
core types, its format can change between versions, and it is **not secure**
against crafted data. For your own data use `json` or `pickle` (Phase 4).

In [126]:
payload = marshal.dumps({"sku": "TEA-01", "sizes": (250, 500), "code": with_tax.__code__})
restored = marshal.loads(payload)
print(f"{len(payload)=}  {restored['sku']=}  {restored['sizes']=}")
print(f"{restored['code'] == with_tax.__code__=}  (a code object round-trips)")
print(f"{raised_message(lambda: marshal.dumps(CartLine(1, 1)))=}")

len(payload)=295  restored['sku']='TEA-01'  restored['sizes']=(250, 500)
restored['code'] == with_tax.__code__=True  (a code object round-trips)
raised_message(lambda: marshal.dumps(CartLine(1, 1)))='ValueError: unmarshallable object'


### 3.2.14 Other implementations

Everything in this unit describes **CPython**. Other implementations run the
same language very differently:

| Implementation | How it runs Python | Memory | C extensions |
| --- | --- | --- | --- |
| **CPython** | Bytecode interpreter + specialization (+ experimental JIT) | Refcounting + cycle GC | Native (the C API is CPython's) |
| **PyPy** | Written in RPython; a **tracing JIT** compiles hot loops to machine code | Tracing, moving GC: no refcounts, finalizers run late, `id()` is synthetic | Through an emulation layer (cpyext), slower |
| **GraalPy** | On GraalVM: Truffle AST interpreter + Graal JIT | JVM garbage collector | Partial support; can call Java |
| **Cython** (a compiler, not an interpreter) | Translates Python (plus optional C types) into C, compiled into an extension module | CPython's | Produces them |
| MicroPython | Small bytecode VM for microcontrollers | Simple GC | Its own native modules |

`sys.implementation` and `platform.python_implementation()` tell you which one is
running; code that depends on refcount timing, `sys.getrefcount` or `dis` output
is CPython-specific. The cell checks for PyPy and GraalPy executables.

In [127]:
print(f"{sys.implementation.name=}  {platform.python_implementation()=}")
for executable in ("pypy3", "graalpy"):
    found = shutil.which(executable)
    print(f"{executable:<8} {'installed' if found else 'not installed (skipped)'}")

sys.implementation.name='cpython'  platform.python_implementation()='CPython'


pypy3    not installed (skipped)
graalpy  not installed (skipped)


**A Cython-compiled module, from the inside.** pyzmq (Jupyter's messaging library,
installed with the kernel) is compiled with Cython. Its functions are not Python
function objects; they carry a placeholder code object for introspection, with
no real bytecode, because the body is machine code compiled from C.

In [128]:
cython_module_name = "zmq.backend.cython._zmq"
if importlib.util.find_spec("zmq") is None or importlib.util.find_spec(cython_module_name) is None:
    print("Skipped: pyzmq's Cython backend is not installed")
else:
    cython_module = importlib.import_module(cython_module_name)
    cython_function = cython_module.has
    print(f"module file: {Path(cython_module.__file__).name}")
    print(f"{type(cython_function).__name__=}")
    print(f"{inspect.isfunction(cython_function)=}  (not a Python function object)")
    placeholder = getattr(cython_function, "__code__", None)
    if placeholder is not None:
        real_instructions = [
            name
            for name in (i.opname for i in dis.get_instructions(placeholder))
            if name != "CACHE"
        ]
        print(f"bytecode instructions besides CACHE placeholders: {real_instructions}")

module file: _zmq.abi3.so
type(cython_function).__name__='cython_function_or_method'
inspect.isfunction(cython_function)=False  (not a Python function object)
bytecode instructions besides CACHE placeholders: []


**Predict the output.**

```python
import dis

def tax_tables():
    seconds = 24 * 60 * 60
    return seconds, [rate * 2 for rate in (5, 12, 18)]

print(86400 in tax_tables.__code__.co_consts)
print([c.co_name for c in tax_tables.__code__.co_consts if hasattr(c, "co_name")])
```

<details><summary>Answer</summary>

```text
True
[]
```

`24 * 60 * 60` is folded to `86400` at compile time, so the constant is stored in
`co_consts`. Since 3.12 (PEP 709) the list comprehension is inlined into the function,
so there is no nested `<listcomp>` code object. On 3.11 the second line printed
`['<listcomp>']`.

</details>

Run the next cell to check your prediction.

In [129]:
def tax_tables() -> tuple[int, list[int]]:
    """A folded constant and an inlined comprehension."""
    seconds = 24 * 60 * 60
    return seconds, [rate * 2 for rate in (5, 12, 18)]


print(86400 in tax_tables.__code__.co_consts)
print([c.co_name for c in tax_tables.__code__.co_consts if hasattr(c, "co_name")])

True
[]


### Lab 3.2

Seven exercises. For each, complete the stub, then run the check cell. Until then
the checks print FAIL; that is expected. Checks compare your answers with what the
**running** interpreter's `dis` reports, so they work on any version.

#### Exercise 1 — annotate a function's `dis` output with stack states

Run `show_dis(taxi_fare)`, then return one `(opname, stack_after)` pair per
instruction, **skipping `RESUME`**. `stack_after` lists what is on the value stack
after the instruction, bottom to top, as short strings such as
`"distance_km * rate_per_km"`. After `RETURN_VALUE` the stack is empty.

In [130]:
def taxi_fare(base_fare: float, distance_km: float, rate_per_km: float, discount: float) -> float:
    """The function to annotate (given)."""
    return base_fare + distance_km * rate_per_km - discount


def annotate_stack() -> list[tuple[str, list[str]]]:
    """(opname, stack after) for each instruction of taxi_fare except RESUME."""
    raise NotImplementedError("Lab 3.2: annotate taxi_fare")

Run the checks. They compare the opnames with `dis.get_instructions(taxi_fare)` and
the stack **depths** with `dis.stack_effect`; the wording of your stack entries is
up to you.

In [131]:
def live_instructions(function: Callable[..., object]) -> list[dis.Instruction]:
    """Instructions without RESUME."""
    return [ins for ins in dis.get_instructions(function) if ins.opname != "RESUME"]


def expected_depths(function: Callable[..., object]) -> list[int]:
    """Stack depth after each instruction, from dis.stack_effect."""
    depth, depths = 0, []
    for instruction in live_instructions(function):
        if instruction.opname == "RETURN_VALUE":
            depth -= 1
        elif instruction.arg is None:
            depth += dis.stack_effect(instruction.opcode)
        else:
            depth += dis.stack_effect(instruction.opcode, instruction.arg)
        depths.append(depth)
    return depths


run_checks(
    [
        (
            "opnames in order",
            lambda: [name for name, _ in annotate_stack()],
            [ins.opname for ins in live_instructions(taxi_fare)],
        ),
        (
            "stack depths",
            lambda: [len(stack) for _, stack in annotate_stack()],
            expected_depths(taxi_fare),
        ),
        (
            "entries are strings",
            lambda: all(isinstance(item, str) for _, stack in annotate_stack() for item in stack),
            True,
        ),
        ("empty after RETURN_VALUE", lambda: annotate_stack()[-1], ("RETURN_VALUE", [])),
    ]
)

FAIL  opnames in order: raised NotImplementedError: Lab 3.2: annotate taxi_fare
FAIL  stack depths: raised NotImplementedError: Lab 3.2: annotate taxi_fare
FAIL  entries are strings: raised NotImplementedError: Lab 3.2: annotate taxi_fare
FAIL  empty after RETURN_VALUE: raised NotImplementedError: Lab 3.2: annotate taxi_fare
0/4 checks passed


#### Exercise 2 — `count_opcodes(func)`

Return a `collections.Counter` mapping each opname to how often it appears in
`func`'s bytecode **and in every nested code object** (inner functions, lambdas,
generator expressions, found recursively in `co_consts`). Use the non-adaptive
instructions from `dis.get_instructions(code_object)`; `CACHE` entries are not
instructions and must not be counted.

> **Note:** since 3.14, annotations are evaluated lazily: a function with type
> hints carries a hidden `__annotate__` code object, which counts as nested code
> too.

In [132]:
def count_opcodes(func: Callable[..., object]) -> Counter[str]:
    """Opname counts for func and all nested code objects."""
    raise NotImplementedError("Lab 3.2: implement count_opcodes")

Run the checks.

In [133]:
def three_calls(values: list[int]) -> int:
    print(values)
    print(values)
    return len(values)


def outer_with_inner() -> Callable[[], int]:
    def inner() -> int:
        return 1

    return inner


def with_generator(values: list[int]) -> int:
    return sum(value * 2 for value in values)


def total_instructions(code_object: types.CodeType) -> int:
    """Instruction count of a code object and its nested code objects (check helper)."""
    nested = [c for c in code_object.co_consts if isinstance(c, types.CodeType)]
    return len(list(dis.get_instructions(code_object))) + sum(total_instructions(c) for c in nested)


run_checks(
    [
        ("returns a Counter", lambda: type(count_opcodes(three_calls)).__name__, "Counter"),
        (
            "flat total",
            lambda: count_opcodes(three_calls).total(),
            len(list(dis.get_instructions(three_calls))),
        ),
        ("three CALLs", lambda: count_opcodes(three_calls)["CALL"], 3),
        (
            "inner function counted",
            lambda: (
                count_opcodes(outer_with_inner)["RETURN_VALUE"]
                > opnames(outer_with_inner).count("RETURN_VALUE")
            ),
            True,
        ),
        (
            "generator expression counted",
            lambda: count_opcodes(with_generator).total(),
            total_instructions(with_generator.__code__),
        ),
        ("no CACHE entries", lambda: "CACHE" in count_opcodes(with_generator), False),
    ]
)

FAIL  returns a Counter: raised NotImplementedError: Lab 3.2: implement count_opcodes
FAIL  flat total: raised NotImplementedError: Lab 3.2: implement count_opcodes
FAIL  three CALLs: raised NotImplementedError: Lab 3.2: implement count_opcodes
FAIL  inner function counted: raised NotImplementedError: Lab 3.2: implement count_opcodes
FAIL  generator expression counted: raised NotImplementedError: Lab 3.2: implement count_opcodes
FAIL  no CACHE entries: raised NotImplementedError: Lab 3.2: implement count_opcodes
0/6 checks passed


#### Exercise 3 — an AST lint rule for mutable defaults and bare `except`

`lint_source(source)` parses `source` and returns a sorted list of
`(line_number, code)` tuples:

- `"B006"` for every **mutable default** of a `def` or `async def`, positional or
  keyword-only: a list, dict or set display, a list/dict/set comprehension, or a
  call to `list()`, `dict()` or `set()`. Report the line of the default value.
- `"E722"` for every bare `except:` (an `ExceptHandler` with no type), reported at
  the `except` line.

Lambdas and `None` defaults are fine.

In [134]:
def lint_source(source: str) -> list[tuple[int, str]]:
    """(line, code) findings for mutable defaults (B006) and bare except (E722)."""
    raise NotImplementedError("Lab 3.2: implement lint_source")

Run the checks.

In [135]:
LINT_SAMPLE = textwrap.dedent(
    """
    def add_item(item, basket=[]):
        basket.append(item)
        return basket

    async def fetch(url, *, headers={}, retries=3, tags=set()):
        try:
            return url
        except:
            return None

    def fine(value=None, labels=(), factor=lambda x: x):
        try:
            return value
        except ValueError:
            return None

    def comprehension_default(cache={key: 0 for key in "ab"}):
        return cache
    """
)

run_checks(
    [
        (
            "sample findings",
            lambda: lint_source(LINT_SAMPLE),
            [(2, "B006"), (6, "B006"), (6, "B006"), (9, "E722"), (18, "B006")],
        ),
        ("clean code", lambda: lint_source("def f(x=None):\n    return x\n"), []),
        ("list() call default", lambda: lint_source("def f(x=list()):\n    pass\n"), [(1, "B006")]),
        (
            "bare except in a method",
            lambda: lint_source(
                "class A:\n    def m(self):\n        try:\n            pass\n"
                "        except:\n            pass\n"
            ),
            [(5, "E722")],
        ),
    ]
)

FAIL  sample findings: raised NotImplementedError: Lab 3.2: implement lint_source
FAIL  clean code: raised NotImplementedError: Lab 3.2: implement lint_source
FAIL  list() call default: raised NotImplementedError: Lab 3.2: implement lint_source
FAIL  bare except in a method: raised NotImplementedError: Lab 3.2: implement lint_source
0/4 checks passed


#### Exercise 4 — a `print` → `logger.info` transformer

Write `PrintToLogger(ast.NodeTransformer)` and `rewrite_prints(source)`, which
parses, transforms, fixes locations and returns `ast.unparse(tree)`:

- `print(a, b, ...)` with only positional, non-starred arguments becomes
  `logger.info('%s %s ...', a, b, ...)` (one `%s` per argument, joined by spaces).
- `print()` becomes `logger.info('')`.
- Calls with keyword arguments (`sep=`, `end=`, `file=`) or starred arguments are
  left unchanged.
- Other calls are untouched; prints nested in functions and arguments are
  rewritten too.

In [136]:
class PrintToLogger(ast.NodeTransformer):
    """Replace print(...) calls with logger.info(...)."""


def rewrite_prints(source: str) -> str:
    """Source with eligible print calls turned into logger.info calls."""
    raise NotImplementedError("Lab 3.2: implement rewrite_prints")

Run the checks.

In [137]:
run_checks(
    [
        ("one argument", lambda: rewrite_prints("print(total)"), "logger.info('%s', total)"),
        (
            "two arguments",
            lambda: rewrite_prints("print('total', total)"),
            "logger.info('%s %s', 'total', total)",
        ),
        ("no arguments", lambda: rewrite_prints("print()"), "logger.info('')"),
        (
            "keywords untouched",
            lambda: rewrite_prints("print(x, file=sys.stderr)"),
            "print(x, file=sys.stderr)",
        ),
        ("starred untouched", lambda: rewrite_prints("print(*rows)"), "print(*rows)"),
        ("other calls untouched", lambda: rewrite_prints("pprint(data)"), "pprint(data)"),
        (
            "nested in a function",
            lambda: rewrite_prints("def report(n):\n    print('rows', n)\n    return n"),
            "def report(n):\n    logger.info('%s %s', 'rows', n)\n    return n",
        ),
        (
            "result compiles",
            lambda: bool(compile(rewrite_prints("print(1, 2)"), "<t>", "exec")),
            True,
        ),
    ]
)

FAIL  one argument: raised NotImplementedError: Lab 3.2: implement rewrite_prints
FAIL  two arguments: raised NotImplementedError: Lab 3.2: implement rewrite_prints
FAIL  no arguments: raised NotImplementedError: Lab 3.2: implement rewrite_prints
FAIL  keywords untouched: raised NotImplementedError: Lab 3.2: implement rewrite_prints
FAIL  starred untouched: raised NotImplementedError: Lab 3.2: implement rewrite_prints
FAIL  other calls untouched: raised NotImplementedError: Lab 3.2: implement rewrite_prints
FAIL  nested in a function: raised NotImplementedError: Lab 3.2: implement rewrite_prints
FAIL  result compiles: raised NotImplementedError: Lab 3.2: implement rewrite_prints
0/8 checks passed


#### Exercise 5 — a safe arithmetic evaluator

`safe_eval(expression, variables=None)` evaluates arithmetic **without** `eval`,
by walking the AST from `ast.parse(expression, mode="eval")`:

- Numbers: `int` and `float` constants only (`bool`, strings and others raise
  `ValueError(f"unsupported constant: {type(value).__name__}")`).
- Names: looked up in `variables`; unknown names raise
  `ValueError(f"unknown name: {name}")`.
- Operators: binary `+ - * / // % **` and unary `+ -`. For `**`, an exponent whose
  absolute value is over 100 raises `ValueError("exponent too large")`.
- Any other node raises `ValueError(f"unsupported expression: {type(node).__name__}")`.
- Text that does not parse raises `ValueError("invalid expression")`.
- Arithmetic errors such as `ZeroDivisionError` propagate unchanged.

In [138]:
def safe_eval(expression: str, variables: dict[str, float] | None = None) -> float:
    """Evaluate an arithmetic expression safely."""
    raise NotImplementedError("Lab 3.2: implement safe_eval")

Run the checks.

In [139]:
prices = {"price": 200.0, "rate": 0.25, "quantity": 3}
run_checks(
    [
        ("precedence", lambda: safe_eval("2 + 3 * 4"), 14),
        ("parentheses", lambda: safe_eval("(2 + 3) * 4"), 20),
        ("unary minus and power", lambda: safe_eval("-x ** 2", {"x": 3}), -9),
        ("variables", lambda: safe_eval("price * quantity * (1 + rate)", prices), 750.0),
        ("floor division and modulo", lambda: safe_eval("17 // 5 + 17 % 5"), 5),
        (
            "function call refused",
            lambda: raised_message(lambda: safe_eval("__import__('os')")),
            "ValueError: unsupported expression: Call",
        ),
        (
            "attribute refused",
            lambda: raised_message(lambda: safe_eval("price.real", prices)),
            "ValueError: unsupported expression: Attribute",
        ),
        (
            "huge exponent refused",
            lambda: raised_message(lambda: safe_eval("2 ** 1000")),
            "ValueError: exponent too large",
        ),
        (
            "string refused",
            lambda: raised_message(lambda: safe_eval("'a' * 3")),
            "ValueError: unsupported constant: str",
        ),
        (
            "bool refused",
            lambda: raised_message(lambda: safe_eval("True + 1")),
            "ValueError: unsupported constant: bool",
        ),
        (
            "unknown name",
            lambda: raised_message(lambda: safe_eval("tax * 2")),
            "ValueError: unknown name: tax",
        ),
        (
            "invalid syntax",
            lambda: raised_message(lambda: safe_eval("2 +")),
            "ValueError: invalid expression",
        ),
        (
            "division by zero propagates",
            lambda: raised_error_name(lambda: safe_eval("1 / 0")),
            "ZeroDivisionError",
        ),
    ]
)

FAIL  precedence: raised NotImplementedError: Lab 3.2: implement safe_eval
FAIL  parentheses: raised NotImplementedError: Lab 3.2: implement safe_eval
FAIL  unary minus and power: raised NotImplementedError: Lab 3.2: implement safe_eval
FAIL  variables: raised NotImplementedError: Lab 3.2: implement safe_eval
FAIL  floor division and modulo: raised NotImplementedError: Lab 3.2: implement safe_eval
FAIL  function call refused: expected 'ValueError: unsupported expression: Call', got 'NotImplementedError: Lab 3.2: implement safe_eval'
FAIL  attribute refused: expected 'ValueError: unsupported expression: Attribute', got 'NotImplementedError: Lab 3.2: implement safe_eval'
FAIL  huge exponent refused: expected 'ValueError: exponent too large', got 'NotImplementedError: Lab 3.2: implement safe_eval'
FAIL  string refused: expected 'ValueError: unsupported constant: str', got 'NotImplementedError: Lab 3.2: implement safe_eval'
FAIL  bool refused: expected 'ValueError: unsupported constant: bo

#### Exercise 6 — observe specialization and deoptimization

`binary_op_forms(func, argument_sets, *, warmups=1000)`: for each tuple in
`argument_sets`, call `func(*arguments)` `warmups` times, then record the
**adaptive** opname of the first instruction whose opname starts with
`"BINARY_OP"` (`dis.get_instructions(func, adaptive=True)`). Return the list of
recorded names.

Feeding ints, then floats, then strings shows the instruction specializing,
deoptimizing when its guard fails, and re-specializing for the new type.

In [140]:
def binary_op_forms(
    func: Callable[..., object], argument_sets: list[tuple[object, ...]], *, warmups: int = 1000
) -> list[str]:
    """The adaptive BINARY_OP form after warming func with each argument set in turn."""
    raise NotImplementedError("Lab 3.2: implement binary_op_forms")

Run the checks. Specialization state lives in the **code object**, and every
function created by the same nested `def` shares one code object. So
`make_adder()` compiles a brand-new code object with `exec` each time; otherwise
the "cold" function would already be warm from an earlier check. Exact names are
CPython internals; the checks only look for the type in the name.

In [141]:
def make_adder() -> Callable[[object, object], object]:
    """A never-called function with its own, freshly compiled code object."""
    namespace: dict[str, object] = {}
    exec("def add(left, right):\n    return left + right", namespace)
    return namespace["add"]


def type_shifts() -> list[str]:
    return binary_op_forms(make_adder(), [(1, 2), (1.5, 2.5), ("a", "b")])


try:
    print(f"[{interpreter_label()}] {type_shifts()=}")
except NotImplementedError:
    print("binary_op_forms is not implemented yet")
run_checks(
    [
        (
            "cold function is generic",
            lambda: binary_op_forms(make_adder(), [(1, 2)], warmups=0),
            ["BINARY_OP"],
        ),
        ("ints specialize", lambda: "INT" in type_shifts()[0], True),
        ("then floats", lambda: "FLOAT" in type_shifts()[1], True),
        ("then strings", lambda: "UNICODE" in type_shifts()[2], True),
        (
            "always a BINARY_OP form",
            lambda: all(name.startswith("BINARY_OP") for name in type_shifts()),
            True,
        ),
    ]
)

binary_op_forms is not implemented yet
FAIL  cold function is generic: raised NotImplementedError: Lab 3.2: implement binary_op_forms
FAIL  ints specialize: raised NotImplementedError: Lab 3.2: implement binary_op_forms
FAIL  then floats: raised NotImplementedError: Lab 3.2: implement binary_op_forms
FAIL  then strings: raised NotImplementedError: Lab 3.2: implement binary_op_forms
FAIL  always a BINARY_OP form: raised NotImplementedError: Lab 3.2: implement binary_op_forms
0/5 checks passed


#### Exercise 7 — a code-quality scanner over a folder

`scan_folder(folder)` walks `folder` recursively and runs `lint_source`
(exercise 3) on every `.py` file:

- Skip directories named `__pycache__` or starting with `.` (such as `.venv`).
- Keys are paths **relative to `folder`**, in POSIX form (`"pkg/models.py"`).
- A file that does not parse gets `[(error.lineno, "E999")]`.
- Files with no findings are left out; return the dict with keys sorted.

In [142]:
def scan_folder(folder: Path) -> dict[str, list[tuple[int, str]]]:
    """Lint findings for every Python file under folder."""
    raise NotImplementedError("Lab 3.2: implement scan_folder")

Run the checks. They build a small project in a temporary folder.

In [143]:
def scan_sample_project() -> dict[str, list[tuple[int, str]]]:
    """Write a sample tree, scan it, delete it."""
    files = {
        "app.py": "def handler(event, seen=[]):\n    return seen\n",
        "pkg/__init__.py": "",
        "pkg/models.py": "def load():\n    try:\n        pass\n    except:\n        pass\n",
        "pkg/clean.py": "def ok(value=None):\n    return value\n",
        "broken.py": "def broken(:\n    pass\n",
        ".venv/lib/site.py": "def bad(x=[]):\n    pass\n",
        "pkg/__pycache__/junk.py": "def bad(x={}):\n    pass\n",
        "notes.txt": "def not_python(x=[]): pass\n",
    }
    with tempfile.TemporaryDirectory() as folder:
        for relative, text in files.items():
            path = Path(folder) / relative
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(text)
        return scan_folder(Path(folder))


run_checks(
    [
        (
            "findings per file",
            scan_sample_project,
            {"app.py": [(1, "B006")], "broken.py": [(1, "E999")], "pkg/models.py": [(4, "E722")]},
        ),
        (
            "keys sorted",
            lambda: list(scan_sample_project()),
            ["app.py", "broken.py", "pkg/models.py"],
        ),
    ]
)

FAIL  findings per file: raised NotImplementedError: Lab 3.2: implement scan_folder
FAIL  keys sorted: raised NotImplementedError: Lab 3.2: implement scan_folder
0/2 checks passed


### Recap — Unit 3.2

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Architecture | `Parser/` → `Python/compile.c` → `Python/ceval.c`; types in `Objects/` | V8 parser → Ignition → TurboFan |
| Tokens and AST | `tokenize`, `ast.parse`, `ast.dump`, `ast.unparse` | Acorn / Babel ASTs |
| Code objects | Immutable bytecode + constants + names + flags + exception table; `replace()` | `SharedFunctionInfo` + bytecode array |
| Function objects | Code + globals + defaults + closure; `types.FunctionType` builds one | Closure objects |
| Reading `dis` | Columns, a value stack, `*_FAST` / `*_DEREF` / `*_GLOBAL` / `*_ATTR`, `CALL`, jumps, `FOR_ITER` | `node --print-bytecode` |
| Compiler optimizations | Constant folding, dead code, constant containers, inlined comprehensions (3.12) | Parser-level folding |
| Eval loop | Frames with an instruction pointer; Python-to-Python calls inlined (3.11) | Ignition dispatch loop |
| Specialization | Instructions rewrite themselves for observed types; guards and deopt | Inline caches + hidden classes |
| JIT | Hot loops → uop traces → copy-and-patch machine code; `PYTHON_JIT=1` | Sparkplug / Maglev / TurboFan |
| GIL and eval breaker | Switches, signals and GC only between instructions | Single-threaded event loop |
| `compile` / `exec` / `eval` | Never on untrusted input; `ast.literal_eval` for literals | `eval`, `new Function` |
| AST transformation | `NodeVisitor`, `NodeTransformer`, `fix_missing_locations`; pytest asserts | Babel plugins |
| `.pyc` | 16-byte header (magic, flags, mtime/size or hash) + `marshal` | V8 code cache |
| Other implementations | PyPy (tracing JIT), GraalPy, Cython-compiled extensions | SpiderMonkey, JavaScriptCore |

**Further reading**

- [`dis`](https://docs.python.org/3/library/dis.html),
  [`ast`](https://docs.python.org/3/library/ast.html),
  [`tokenize`](https://docs.python.org/3/library/tokenize.html),
  [`inspect` — code objects](https://docs.python.org/3/library/inspect.html#types-and-members),
  [`types.CodeType`](https://docs.python.org/3/library/types.html#types.CodeType)
- [`py_compile`](https://docs.python.org/3/library/py_compile.html),
  [`compileall`](https://docs.python.org/3/library/compileall.html),
  [`marshal`](https://docs.python.org/3/library/marshal.html),
  [`sys.setswitchinterval`](https://docs.python.org/3/library/sys.html#sys.setswitchinterval)
- [CPython internals docs](https://github.com/python/cpython/tree/main/InternalDocs)
  (interpreter, compiler, frames, exception table)
- [PEP 659 — Specializing adaptive interpreter](https://peps.python.org/pep-0659/),
  [PEP 709 — Inlined comprehensions](https://peps.python.org/pep-0709/),
  [PEP 744 — JIT compilation](https://peps.python.org/pep-0744/),
  [PEP 552 — Deterministic pycs](https://peps.python.org/pep-0552/),
  [PEP 703 — Making the GIL optional](https://peps.python.org/pep-0703/)

## Unit 3.3 — Modules, Packages and the Import System

**Objectives**

- Treat modules as ordinary objects and read their metadata (`__spec__`,
  `__loader__`, `__path__`).
- Describe exactly what `import x` does: the `sys.modules` cache, finders on
  `sys.meta_path`, loaders, execution and name binding.
- Predict the effect of every import form, including `from x import *`, and why
  `from x import y` copies a binding.
- Explain how `sys.path` is built and how `python -m`, `python file.py` and
  `python -P` differ.
- Structure packages (regular and namespace), choose between absolute and relative
  imports, and write entry points.
- Diagnose and fix circular imports.
- Use `importlib`, `importlib.resources`, `importlib.metadata` and `pkgutil`;
  write an import hook; cut startup time with lazy imports (*New in 3.15*).

**JS bridge**

| JavaScript (Node.js) | Python | The difference that bites |
| --- | --- | --- |
| `require.cache` | `sys.modules` | Both run a module once and cache it; Python's cache is an ordinary dict you can edit |
| ESM `import { x } from "./m.js"` (live binding) | `from m import x` (a **copy** of the binding) | Rebinding `m.x` later does not change your `x` |
| `node_modules` lookup, walking up the directories | A flat list, `sys.path`, searched in order | No per-folder dependency trees; one version per environment |
| `package.json` `"main"` / `"exports"` | `__init__.py`, `__all__` | `__init__.py` is code that runs on import |
| `require.main === module` | `if __name__ == "__main__":` | Same idea |
| Circular `require` returns a partial `exports` | Circular import sees a *partially initialized module* | Same root cause, different error |
| `import()` (dynamic) | `importlib.import_module(name)` | Python's is synchronous |
| Hoisted, static ESM imports | `import` is a statement executed in order | Imports can sit inside functions and `if` blocks |

**How the demos run.** Importing a throwaway module into this kernel would leave it
in `sys.modules` (and its folder in `sys.path`) for the rest of the notebook. So
every import demo writes its small project under `notebooks/_scratch/` with
`scratch_project(name, files)` and runs it in a **child interpreter** with
`run_script(root, script)`, which saves the script as `main.py` in that folder and
runs it from there. The folder is deleted afterwards, and absolute paths in the
child's output are shown as `<project>`. A check after 3.3.13 proves the kernel
stayed clean.

In [144]:
import ast
import importlib
import importlib.abc
import importlib.machinery
import importlib.metadata
import importlib.resources
import importlib.util
import json
import os
import pkgutil
import platform
import shutil
import site
import subprocess
import sys
import sysconfig
import tempfile
import textwrap
import types
import zipfile
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from pathlib import Path

CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0"}  # plain output from child interpreters
scratch_dir = find_repo_root() / "notebooks" / "_scratch"  # git-ignored
scratch_dir.mkdir(parents=True, exist_ok=True)
KERNEL_PATH_AT_START = list(sys.path)  # compared again after the demos


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def compile_error(source: str) -> str:
    """Compile without running; return "SyntaxError: <msg>" or "compiles"."""
    try:
        compile(source, "<demo>", "exec")
    except SyntaxError as error:
        return f"SyntaxError: {error.msg}"
    return "compiles"


def interpreter_label() -> str:
    """Implementation, version and build flavour of the running interpreter."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"


def run_python(
    *arguments: str, cwd: Path | None = None, extra_env: dict[str, str] | None = None
) -> str:
    """Run the kernel's interpreter in a child process; return its output, stripped.

    The folder passed as cwd is shown as <project>, so no absolute path is printed.
    """
    completed = subprocess.run(
        [sys.executable, *arguments],
        capture_output=True, text=True, cwd=cwd, timeout=120,
        env={**CHILD_ENV, **(extra_env or {})},
    )  # fmt: skip
    output = (completed.stdout + completed.stderr).strip()
    return output.replace(str(cwd), "<project>") if cwd is not None else output


@contextmanager
def scratch_project(name: str, files: dict[str, str]) -> Iterator[Path]:
    """Write files under notebooks/_scratch/<name>/ for a child process; delete them afterwards."""
    root = scratch_dir / name
    shutil.rmtree(root, ignore_errors=True)
    root.mkdir(parents=True)
    for relative_path, text in files.items():
        target = root / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(textwrap.dedent(text).lstrip("\n"))
    try:
        yield root
    finally:
        shutil.rmtree(root, ignore_errors=True)


def run_script(
    root: Path, script: str, *arguments: str, extra_env: dict[str, str] | None = None
) -> str:
    """Save script as root/main.py and run it in a child interpreter from root.

    The child's sys.path[0] is root, so the project's modules are imported there, never in
    the kernel.
    """
    (root / "main.py").write_text(textwrap.dedent(script).lstrip("\n"))
    return run_python("main.py", *arguments, cwd=root, extra_env=extra_env)


def in_src(files: dict[str, str]) -> dict[str, str]:
    """The same files moved into src/, a folder the script adds to sys.path itself."""
    return {f"src/{relative_path}": text for relative_path, text in files.items()}


def child_json(output: str) -> object:
    """Parse the last line of a child's output as JSON; raise with the child's error if not."""
    last_line = output.splitlines()[-1] if output else ""
    try:
        return json.loads(last_line)
    except ValueError:
        raise RuntimeError(f"child process failed: {last_line}") from None


def _comes_from(module: object, folder: str) -> bool:
    """Was this module loaded from somewhere under folder?"""
    spec = getattr(module, "__spec__", None)
    locations = [
        getattr(spec, "origin", None) or "",
        *(getattr(spec, "submodule_search_locations", None) or []),
    ]
    return any(str(location).startswith(folder) for location in locations)


@contextmanager
def temporary_modules(files: dict[str, str], *, zipped: bool = False) -> Iterator[Path]:
    """For lab checks that must import into the kernel: write files under notebooks/_scratch
    (or zip them), put the folder first on sys.path, and on exit remove it and forget every
    module imported from it."""
    with tempfile.TemporaryDirectory(prefix="unit33_lab_", dir=scratch_dir) as folder:
        root = Path(folder) / "src"
        root.mkdir()
        for relative_path, text in files.items():
            target = root / relative_path
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_text(textwrap.dedent(text))
        entry = root
        if zipped:
            entry = Path(folder) / "bundle.zip"
            with zipfile.ZipFile(entry, "w") as archive:
                for path in sorted(root.rglob("*")):
                    if path.is_file():
                        archive.write(path, path.relative_to(root).as_posix())
        sys.path.insert(0, str(entry))
        importlib.invalidate_caches()
        try:
            yield root
        finally:
            sys.path.remove(str(entry))
            sys.path_importer_cache.pop(str(entry), None)
            for name, module in list(sys.modules.items()):
                if _comes_from(module, folder):
                    del sys.modules[name]
            importlib.invalidate_caches()


print(interpreter_label())

CPython 3.15.1, default (GIL) build, x86_64


### 3.3.1 Modules as objects

A module is an instance of `types.ModuleType` whose `__dict__` is the module's
global namespace. Its special attributes record where it came from:

| Attribute | Meaning | Example (`json.decoder`) |
| --- | --- | --- |
| `__name__` | Fully qualified name | `"json.decoder"` |
| `__file__` | Source or extension file (absent for built-in and namespace modules) | `.../json/decoder.py` |
| `__spec__` | `ModuleSpec`: name, loader, origin, search locations; the import system's record | `ModuleSpec(name='json.decoder', ...)` |
| `__loader__` | The object that executed the module | `SourceFileLoader` |
| `__package__` | Package used to resolve relative imports | `"json"` |
| `__path__` | **Packages only**: where to look for submodules | `[".../json"]` |

**`types.ModuleType`.** You can create a module by hand and run code in its
namespace; that is essentially what a loader does.

In [145]:
pricing = types.ModuleType("pricing", "Hand-made module for GST maths.")
exec(
    "GST_RATE = 0.18\ndef with_gst(amount):\n    return round(amount * (1 + GST_RATE), 2)",
    pricing.__dict__,
)
print(f"{type(pricing).__name__=}  {pricing.__name__=}  {pricing.__doc__=}")
print(f"{pricing.with_gst(100)=}")
print(
    f"{pricing.with_gst.__globals__ is vars(pricing)=}  (the module dict is the functions' globals)"
)
print(f"{type(json) is types.ModuleType=}")

type(pricing).__name__='module'  pricing.__name__='pricing'  pricing.__doc__='Hand-made module for GST maths.'
pricing.with_gst(100)=118.0
pricing.with_gst.__globals__ is vars(pricing)=True  (the module dict is the functions' globals)
type(json) is types.ModuleType=True


**`__name__` and `__package__`.** A package's `__package__` is its own name; a
submodule's is the package it lives in. A top-level module has `""`.

In [146]:
import json.decoder

for module in (json, json.decoder, textwrap):
    print(f"{module.__name__:<13} __package__={module.__package__!r}")

json          __package__='json'
json.decoder  __package__='json'
textwrap      __package__=''


**`__file__` and `__path__`.** Only packages have `__path__`. Built-in modules
(compiled into the interpreter) have no `__file__`. File names only are printed.

In [147]:
for module in (json, json.decoder, sys):
    file_name = Path(module.__file__).name if getattr(module, "__file__", None) else None
    package_path = [Path(entry).name for entry in getattr(module, "__path__", [])] or None
    print(f"{module.__name__:<13} __file__={file_name!r:<16} __path__={package_path}")

json          __file__='__init__.py'    __path__=['json']
json.decoder  __file__='decoder.py'     __path__=None
sys           __file__=None             __path__=None


**`__spec__` and `__loader__`.** The spec is what the finder returned; the loader
is what ran the module. Different kinds of module have different loaders.

In [148]:
for module in (
    json.decoder,
    sys,
    importlib.util.module_from_spec(importlib.util.find_spec("zipimport")),
):
    spec = module.__spec__
    origin = spec.origin if spec.origin in {"built-in", "frozen"} else Path(spec.origin).name
    loader = module.__loader__
    loader_name = loader.__name__ if isinstance(loader, type) else type(loader).__name__
    print(f"{spec.name:<13} origin={origin!r:<14} loader={loader_name}")
print(f"{json.decoder.__spec__.parent=}  {json.__spec__.submodule_search_locations is not None=}")

json.decoder  origin='decoder.py'   loader=SourceFileLoader
sys           origin='built-in'     loader=BuiltinImporter
zipimport     origin='frozen'       loader=FrozenImporter
json.decoder.__spec__.parent='json'  json.__spec__.submodule_search_locations is not None=True


### 3.3.2 What `import` does

```mermaid
flowchart TD
    A["import checkout_rules"] --> B{"in sys.modules?"}
    B -- yes --> H["bind the name"]
    B -- no --> C["ask each finder on sys.meta_path: find_spec(name, path)"]
    C -- "all return None" --> E["ModuleNotFoundError"]
    C -- "a ModuleSpec" --> D["loader.create_module(spec) or a new ModuleType"]
    D --> F["sys.modules[name] = module  (BEFORE running it)"]
    F --> G["loader.exec_module(module): run its code"]
    G --> H
```

<details><summary>ASCII fallback</summary>

```text
import x
  1. x in sys.modules?            yes -> step 6
  2. for finder in sys.meta_path: spec = finder.find_spec("x", path)   (None = next finder)
     no finder found it           -> ModuleNotFoundError
  3. module = spec.loader.create_module(spec) or ModuleType("x")
  4. sys.modules["x"] = module    (inserted before execution: this is what makes cycles possible)
  5. spec.loader.exec_module(module)   (runs the module's code in module.__dict__)
  6. bind the name x in the importing namespace
```

</details>

**Step 1, the cache.** A second import of the same name is a dict lookup. Whatever
object sits in `sys.modules` **is** the module, even a fake one. (The script runs in
a child interpreter, so the fake never reaches this kernel.)

In [149]:
CACHE_DEMO = """
import json
import json as json_again
import sys
import types

print("json_again is json is sys.modules['json']:", json_again is json is sys.modules["json"])
sys.modules["fake_payments"] = types.SimpleNamespace(charge=lambda amount: f"faked {amount}")
import fake_payments

print("fake_payments.charge(99) ->", fake_payments.charge(99), "(whatever was cached)")
"""
print(run_python("-c", textwrap.dedent(CACHE_DEMO)))

json_again is json is sys.modules['json']: True
fake_payments.charge(99) -> faked 99 (whatever was cached)


**Step 2, the finders.** `sys.meta_path` lists finder objects, asked in order.
`importlib.util.find_spec(name)` runs the same search **without executing**
anything, so it is safe to call in the kernel.

In [150]:
finder_names = [getattr(finder, "__name__", type(finder).__name__) for finder in sys.meta_path]
print(f"{finder_names=}")
for name in ("sys", "zipimport", "json", "_json", "no_such_module_here"):
    spec = importlib.util.find_spec(name)
    if spec is None:
        where = "None (ModuleNotFoundError on import)"
    elif spec.origin in {"built-in", "frozen"}:
        where = spec.origin
    else:
        where = Path(spec.origin).name
    print(f"find_spec({name!r:<22}) -> {where}")

finder_names=['BuiltinImporter', 'FrozenImporter', 'PathFinder', '_SixMetaPathImporter']
find_spec('sys'                 ) -> built-in
find_spec('zipimport'           ) -> frozen
find_spec('json'                ) -> __init__.py
find_spec('_json'               ) -> built-in
find_spec('no_such_module_here' ) -> None (ModuleNotFoundError on import)


**Watching the steps.** A finder at the front of `sys.meta_path` that only records
requests shows which imports reach the finders: `checkout_rules` and its helper do,
the already-cached `json` and `sys` do not. Inside the module,
`__name__ in sys.modules` is already `True` while its code runs (step 4 before
step 5).

In [151]:
TRACING_FILES = {
    "checkout_rules.py": """
        import json
        import sys

        import checkout_helpers

        IN_SYS_MODULES_WHILE_RUNNING = __name__ in sys.modules
        """,
    "checkout_helpers.py": "FREE_SHIPPING_ABOVE = 499\n",
}
TRACING_SCRIPT = """
import json  # loaded before the tracer is installed, so later imports hit the cache
import sys


class TracingFinder:
    # A meta path finder that records requests and lets the next finder answer.
    def __init__(self):
        self.requests = []

    def find_spec(self, fullname, path, target=None):
        self.requests.append(fullname)
        return None


tracer = TracingFinder()
sys.meta_path.insert(0, tracer)
import checkout_rules

sys.meta_path.remove(tracer)
print("requests that reached the finders:", tracer.requests)
print("in sys.modules while its code ran:", checkout_rules.IN_SYS_MODULES_WHILE_RUNNING)
"""
with scratch_project("unit33_tracing", TRACING_FILES) as root:
    print(run_script(root, TRACING_SCRIPT))

requests that reached the finders: ['checkout_rules', 'checkout_helpers']
in sys.modules while its code ran: True


**The whole algorithm in a few lines.** `simple_import` follows the steps for a
top-level module with the real `importlib` building blocks (the real code also
handles packages, parent imports and locking).

In [152]:
SIMPLE_IMPORT_SCRIPT = """
import importlib.util
import sys


def simple_import(module_name):
    # The core of `import module_name` for a top-level module.
    if module_name in sys.modules:  # 1. cache
        return sys.modules[module_name]
    spec = importlib.util.find_spec(module_name)  # 2. finders
    if spec is None:
        raise ModuleNotFoundError(f"No module named {module_name!r}")
    module = importlib.util.module_from_spec(spec)  # 3. create
    sys.modules[module_name] = module  # 4. insert before executing
    try:
        spec.loader.exec_module(module)  # 5. execute
    except BaseException:
        del sys.modules[module_name]
        raise
    return module  # 6. the caller binds the name


discounts = simple_import("discounts")
print("FESTIVE =", discounts.FESTIVE)
print("cached on the second call:", simple_import("discounts") is discounts)
try:
    simple_import("no_such_discounts")
except ModuleNotFoundError as error:
    print("ModuleNotFoundError:", error)
"""
discount_files = {"discounts.py": "print('discounts executing')\nFESTIVE = 0.1\n"}
with scratch_project("unit33_simple_import", discount_files) as root:
    print(run_script(root, SIMPLE_IMPORT_SCRIPT))

discounts executing
FESTIVE = 0.1
cached on the second call: True
ModuleNotFoundError: No module named 'no_such_discounts'


### 3.3.3 Import forms

| Form | Binds | Notes |
| --- | --- | --- |
| `import config` | `config` → the module | Attribute access later sees changes |
| `import shop.cart` | `shop` → the **package** | Imports `shop` first; use `shop.cart.Cart` |
| `import shop.cart as cart` | `cart` → the submodule | Alias |
| `from config import DEBUG` | `DEBUG` → the object `config.DEBUG` pointed to **at that moment** | A copy of the binding |
| `from config import DEBUG as debug_flag` | `debug_flag` | Alias |
| `from config import *` | every name in `config.__all__`, or every public name (no leading `_`) | Avoid outside the REPL and `__init__.py` re-exports |

The cell builds a `config` module and a small `shop` package and tries each form in
a child interpreter.

In [153]:
FORM_FILES = {
    "config.py": """
        __all__ = ["DEBUG", "region"]
        DEBUG = False
        _SECRET = "hidden"
        TIMEOUT = 30

        def region():
            return "south"
        """,
    "shop/__init__.py": "",
    "shop/cart.py": "class Cart:\n    pass\n",
}
FORMS_SCRIPT = """
import shop.cart

print("import shop.cart binds", repr(shop.__name__), "and reaches", shop.cart.Cart.__name__)
import shop.cart as cart_module

print("import shop.cart as cart_module ->", repr(cart_module.__name__))
from config import region as region_name

print("from config import region as region_name ->", repr(region_name()))
star_namespace = {}
exec("from config import *", star_namespace)
star_names = sorted(name for name in star_namespace if not name.startswith("__"))
print("from config import * ->", star_names, "(__all__ wins over TIMEOUT)")
"""
with scratch_project("unit33_forms", FORM_FILES) as root:
    print(run_script(root, FORMS_SCRIPT))

import shop.cart binds 'shop' and reaches Cart
import shop.cart as cart_module -> 'shop.cart'
from config import region as region_name -> 'south'
from config import * -> ['DEBUG', 'region'] (__all__ wins over TIMEOUT)


**Copied bindings.** `from config import DEBUG` creates a new name bound to the
same object. Rebinding `config.DEBUG` afterwards changes the module's name, not
yours. Code that must see updates should use `config.DEBUG` (or a function).
Mutating a shared **mutable** object is visible through both names, because there
is only one object.

In [154]:
BINDING_SCRIPT = """
import settings
from settings import DEBUG, FEATURE_FLAGS

settings.DEBUG = True  # rebinding inside the module
settings.FEATURE_FLAGS["dark_mode"] = True  # mutating the shared dict
print("DEBUG =", DEBUG, "| settings.DEBUG =", settings.DEBUG, "(copied binding vs live attribute)")
print("FEATURE_FLAGS =", FEATURE_FLAGS, "(same dict object, so the mutation shows)")
"""
binding_files = {"settings.py": "DEBUG = False\nFEATURE_FLAGS = {'dark_mode': False}\n"}
with scratch_project("unit33_bindings", binding_files) as root:
    print(run_script(root, BINDING_SCRIPT))

DEBUG = False | settings.DEBUG = True (copied binding vs live attribute)
FEATURE_FLAGS = {'dark_mode': True} (same dict object, so the mutation shows)


### 3.3.4 Modules run once

A module's top-level code runs on the **first** import only. That gives two things:

- **Import-time side effects** (prints, network calls, reading files, registering
  handlers) happen once, at a time you may not control. Keep module top levels
  cheap and free of I/O.
- **Module singletons.** A module-level object is shared by every importer: the
  Pythonic singleton (Unit 2.6).

In [155]:
SINGLETON_FILES = {
    "app_state.py": """
        print("app_state: top-level code runs")
        connection_pool = {"opened": 0}

        def open_connection():
            connection_pool["opened"] += 1
            return connection_pool["opened"]
        """,
    "billing.py": "import app_state\n\ndef charge():\n    return app_state.open_connection()\n",
    "shipping.py": "import app_state\n\ndef ship():\n    return app_state.open_connection()\n",
}
SINGLETON_SCRIPT = """
import billing
import shipping

print("billing.charge() =", billing.charge(), "| shipping.ship() =", shipping.ship(), "(one pool)")
print("billing.app_state is shipping.app_state:", billing.app_state is shipping.app_state)
"""
with scratch_project("unit33_singleton", SINGLETON_FILES) as root:
    print(run_script(root, SINGLETON_SCRIPT))

app_state: top-level code runs
billing.charge() = 1 | shipping.ship() = 2 (one pool)
billing.app_state is shipping.app_state: True


**`importlib.reload` caveats.** `reload(module)` re-executes the module's code
**in the same module object**. It does not update:

- names copied with `from module import name`,
- instances of classes defined in the module (they keep the **old** class, so
  `isinstance` against the new class fails),
- other modules that hold references to old functions.

Use it in a REPL, not as a deployment mechanism; restart the process instead.
(The script replaces `rates.py` with a file of a different size on purpose: a
same-size rewrite within the same second would leave the timestamp-based `.pyc`
looking valid, 3.2.13, and `reload` would run the old bytecode.)

In [156]:
RELOAD_SCRIPT = """
import importlib
from pathlib import Path

import rates
from rates import GST

old_rate = rates.Rate()
Path("rates.py").write_text(Path("rates_v2.txt").read_text())
importlib.invalidate_caches()
reloaded = importlib.reload(rates)
print("reloaded is rates:", reloaded is rates, "(same module object)")
print("rates.GST =", rates.GST, "| copied GST =", GST, "(the copied name is stale)")
print("old instance of new class:", isinstance(old_rate, rates.Rate), "(old object, new class)")
"""
reload_files = {
    "rates.py": "class Rate:\n    pass\n\nGST = 0.18\n",
    "rates_v2.txt": "class Rate:\n    pass\n\nGST = 0.12  # festive rate\n",
}
with scratch_project("unit33_reload", reload_files) as root:
    print(run_script(root, RELOAD_SCRIPT))

reloaded is rates: True (same module object)
rates.GST = 0.12 | copied GST = 0.18 (the copied name is stale)
old instance of new class: False (old object, new class)


### 3.3.5 `sys.path`

The path-based finder searches the folders (and zip files) in `sys.path`, in
order. At start-up it is built from:

1. The **script's folder** (`python app.py`) or the **current directory**
   (`python -m pkg`, `python -c`, the REPL). Skipped with `python -P` or
   `PYTHONSAFEPATH=1` (*New in 3.11*).
2. `PYTHONPATH` entries.
3. The standard library: a zip (`python315.zip`), the `python3.15` folder,
   `lib-dynload` (C extensions).
4. `site-packages`, added by the `site` module at start-up, plus anything listed in
   `.pth` files found there.

The cell labels this kernel's entries (reading `sys.path` changes nothing) instead
of printing full paths.

In [157]:
def describe_path_entry(entry: str) -> str:
    """A short label for a sys.path entry."""
    path = Path(entry)
    if entry in {"", os.getcwd()}:
        return "current directory"
    if entry.endswith(".zip"):
        return f"{path.name} (zipped standard library)"
    if path.name == "lib-dynload":
        return "lib-dynload (standard C extensions)"
    if "site-packages" in path.parts:
        return "site-packages (third-party, added by site)"
    if path.name.startswith("python3"):
        return f"{path.name} (standard library)"
    return f".../{path.name}"


for index, entry in enumerate(sys.path):
    print(f"{index}: {describe_path_entry(entry)}")
print(f"{sys.flags.no_site=}  {sys.flags.safe_path=}  {site.ENABLE_USER_SITE=}")

0: python315.zip (zipped standard library)
1: python3.15 (standard library)
2: lib-dynload (standard C extensions)
3: current directory
4: site-packages (third-party, added by site)
sys.flags.no_site=0  sys.flags.safe_path=False  site.ENABLE_USER_SITE=False


**`.pth` files.** When `site` processes a site directory, each `*.pth` file in it
adds one path per line (relative paths are resolved against the site directory),
and lines starting with `import` are **executed**. Editable installs
(`uv pip install -e .`) work this way, and so does some malware; review `.pth`
files in your environment. `site.addsitedir(folder)` processes a folder the same
way. The child interpreter below does that with a scratch site folder.

In [158]:
PTH_FILES = {
    "site_folder/course_extra.pth": """
        ../extra_folder
        import os; os.environ["UNIT33_PTH_RAN"] = "yes"
        """,
    "extra_folder/README.txt": "a folder named in the .pth file\n",
}
PTH_SCRIPT = """
import os
import site
import sys
from pathlib import Path

before = list(sys.path)
site.addsitedir(str(Path("site_folder").resolve()))
print("entries added:", [Path(entry).name for entry in sys.path if entry not in before])
print("import line executed:", os.environ.get("UNIT33_PTH_RAN"))
"""
with scratch_project("unit33_pth", PTH_FILES) as root:
    print(run_script(root, PTH_SCRIPT))

entries added: ['site_folder', 'extra_folder']
import line executed: yes


**`python -m` vs running a file vs `python -P`.** A tiny script reports what
`sys.path[0]` is in each mode:

| Command | `sys.path[0]` | Consequence |
| --- | --- | --- |
| `python tools/where.py` | the script's folder (`tools/`) | `import tools` fails from inside the script |
| `python -m tools.where` | the current directory | package imports work; relative imports work |
| `python -P tools/where.py` | the standard library | a local file named `json.py` can no longer shadow `json` |
| `python -P -m tools.where` | the standard library | `tools` is not found unless installed |

In [159]:
WHERE_FILES = {
    "tools/__init__.py": "",
    "tools/where.py": """
        import os, sys
        first = sys.path[0]
        script_folder = os.path.dirname(os.path.abspath(__file__))
        if first == script_folder:
            print("sys.path[0] = the script's folder")
        elif first in ("", os.getcwd()):
            print("sys.path[0] = the current directory")
        else:
            print("sys.path[0] = " + os.path.basename(first) + " (no script folder or cwd)")
        """,
}
modes = [
    ["tools/where.py"],
    ["-m", "tools.where"],
    ["-P", "tools/where.py"],
    ["-P", "-m", "tools.where"],
]
with scratch_project("unit33_sys_path", WHERE_FILES) as root:
    for arguments in modes:
        output = run_python(*arguments, cwd=root).splitlines()[-1].split("python: ")[-1]
        print(f"python {' '.join(arguments):<22} {output}")

python tools/where.py         sys.path[0] = the script's folder
python -m tools.where         sys.path[0] = the current directory


python -P tools/where.py      sys.path[0] = python315.zip (no script folder or cwd)
python -P -m tools.where      Error while finding module specification for 'tools.where' (ModuleNotFoundError: No module named 'tools')


### 3.3.6 Packages

A **package** is a module with a `__path__`, so it can contain submodules.

- A **regular package** is a folder with `__init__.py`. Importing the package
  runs `__init__.py`; importing `inventory.reports.daily` runs `inventory/__init__.py`,
  then `inventory/reports/__init__.py`, then `daily.py`, parents first.
- `__init__.py` often **re-exports** the public API (`from .stock import
  StockLevel`) so users write `from inventory import StockLevel`.
- A **namespace package** (PEP 420) is a folder **without** `__init__.py`. Folders
  with the same name in several `sys.path` entries merge into one package; there is
  no `__init__` code and no `__file__`. Useful for plugins split across
  distributions.

In [160]:
INVENTORY_FILES = {
    "inventory/__init__.py": """
        print("run inventory/__init__.py")
        from .stock import StockLevel

        __all__ = ["StockLevel"]
        """,
    "inventory/stock.py": 'print("run inventory/stock.py")\nclass StockLevel:\n    pass\n',
    "inventory/reports/__init__.py": 'print("run inventory/reports/__init__.py")\n',
    "inventory/reports/daily.py": """
        print("run inventory/reports/daily.py")
        from ..stock import StockLevel
        """,
}
PACKAGE_SCRIPT = """
from pathlib import Path

import inventory.reports.daily
from inventory import StockLevel

print("re-export is the same class:", StockLevel is inventory.stock.StockLevel)
print("__all__ =", inventory.__all__, "| __path__ =", [Path(p).name for p in inventory.__path__])
"""
with scratch_project("unit33_packages", INVENTORY_FILES) as root:
    print(run_script(root, PACKAGE_SCRIPT))

run inventory/__init__.py
run inventory/stock.py
run inventory/reports/__init__.py
run inventory/reports/daily.py
re-export is the same class: True
__all__ = ['StockLevel'] | __path__ = ['inventory']


**Namespace packages.** Two separate folders, `first/` and `second/`, each contain a
`plugins/` folder without `__init__.py`. With both on `sys.path`, `plugins` is one
package whose `__path__` lists both folders.

In [161]:
NAMESPACE_FILES = {
    "first/plugins/csv_export.py": "FORMAT = 'csv'\n",
    "second/plugins/pdf_export.py": "FORMAT = 'pdf'\n",
}
NAMESPACE_SCRIPT = """
import sys

sys.path[:0] = ["first", "second"]
import plugins
import plugins.csv_export
import plugins.pdf_export

print("__path__ type:", type(plugins.__path__).__name__, "| entries:", len(list(plugins.__path__)))
print("__file__:", getattr(plugins, "__file__", None), "| origin:", plugins.__spec__.origin)
print("formats:", plugins.csv_export.FORMAT, plugins.pdf_export.FORMAT)
"""
with scratch_project("unit33_namespace", NAMESPACE_FILES) as root:
    print(run_script(root, NAMESPACE_SCRIPT))

__path__ type: NamespacePath | entries: 2
__file__: None | origin: None
formats: csv pdf


### 3.3.7 Absolute vs relative imports

| Style | Example | Resolved against |
| --- | --- | --- |
| Absolute | `from inventory.stock import StockLevel` | `sys.path` |
| Relative, same package | `from . import formatting`, `from .stock import StockLevel` | `__package__` |
| Relative, parent package | `from ..stock import StockLevel` | `__package__` minus one level |

PEP 8 prefers absolute imports; relative imports are fine inside a package and
survive renaming the top-level package. A relative import needs a known parent
package, so it **fails when the file is run as a script**: `__package__` is empty
and Python cannot resolve the dots. Run modules inside packages with `python -m`.

In [162]:
RELATIVE_FILES = {
    "inventory/__init__.py": "",
    "inventory/stock.py": "LEVEL = 7\n",
    "inventory/reports/__init__.py": "",
    "inventory/reports/formatting.py": "def bold(text):\n    return f'**{text}**'\n",
    "inventory/reports/daily.py": """
        from . import formatting
        from ..stock import LEVEL

        if __name__ == "__main__":
            print(f"__package__={__package__!r}:", formatting.bold(f"stock {LEVEL}"))
        """,
}
with scratch_project("unit33_relative", RELATIVE_FILES) as root:
    print(
        "python -m inventory.reports.daily ->",
        run_python("-m", "inventory.reports.daily", cwd=root),
    )
    script_output = run_python("inventory/reports/daily.py", cwd=root).splitlines()[-1]
    print("python inventory/reports/daily.py ->", script_output)
    imported = run_script(root, "import inventory.reports.daily as daily\nprint(daily.__package__)")
    print(f"imported from main.py: __package__ = {imported!r}")

python -m inventory.reports.daily -> __package__='inventory.reports': **stock 7**
python inventory/reports/daily.py -> ImportError: attempted relative import with no known parent package
imported from main.py: __package__ = 'inventory.reports'


### 3.3.8 Entry points

- **`if __name__ == "__main__":`** A module run as the program (`python file.py`,
  `python -m pkg.module`) gets `__name__ == "__main__"`; imported, it gets its real
  name. The guard keeps "run me" code from running on import (and is required for
  `multiprocessing` on spawn platforms, Phase 5).
- **`__main__.py`** inside a package makes `python -m package` work. Keep it a
  one-liner that calls a `main()` function, so the logic stays importable and
  testable.
- Installed command-line tools come from `[project.scripts]` in `pyproject.toml`,
  which point at a `module:function` (Phase 4).

`runpy.run_module(name, run_name="__main__")` is what `python -m` uses; the last
child script calls it directly, then imports the same module normally.

In [163]:
GREETER_FILES = {
    "greeter/__init__.py": "",
    "greeter/__main__.py": "from greeter.cli import main\n\nmain()\n",
    "greeter/cli.py": """
        import sys

        def main(argv=None):
            names = (sys.argv[1:] if argv is None else argv) or ["world"]
            print(f"[{__name__}] Namaste, " + " and ".join(names))

        if __name__ == "__main__":
            main()
        """,
}
ENTRY_SCRIPT = """
import runpy
import sys

sys.argv = ["greeter.cli", "Meena"]
runpy.run_module("greeter.cli", run_name="__main__")  # what python -m does
import greeter.cli

print("import greeter.cli: main() was not called on import")
greeter.cli.main(["Asha"])
"""
with scratch_project("unit33_entry_points", GREETER_FILES) as root:
    print("python -m greeter Ravi ->", run_python("-m", "greeter", "Ravi", cwd=root))
    print("python -m greeter.cli  ->", run_python("-m", "greeter.cli", cwd=root))
    print(run_script(root, ENTRY_SCRIPT))

python -m greeter Ravi -> [greeter.cli] Namaste, Ravi


python -m greeter.cli  -> [__main__] Namaste, world


[__main__] Namaste, Meena
import greeter.cli: main() was not called on import
[greeter.cli] Namaste, Asha


### 3.3.9 Circular imports

`invoices` imports `tax`, and `tax` imports `invoices`. Importing `invoices`
inserts a half-built `invoices` into `sys.modules`, starts running it, and its first
line imports `tax`; `tax` then asks for `invoices.Invoice`, which does not exist yet:

```text
import invoices
  invoices: sys.modules["invoices"] = <empty module>; line 1: from tax import tax_for
    tax: line 1: from invoices import Invoice
         invoices is in sys.modules but has no Invoice yet  -> ImportError
```

The modules live in `src/`, which the script puts on `sys.path`. If they sat in
the script's own folder, 3.13+ would print a different hint, *consider renaming
'…/invoices.py' if it has the same name as a library you intended to import*: a
half-imported module from the script's folder is usually a file that shadows a
standard module (a `random.py` next to your script), so CPython suggests that first.

In [164]:
BROKEN_INVOICES = {
    "invoices.py": """
        from tax import tax_for

        class Invoice:
            def __init__(self, amount):
                self.amount = amount

            def total(self):
                return self.amount + tax_for(self)
        """,
    "tax.py": """
        from invoices import Invoice

        RATE = 0.18

        def tax_for(invoice: Invoice) -> float:
            return round(invoice.amount * RATE, 2)
        """,
}
FROM_SRC = "import sys\nsys.path.insert(0, 'src')\n"
with scratch_project("unit33_circular", in_src(BROKEN_INVOICES)) as root:
    print(run_script(root, FROM_SRC + "import invoices").splitlines()[-1])

ImportError: cannot import name 'Invoice' from partially initialized module 'invoices' (most likely due to a circular import) (<project>/src/invoices.py)


Four standard fixes, each checked in child processes with the same scenario
(`Invoice(1000).total()` should be `1180.0`), importing each module first in turn:

| Fix | Change | When to use |
| --- | --- | --- |
| 1. Move shared code to a third module | Both import from a new module; neither imports the other | The best fix when the cycle reveals a misplaced piece |
| 2. Import the **module**, not the name | `import tax` … `tax.tax_for(...)` at call time | The attribute is looked up after both modules finished |
| 3. Import inside the function | `from tax import tax_for` inside `total()` | Quick fix; hides the dependency |
| 4. `if TYPE_CHECKING:` | Import only for type checkers when a module needs the other **only for annotations** | Type-hint-only cycles; 3.14+ lazy annotations need no quotes |

In [165]:
ORDER_SCRIPT = """
import importlib
import sys

sys.path.insert(0, "src")
importlib.import_module(sys.argv[1])  # import this module first
import invoices

print(invoices.Invoice(1000).total())
"""


def check_invoice_fix(files: dict[str, str]) -> str:
    """Run the scenario in child processes, importing each module first in turn."""
    totals = []
    for first_import in ("invoices", "tax"):
        with scratch_project("unit33_circular", in_src(files)) as root:
            totals.append(run_script(root, ORDER_SCRIPT, first_import).splitlines()[-1])
    return f"total={totals[0]} (both import orders work: {totals[0] == totals[1] == '1180.0'})"


INVOICE_CLASS = textwrap.dedent(
    """
    class Invoice:
        def __init__(self, amount):
            self.amount = amount

        def total(self):
            return self.amount + tax_for(self)
    """
)
BROKEN_TAX = textwrap.dedent(BROKEN_INVOICES["tax.py"])
invoice_fixes = {
    "1. third module": {
        "rates.py": """
            RATE = 0.18

            def tax_for(invoice):
                return round(invoice.amount * RATE, 2)
            """,
        "invoices.py": "from rates import tax_for\n" + INVOICE_CLASS,
        "tax.py": "from invoices import Invoice\nfrom rates import RATE, tax_for\n",
    },
    "2. module import": {
        "invoices.py": "import tax\n" + INVOICE_CLASS.replace("tax_for(self)", "tax.tax_for(self)"),
        "tax.py": BROKEN_TAX.replace("from invoices import Invoice", "import invoices").replace(
            "invoice: Invoice", "invoice: invoices.Invoice"
        ),
    },
    "3. local import": {
        "invoices.py": INVOICE_CLASS.replace(
            "return self.amount", "from tax import tax_for\n\n        return self.amount"
        ),
        "tax.py": BROKEN_TAX,
    },
    "4. TYPE_CHECKING": {
        "invoices.py": BROKEN_INVOICES["invoices.py"],
        "tax.py": BROKEN_TAX.replace(
            "from invoices import Invoice",
            "from typing import TYPE_CHECKING\n\n"
            "if TYPE_CHECKING:\n    from invoices import Invoice",
        ),
    },
}
for label, files in invoice_fixes.items():
    print(f"{label:<18} {check_invoice_fix(files)}")

1. third module    total=1180.0 (both import orders work: True)


2. module import   total=1180.0 (both import orders work: True)


3. local import    total=1180.0 (both import orders work: True)


4. TYPE_CHECKING   total=1180.0 (both import orders work: True)


> **Note:** fix 4 works because `tax` uses `Invoice` only in an annotation, and since
> 3.14 annotations are evaluated lazily (PEP 649), so the unquoted `Invoice` is never
> looked up at import time. `typing.TYPE_CHECKING` is `False` at run time and `True`
> for mypy / pyright.

### 3.3.10 `importlib`

| API | Use |
| --- | --- |
| `importlib.import_module(name, package=None)` | Import by a string name (config-driven backends, plugins); relative names need `package` |
| `importlib.resources.files(package)` | Read data files shipped **inside** a package, from a folder, a zip or a wheel |
| `importlib.metadata` | Installed distributions: `version`, `metadata`, `requires`, `files`, `entry_points` |
| `pkgutil.iter_modules(path)` | List the modules in a package without importing them |
| Entry points | Plugins advertised by **installed distributions** in their metadata (`[project.entry-points."group"]`) |

#### `importlib.import_module`

`json` and its submodules are already loaded in this kernel, so these calls only
look them up in `sys.modules`.

In [166]:
backend_name = "json.decoder"  # e.g. read from a settings file
decoder_module = importlib.import_module(backend_name)
encoder_module = importlib.import_module(".encoder", package="json")
print(f"{decoder_module.__name__=}  {encoder_module.__name__=}")
print(raised_message(lambda: importlib.import_module(".encoder")))

decoder_module.__name__='json.decoder'  encoder_module.__name__='json.encoder'
TypeError: the 'package' argument is required to perform a relative import for '.encoder'


#### `importlib.resources`

`files(package)` returns a `Traversable` (like a read-only `pathlib.Path`) for the
package's folder, wherever it really lives. `as_file()` gives a real file path when
a library insists on one (extracting from a zip if needed). Reading the
already-imported `json` package is safe in the kernel; the `brand` package with a
data file runs in a child.

In [167]:
json_files = sorted(item.name for item in importlib.resources.files("json").iterdir())
print(f"json package files (read in the kernel): {json_files[:4]}")
BRAND_FILES = {"brand/__init__.py": "", "brand/assets/tagline.txt": "Fresh groceries, fast.\n"}
RESOURCES_SCRIPT = """
import importlib.resources

assets = importlib.resources.files("brand") / "assets"
print("type:", type(assets).__name__, "| contents:", [item.name for item in assets.iterdir()])
print("tagline:", (assets / "tagline.txt").read_text().strip())
with importlib.resources.as_file(assets / "tagline.txt") as real_path:
    print("as_file gives a real path:", real_path.exists())
"""
with scratch_project("unit33_resources", BRAND_FILES) as root:
    print(run_script(root, RESOURCES_SCRIPT))

json package files (read in the kernel): ['__init__.py', '__main__.py', '__pycache__', 'decoder.py']


type: PosixPath | contents: ['tagline.txt']
tagline: Fresh groceries, fast.
as_file gives a real path: True


#### `importlib.metadata`

Information about **installed distributions** (what `uv add` installed), read from
their `.dist-info` folders, without importing them.

In [168]:
nbformat_meta = importlib.metadata.metadata("nbformat")
print(f"{importlib.metadata.version('nbformat')=}")
print(f"Summary: {nbformat_meta['Summary']}")
print(f"first requirements: {sorted(importlib.metadata.requires('nbformat'))[:3]}")
print(f"files installed: {len(importlib.metadata.files('nbformat'))}")
providers = importlib.metadata.packages_distributions().get("nbformat")
print(f"distributions providing the nbformat package: {providers}")
print(raised_error_name(lambda: importlib.metadata.version("not-a-real-distribution")))

importlib.metadata.version('nbformat')='5.11.1'
Summary: The Jupyter Notebook format
first requirements: ['fastjsonschema>=2.15', "intersphinx-registry; extra == 'docs'", 'jsonschema>=2.6']
files installed: 57
distributions providing the nbformat package: ['nbformat']
PackageNotFoundError


#### `pkgutil.iter_modules`

In [169]:
json_parts = sorted((info.name, info.ispkg) for info in pkgutil.iter_modules(json.__path__))
print(f"{json_parts=}")
print(f"{[info.name for info in pkgutil.iter_modules(importlib.__path__)][:6]=}")

json_parts=[('__main__', False), ('decoder', False), ('encoder', False), ('scanner', False), ('tool', False)]


[info.name for info in pkgutil.iter_modules(importlib.__path__)][:6]=['_abc', '_bootstrap', '_bootstrap_external', 'abc', 'machinery', 'metadata']


#### Entry-point plugins

A distribution can advertise objects under a named **group** in its metadata;
the host application discovers and loads them without knowing their modules in
advance. `console_scripts` is the group behind installed commands; nbconvert finds
its exporters through the `nbconvert.exporters` group. Listing entry points only
reads metadata; **loading** one imports its module, so that part runs in a child.

In [170]:
console_scripts = importlib.metadata.entry_points(group="console_scripts")
jupyter_commands = sorted(ep.name for ep in console_scripts if ep.name.startswith("jupyter"))
print(f"jupyter commands: {jupyter_commands[:5]}")
exporters = importlib.metadata.entry_points(group="nbconvert.exporters")
if exporters:
    print(f"nbconvert exporters advertised: {sorted(ep.name for ep in exporters)[:6]}")
    LOAD_SCRIPT = """
    from importlib.metadata import entry_points

    markdown = next(ep for ep in entry_points(group="nbconvert.exporters") if ep.name == "markdown")
    print(markdown.name, "->", markdown.value, "| loaded:", markdown.load().__name__)
    """
    print(run_python("-c", textwrap.dedent(LOAD_SCRIPT)))
else:
    print("Skipped: no nbconvert.exporters entry points installed")

jupyter commands: ['jupyter', 'jupyter-builder', 'jupyter-dejavu', 'jupyter-events', 'jupyter-execute']
nbconvert exporters advertised: ['asciidoc', 'custom', 'html', 'latex', 'markdown', 'notebook']


markdown -> nbconvert.exporters:MarkdownExporter | loaded: MarkdownExporter


### 3.3.11 Import hooks

Two extension points:

- **`sys.meta_path`**: finders asked for **every** import (3.3.2). A custom one
  can serve modules from anywhere: a database, the network, an encrypted bundle.
- **`sys.path_hooks`**: factories that turn a `sys.path` entry into a finder for
  that entry (this is how zip files on `sys.path` work); results are cached in
  `sys.path_importer_cache`.

A finder implements `find_spec(fullname, path, target=None)` and returns a
`ModuleSpec` or `None`; the spec names a loader that implements
`create_module(spec)` (return `None` for a default module) and
`exec_module(module)`. The demo's `json_importer.py` serves `<name>.json` files
from a folder as modules whose attributes are the JSON keys; a child script
installs it and imports `store_settings` from `settings/store_settings.json`.

In [171]:
HOOK_FILES = {
    "json_importer.py": """
        import importlib.abc
        import importlib.util
        import json
        from pathlib import Path


        class JsonModuleImporter(importlib.abc.MetaPathFinder, importlib.abc.Loader):
            # Import `name` from name.json in a folder: the JSON keys become attributes.

            def __init__(self, folder):
                self.folder = Path(folder)

            def find_spec(self, fullname, path, target=None):
                candidate = self.folder / f"{fullname}.json"
                if "." in fullname or not candidate.exists():
                    return None  # not ours: the next finder tries
                return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))

            def create_module(self, spec):
                return None  # use the default module object

            def exec_module(self, module):
                module.__dict__.update(json.loads(Path(module.__spec__.origin).read_text()))
        """,
    "settings/store_settings.json": '{"currency": "INR", "free_shipping_above": 499}',
}
HOOK_SCRIPT = """
import sys

from json_importer import JsonModuleImporter

sys.meta_path.insert(0, JsonModuleImporter("settings"))
import store_settings

print("currency:", store_settings.currency, "| free above:", store_settings.free_shipping_above)
print("loader:", type(store_settings.__loader__).__name__)
print("origin:", store_settings.__spec__.origin)
"""
with scratch_project("unit33_import_hook", HOOK_FILES) as root:
    print(run_script(root, HOOK_SCRIPT))
hook_names = [getattr(hook, "__name__", type(hook).__name__) for hook in sys.path_hooks]
print(f"this kernel's sys.path_hooks: {hook_names}")

currency: INR | free above: 499
loader: JsonModuleImporter
origin: settings/store_settings.json
this kernel's sys.path_hooks: ['zipimporter', 'path_hook_for_FileFinder']


### 3.3.12 Lazy imports (3.15)

Command-line tools often import a dozen heavy modules at start-up and use one.
*New in 3.15* (PEP 810): `lazy import x` and `lazy from x import y` bind a
**proxy** instead of importing. The real import happens the first time the name is
used; the module code then runs exactly as before.

| Rule | Detail |
| --- | --- |
| Where allowed | Module level only; not inside functions, classes, `try`/`except` or `with` blocks |
| Not allowed | `lazy from x import *` |
| Errors | A missing module raises `ModuleNotFoundError` at **first use**, not at the import line |
| Global switch | `sys.set_lazy_imports("all")`, `python -X lazy_imports=all`, `PYTHON_LAZY_IMPORTS=all` make ordinary top-level imports lazy too; a filter (`sys.set_lazy_imports_filter`) can opt modules out |
| Inspecting | `sys.lazy_modules` lists modules imported lazily but not yet loaded |

Lazy syntax is 3.15-only, so it never appears in a notebook cell's own code (an
older kernel could not even parse the cell): it lives in **separate files** run by a
child interpreter, or in strings passed to `compile()`, all behind a
`sys.version_info >= (3, 15)` check. First, the proxy and its first use, on a module
that prints when it really runs:

In [172]:
LAZY_FILES = {
    "heavy_report.py": """
        print("   heavy_report: module code runs now")

        def summary():
            return "ok"
        """,
    "lazy_demo.py": """
        import sys

        lazy import heavy_report

        kind = type(globals()["heavy_report"]).__name__
        print("1. after `lazy import`:", "heavy_report" in sys.modules, kind)
        print("2. first use returns:", heavy_report.summary())
        kind = type(globals()["heavy_report"]).__name__
        print("3. after first use:", "heavy_report" in sys.modules, kind)
        """,
    "missing_demo.py": """
        lazy import no_such_reporting_lib

        print("the lazy import line passed")
        try:
            no_such_reporting_lib.render()
        except ModuleNotFoundError as error:
            print("first use ->", type(error).__name__ + ":", error)
        """,
}
if sys.version_info >= (3, 15):
    with scratch_project("unit33_lazy", LAZY_FILES) as root:
        print(run_python("lazy_demo.py", cwd=root))
        print(run_python("missing_demo.py", cwd=root))
else:
    print("Skipped: lazy imports need Python 3.15+")

1. after `lazy import`: False lazy_import
   heavy_report: module code runs now
2. first use returns: ok
3. after first use: True module
the lazy import line passed
first use -> ModuleNotFoundError: No module named 'no_such_reporting_lib'


**Restrictions.** Where the keyword is refused, checked with `compile()` on strings
(nothing is imported), plus this kernel's global mode.

In [173]:
if sys.version_info >= (3, 15):
    print(f"in a function: {compile_error('def load():\n    lazy import json\n')}")
    in_try = "try:\n    lazy import json\nexcept ImportError:\n    pass\n"
    print(f"in try:        {compile_error(in_try)}")
    print(f"star import:   {compile_error('lazy from json import *\n')}")
    print(f"module level:  {compile_error('lazy import json\n')}")
    pending_count = len(sys.lazy_modules)
    print(
        f"{sys.get_lazy_imports()=}  ({pending_count} stdlib modules still pending in this kernel)"
    )
else:
    print("Skipped: lazy imports need Python 3.15+")

in a function: SyntaxError: lazy import not allowed inside functions
in try:        SyntaxError: lazy import not allowed inside try/except blocks
star import:   SyntaxError: lazy from ... import * is not allowed
module level:  compiles
sys.get_lazy_imports()='normal'  (10 stdlib modules still pending in this kernel)


**The global switch.** With `-X lazy_imports=all`, a plain `import json` at the top
of a program is lazy too.

In [174]:
switch_probe = (
    "import sys, json\n"
    "print('json loaded before use:', 'json' in sys.modules)\n"
    "json.dumps(1)\n"
    "print('after use:', 'json' in sys.modules)"
)
if sys.version_info >= (3, 15):
    print("default:              ", run_python("-c", switch_probe).replace("\n", " | "))
    print(
        "-X lazy_imports=all:  ",
        run_python("-X", "lazy_imports=all", "-c", switch_probe).replace("\n", " | "),
    )
else:
    print("Skipped: lazy imports need Python 3.15+")

default:               json loaded before use: True | after use: True
-X lazy_imports=all:   json loaded before use: False | after use: True


**`importlib.util.LazyLoader`** (3.5+) gives the same effect for one module on any
version: the module object is created at once, and its code runs on the first
attribute access. It registers the module in `sys.modules`, so it runs in a child.

In [175]:
LAZY_LOADER_SCRIPT = """
import importlib.util
import sys


def lazy_load(module_name):
    # Import module_name with LazyLoader: its code runs on first attribute access.
    spec = importlib.util.find_spec(module_name)
    spec.loader = importlib.util.LazyLoader(spec.loader)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


report_module = lazy_load("heavy_report")
print("module object exists:", type(report_module).__name__)
print("first attribute access ->", report_module.summary())
"""
with scratch_project(
    "unit33_lazy_loader", {"heavy_report.py": LAZY_FILES["heavy_report.py"]}
) as root:
    print(run_script(root, LAZY_LOADER_SCRIPT))

module object exists: _LazyModule
   heavy_report: module code runs now
first attribute access -> ok


**Measuring with `python -X importtime`.** The interpreter prints one line per
module imported, to stderr: self time and cumulative time in microseconds,
indented by nesting depth. The cell compares `import asyncio` with
`lazy import asyncio`; the counts are lines of the report (times vary by machine).

In [176]:
eager_report = run_python("-X", "importtime", "-c", "import asyncio")
print("\n".join(eager_report.splitlines()[:1] + eager_report.splitlines()[-3:]))
eager_lines = [line for line in eager_report.splitlines() if line.startswith("import time:")]
print(f"eager: {len(eager_lines) - 1} modules reported")
if sys.version_info >= (3, 15):
    lazy_report = run_python("-X", "importtime", "-c", "lazy import asyncio")
    lazy_lines = [line for line in lazy_report.splitlines() if line.startswith("import time:")]
    asyncio_listed = "asyncio" in lazy_report
    print(f"lazy, unused: {len(lazy_lines) - 1} modules reported; asyncio listed: {asyncio_listed}")

import time: self [us] | cumulative | imported package
import time:       555 |        555 |     asyncio.selector_events
import time:       584 |       1416 |   asyncio.unix_events
import time:       394 |      49415 | asyncio
eager: 146 modules reported


lazy, unused: 25 modules reported; asyncio listed: False


### 3.3.13 JS bridge

| Topic | Node.js / browsers | Python |
| --- | --- | --- |
| Module cache | `require.cache[resolvedPath]`; ESM has an internal module map | `sys.modules[name]`, keyed by **name**, not path |
| Forcing a re-run | `delete require.cache[path]` | `del sys.modules[name]` then import, or `importlib.reload` |
| Named imports | ESM bindings are **live**: the importer sees later reassignments | `from m import x` copies the binding (3.3.3) |
| Resolution | `node_modules` searched in the current folder, then each parent; per-package versions | `sys.path` searched in order; one installed version per environment |
| Package entry | `package.json` `"main"` / `"exports"` | `__init__.py` (and `__all__`), `__main__.py` for `-m` |
| Run as program | `require.main === module` / `import.meta.main` | `__name__ == "__main__"` |
| Cycles | CommonJS returns the partial `exports`; ESM can hit TDZ errors | Partially initialized module → `ImportError` / `AttributeError` |
| Dynamic import | `await import("./m.js")` | `importlib.import_module("m")` (synchronous) |
| Lazy loading | dynamic `import()` at the point of use | `lazy import` (3.15), `LazyLoader`, imports inside functions |

Like deleting from `require.cache`, removing a module from `sys.modules` makes the
next import run its code again and create a **new** module object; old
references keep the old one.

In [177]:
RERUN_SCRIPT = """
import sys

import counter_module

first_object = counter_module
del sys.modules["counter_module"]
import counter_module

print("first_object is counter_module:", first_object is counter_module, "(a fresh module object)")
"""
with scratch_project(
    "unit33_rerun", {"counter_module.py": "print('counter_module runs')\n"}
) as root:
    print(run_script(root, RERUN_SCRIPT))

counter_module runs
counter_module runs
first_object is counter_module: False (a fresh module object)


**Predict the output.**

```python
# settings.py contains:  DEBUG = False
import settings
from settings import DEBUG

settings.DEBUG = True
print(DEBUG, settings.DEBUG)
```

<details><summary>Answer</summary>

```text
False True
```

`from settings import DEBUG` bound a new name to the object `False`. Assigning
`settings.DEBUG = True` rebinds the name **inside the module**; your `DEBUG` still
points at `False`. In JavaScript, an ESM named import would see `true` (live
binding).

</details>

Run the next cell to check your prediction.

In [178]:
PREDICT_SCRIPT = """
import settings
from settings import DEBUG

settings.DEBUG = True
print(DEBUG, settings.DEBUG)
"""
with scratch_project("unit33_predict", {"settings.py": "DEBUG = False\n"}) as root:
    print(run_script(root, PREDICT_SCRIPT))

False True


#### Proof that the demos stayed out of this kernel

Every module above was imported by a child interpreter. The kernel's `sys.path` is
the same list as when the setup cell ran, none of the demo modules are in its
`sys.modules`, and every scratch folder has been deleted.

In [179]:
DEMO_MODULE_NAMES = {
    "checkout_rules", "checkout_helpers", "discounts", "fake_payments", "config", "shop",
    "settings", "app_state", "billing", "shipping", "rates", "tools", "inventory", "plugins",
    "greeter", "invoices", "tax", "brand", "json_importer", "store_settings", "heavy_report",
    "lazy_demo", "missing_demo", "counter_module",
}  # fmt: skip
print(f"kernel sys.path unchanged since the setup cell: {sys.path == KERNEL_PATH_AT_START}")
print(f"demo modules in the kernel's sys.modules: {sorted(DEMO_MODULE_NAMES & set(sys.modules))}")
leftover_folders = sorted(path.name for path in scratch_dir.glob("unit33_*"))
print(f"demo folders left in notebooks/_scratch: {leftover_folders}")

kernel sys.path unchanged since the setup cell: True
demo modules in the kernel's sys.modules: []
demo folders left in notebooks/_scratch: []


### Lab 3.3

Six exercises. For each, complete the stub, then run the check cell. Until then the
checks print FAIL; that is expected.

- Exercises 1, 2 and 6 hand over files or source code, so their checks run your
  packages in **child processes** from `notebooks/_scratch/`, like the demos.
- Exercises 3–5 test functions you write **in this kernel** (`load_plugins`,
  `DictImporter`, the `importlib.resources` readers), so those checks must import
  into the kernel. `temporary_modules` writes the packages under
  `notebooks/_scratch/` and, on exit, removes the folder from `sys.path` and forgets
  every module imported from it, so you can re-run checks freely.

#### Exercise 1 — build a `shop/` package and test its imports

Return a dict mapping relative paths to file contents for this package:

| File | Contents |
| --- | --- |
| `shop/models.py` | a frozen dataclass `Product(name: str, price: Decimal)` |
| `shop/cart.py` | `Cart` with `add(product, quantity=1)` and `total() -> Decimal`; imports `Product` with a **relative** import |
| `shop/payments/__init__.py` | a subpackage with `charge(amount: Decimal) -> str` returning `f"charged INR {amount:.2f}"` |
| `shop/__init__.py` | re-exports `Cart` and `Product`, sets `__all__ = ["Cart", "Product"]` and `__version__ = "1.0"` |
| `shop/__main__.py` | prints `f"shop {__version__}"`, so `python -m shop` prints `shop 1.0` |

In [180]:
def shop_package_files() -> dict[str, str]:
    """Relative path -> source for the shop package."""
    raise NotImplementedError("Lab 3.3: write the shop package")

Run the checks. Each one writes your package to `notebooks/_scratch/` and inspects it in a child process.

In [181]:
SHOP_SCRIPT = """
import json
from decimal import Decimal

import shop
import shop.cart
import shop.payments

namespace = {}
exec("from shop import *", namespace)
cart = shop.Cart()
cart.add(shop.Product("tea", Decimal("120.50")), quantity=2)
cart.add(shop.Product("rice", Decimal("80")))
report = {
    "all": sorted(shop.__all__),
    "version": shop.__version__,
    "star": sorted(name for name in namespace if not name.startswith("__")),
    "same_class": shop.Cart is shop.cart.Cart,
    "total": str(cart.total()),
    "charge": shop.payments.charge(Decimal("321")),
}
print(json.dumps(report))
"""


def shop_report() -> dict[str, object]:
    """Write the learner's package and inspect it in a child process."""
    with scratch_project("unit33_lab_shop", shop_package_files()) as root:
        return child_json(run_script(root, SHOP_SCRIPT))


def shop_run_as_module() -> str:
    """Output of `python -m shop` in the package folder."""
    with scratch_project("unit33_lab_shop", shop_package_files()) as root:
        return run_python("-m", "shop", cwd=root)


run_checks(
    [
        ("__all__", lambda: shop_report()["all"], ["Cart", "Product"]),
        ("__version__", lambda: shop_report()["version"], "1.0"),
        ("star import", lambda: shop_report()["star"], ["Cart", "Product"]),
        ("re-export is the same class", lambda: shop_report()["same_class"], True),
        ("cart total", lambda: shop_report()["total"], "321.00"),
        ("subpackage", lambda: shop_report()["charge"], "charged INR 321.00"),
        (
            "relative import in cart.py",
            lambda: "from .models import" in shop_package_files()["shop/cart.py"],
            True,
        ),
        ("python -m shop", shop_run_as_module, "shop 1.0"),
    ]
)

FAIL  __all__: raised NotImplementedError: Lab 3.3: write the shop package
FAIL  __version__: raised NotImplementedError: Lab 3.3: write the shop package
FAIL  star import: raised NotImplementedError: Lab 3.3: write the shop package
FAIL  re-export is the same class: raised NotImplementedError: Lab 3.3: write the shop package
FAIL  cart total: raised NotImplementedError: Lab 3.3: write the shop package
FAIL  subpackage: raised NotImplementedError: Lab 3.3: write the shop package
FAIL  relative import in cart.py: raised NotImplementedError: Lab 3.3: write the shop package
FAIL  python -m shop: raised NotImplementedError: Lab 3.3: write the shop package
0/8 checks passed


#### Exercise 2 — fix a circular import three ways

`orders` and `customers` need each other **at run time**: `Customer.place_order`
creates an `Order`, and `Order.__init__` checks `isinstance(customer, Customer)`.
The given version fails on import (the demo below shows the error). Return three
fixed file sets (each a dict like `BROKEN_ORDERS`), keeping the behaviour:

1. `fix_with_module_imports()`: both files use `import orders` / `import customers`
   at the top and look up classes as attributes at call time.
2. `fix_with_local_import()`: one top-level import stays; the other moves inside the
   function that needs it.
3. `fix_with_third_module()`: move both classes into a new module (for example
   `models.py`); `orders.py` and `customers.py` re-export them.

The scenario: `customers.Customer("Asha").place_order(7).describe()` returns
`"order 7 for Asha"`, and `orders.Order(1, "nobody")` raises `TypeError`.

In [182]:
BROKEN_ORDERS = {
    "orders.py": """
        from customers import Customer

        class Order:
            def __init__(self, order_id, customer):
                if not isinstance(customer, Customer):
                    raise TypeError("customer must be a Customer")
                self.order_id = order_id
                self.customer = customer

            def describe(self):
                return f"order {self.order_id} for {self.customer.name}"
        """,
    "customers.py": """
        from orders import Order

        class Customer:
            def __init__(self, name):
                self.name = name
                self.orders = []

            def place_order(self, order_id):
                order = Order(order_id, self)
                self.orders.append(order)
                return order
        """,
}
FROM_SRC = "import sys\nsys.path.insert(0, 'src')\n"
with scratch_project("unit33_lab_orders", in_src(BROKEN_ORDERS)) as root:
    print(f"given version: {run_script(root, FROM_SRC + 'import customers').splitlines()[-1]}")

given version: ImportError: cannot import name 'Customer' from partially initialized module 'customers' (most likely due to a circular import) (<project>/src/customers.py)


The stubs:

In [183]:
def fix_with_module_imports() -> dict[str, str]:
    """Fix 1: import modules, not names."""
    raise NotImplementedError("Lab 3.3: fix with module imports")


def fix_with_local_import() -> dict[str, str]:
    """Fix 2: move one import into the function that needs it."""
    raise NotImplementedError("Lab 3.3: fix with a local import")


def fix_with_third_module() -> dict[str, str]:
    """Fix 3: move both classes into a third module."""
    raise NotImplementedError("Lab 3.3: fix with a third module")

Run the checks. Each fix runs in child processes, imported in both orders.

In [184]:
ORDERS_SCRIPT = """
import importlib
import json
import sys

sys.path.insert(0, "src")
importlib.import_module(sys.argv[1])  # this module is imported first
import customers
import orders

outcome = [customers.Customer("Asha").place_order(7).describe()]
try:
    orders.Order(1, "nobody")
    outcome.append("no error")
except Exception as error:
    outcome.append(type(error).__name__)
print(json.dumps(outcome))
"""


def orders_scenario(files: dict[str, str]) -> list[str]:
    """Run the scenario in child processes: customers imported first, then orders first."""
    outcomes: list[str] = []
    for first_import in ("customers", "orders"):
        with scratch_project("unit33_lab_orders", in_src(files)) as root:
            outcomes.extend(child_json(run_script(root, ORDERS_SCRIPT, first_import)))
    return outcomes


def top_level_imports(source: str) -> list[str]:
    """Module names imported with plain `import` at the top level of source."""
    return [
        alias.name
        for node in ast.parse(textwrap.dedent(source)).body
        if isinstance(node, ast.Import)
        for alias in node.names
    ]


def has_function_level_import(files: dict[str, str]) -> bool:
    """Is there an import statement inside some function?"""
    for source in files.values():
        for node in ast.walk(ast.parse(textwrap.dedent(source))):
            if isinstance(node, ast.FunctionDef) and any(
                isinstance(inner, ast.Import | ast.ImportFrom) for inner in ast.walk(node)
            ):
                return True
    return False


EXPECTED_SCENARIO = ["order 7 for Asha", "TypeError", "order 7 for Asha", "TypeError"]
run_checks(
    [
        (
            "module imports: scenario",
            lambda: orders_scenario(fix_with_module_imports()),
            EXPECTED_SCENARIO,
        ),
        (
            "module imports: uses `import`",
            lambda: (
                "customers" in top_level_imports(fix_with_module_imports()["orders.py"]),
                "orders" in top_level_imports(fix_with_module_imports()["customers.py"]),
            ),
            (True, True),
        ),
        (
            "local import: scenario",
            lambda: orders_scenario(fix_with_local_import()),
            EXPECTED_SCENARIO,
        ),
        (
            "local import: inside a function",
            lambda: has_function_level_import(fix_with_local_import()),
            True,
        ),
        (
            "third module: scenario",
            lambda: orders_scenario(fix_with_third_module()),
            EXPECTED_SCENARIO,
        ),
        ("third module: three files", lambda: len(fix_with_third_module()) >= 3, True),
    ]
)

FAIL  module imports: scenario: raised NotImplementedError: Lab 3.3: fix with module imports
FAIL  module imports: uses `import`: raised NotImplementedError: Lab 3.3: fix with module imports
FAIL  local import: scenario: raised NotImplementedError: Lab 3.3: fix with a local import
FAIL  local import: inside a function: raised NotImplementedError: Lab 3.3: fix with a local import
FAIL  third module: scenario: raised NotImplementedError: Lab 3.3: fix with a third module
FAIL  third module: three files: raised NotImplementedError: Lab 3.3: fix with a third module
0/6 checks passed


#### Exercise 3 — a plugin loader with `pkgutil`

`load_plugins(package_name)` imports the package, then every **module** (not
subpackage) inside it found with `pkgutil.iter_modules(package.__path__)`:

- skip modules whose name starts with `_`,
- a plugin is a module with a string `PLUGIN_NAME` and a callable `run`; skip
  modules missing either,
- two plugins with the same `PLUGIN_NAME` raise
  `ValueError(f"duplicate plugin name: {name}")`,
- return `{PLUGIN_NAME: run}` with keys sorted.

In [185]:
def load_plugins(package_name: str) -> dict[str, Callable[..., object]]:
    """Discover plugin modules in a package."""
    raise NotImplementedError("Lab 3.3: implement load_plugins")

Run the checks.

In [186]:
EXPORTER_FILES = {
    "exporters/__init__.py": "",
    "exporters/csv_exporter.py": """
        PLUGIN_NAME = "csv"

        def run(rows):
            return "\\n".join(",".join(str(value) for value in row.values()) for row in rows)
        """,
    "exporters/json_exporter.py": """
        import json

        PLUGIN_NAME = "json"

        def run(rows):
            return json.dumps(rows)
        """,
    "exporters/_shared.py": "PLUGIN_NAME = 'shared'\n\ndef run(rows):\n    return ''\n",
    "exporters/draft_exporter.py": "PLUGIN_NAME = 'draft'  # no run() yet\n",
    "exporters/legacy/__init__.py": "PLUGIN_NAME = 'legacy'\n\ndef run(rows):\n    return ''\n",
}
SAMPLE_ROWS = [{"sku": "TEA-01", "qty": 2}, {"sku": "RICE-5", "qty": 1}]


def plugins_from(files: dict[str, str]) -> Callable[[], object]:
    """A thunk that loads the exporters plugins from files."""

    def load() -> object:
        with temporary_modules(files):
            plugins = load_plugins("exporters")
            return list(plugins), plugins["csv"](SAMPLE_ROWS), plugins["json"](SAMPLE_ROWS[:1])

    return load


duplicate_files = {
    **EXPORTER_FILES,
    "exporters/csv_again.py": "PLUGIN_NAME = 'csv'\n\ndef run(rows):\n    return ''\n",
}
run_checks(
    [
        (
            "plugins, sorted, run",
            plugins_from(EXPORTER_FILES),
            (["csv", "json"], "TEA-01,2\nRICE-5,1", '[{"sku": "TEA-01", "qty": 2}]'),
        ),
        (
            "duplicate name",
            lambda: raised_message(plugins_from(duplicate_files)),
            "ValueError: duplicate plugin name: csv",
        ),
    ]
)

FAIL  plugins, sorted, run: raised NotImplementedError: Lab 3.3: implement load_plugins
FAIL  duplicate name: expected 'ValueError: duplicate plugin name: csv', got 'NotImplementedError: Lab 3.3: implement load_plugins'
0/2 checks passed


#### Exercise 4 — an in-memory `DictImporter`

`DictImporter(sources)` serves modules from a dict mapping **dotted module names**
to source code. It is both the finder and the loader:

- `find_spec(fullname, path=None, target=None)`: `None` for names not in
  `sources`; otherwise `importlib.util.spec_from_loader(fullname, self,
  origin="<dict>", is_package=...)`, where a name is a package if some other key
  starts with `fullname + "."`.
- `create_module(spec)` returns `None`; `exec_module(module)` compiles the source
  (filename `f"<dict:{name}>"`) and executes it in `module.__dict__`.

Relative imports inside the served packages must work.

In [187]:
class DictImporter(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    """Serve modules from a dict of dotted name -> source code."""

    def __init__(self, sources: dict[str, str]) -> None:
        raise NotImplementedError("Lab 3.3: implement DictImporter")

Run the checks. `installed(importer)` adds it to `sys.meta_path` and cleans up afterwards.

In [188]:
IN_MEMORY_SOURCES = {
    "greetings": "def hello(name):\n    return f'Namaste, {name}'\n",
    "toolbox": "from .strings import shout\n",
    "toolbox.strings": "def shout(text):\n    return text.upper() + '!'\n",
    "run_counter": "import builtins\nbuiltins.__dict__.setdefault('_run_counter_hits', []).append(1)",  # noqa: E501
}


@contextmanager
def installed(importer: object) -> Iterator[object]:
    """Put importer first on sys.meta_path; remove it and its modules afterwards."""
    sys.meta_path.insert(0, importer)
    try:
        yield importer
    finally:
        sys.meta_path.remove(importer)
        for name in [
            name
            for name, module in sys.modules.items()
            if getattr(module, "__loader__", None) is importer
        ]:
            del sys.modules[name]


def with_importer(action: Callable[[object], object]) -> Callable[[], object]:
    def run() -> object:
        with installed(DictImporter(IN_MEMORY_SOURCES)) as importer:
            return action(importer)

    return run


def imports_once(importer: object) -> int:
    import builtins

    builtins.__dict__.pop("_run_counter_hits", None)
    importlib.import_module("run_counter")
    importlib.import_module("run_counter")
    return len(builtins.__dict__.pop("_run_counter_hits", []))


run_checks(
    [
        (
            "plain module",
            with_importer(lambda imp: importlib.import_module("greetings").hello("Asha")),
            "Namaste, Asha",
        ),
        (
            "spec origin and loader",
            with_importer(
                lambda imp: (
                    importlib.import_module("greetings").__spec__.origin,
                    importlib.import_module("greetings").__loader__ is imp,
                )
            ),
            ("<dict>", True),
        ),
        (
            "package + relative import",
            with_importer(lambda imp: importlib.import_module("toolbox").shout("hi")),
            "HI!",
        ),
        (
            "package has __path__",
            with_importer(lambda imp: hasattr(importlib.import_module("toolbox"), "__path__")),
            True,
        ),
        ("unknown name -> None", with_importer(lambda imp: imp.find_spec("json", None)), None),
        (
            "missing module",
            with_importer(
                lambda imp: raised_error_name(lambda: importlib.import_module("not_in_the_dict"))
            ),
            "ModuleNotFoundError",
        ),
        ("runs once", with_importer(imports_once), 1),
    ]
)

FAIL  plain module: raised NotImplementedError: Lab 3.3: implement DictImporter
FAIL  spec origin and loader: raised NotImplementedError: Lab 3.3: implement DictImporter
FAIL  package + relative import: raised NotImplementedError: Lab 3.3: implement DictImporter
FAIL  package has __path__: raised NotImplementedError: Lab 3.3: implement DictImporter
FAIL  unknown name -> None: raised NotImplementedError: Lab 3.3: implement DictImporter
FAIL  missing module: raised NotImplementedError: Lab 3.3: implement DictImporter
FAIL  runs once: raised NotImplementedError: Lab 3.3: implement DictImporter
0/7 checks passed


#### Exercise 5 — read packaged data with `importlib.resources`

The `catalog` package ships `data/products.csv` (columns `name,price`) and
`data/tax_rates.json`. Write three functions that take the **package name** and use
`importlib.resources.files(...)` only (no `__file__` arithmetic), so they also work
when the package is imported from a zip file:

- `load_tax_rates(package)` → the JSON object as a `dict[str, float]`,
- `product_names(package)` → the `name` column in file order,
- `data_file_names(package)` → sorted file names in `data/`.

The cell below shows why `__file__` arithmetic breaks for a zipped package.

In [189]:
CATALOG_FILES = {
    "catalog/__init__.py": "",
    "catalog/data/products.csv": "name,price\nGreen tea,120.50\nBasmati rice,80\nTurmeric,45\n",
    "catalog/data/tax_rates.json": '{"food": 0.05, "electronics": 0.18}',
}
with temporary_modules(CATALOG_FILES, zipped=True):
    catalog = importlib.import_module("catalog")
    naive_path = Path(catalog.__file__).parent / "data" / "tax_rates.json"
    print(f"zipped: catalog.__file__ ends with {'/'.join(Path(catalog.__file__).parts[-3:])!r}")
    print(f"naive open(): {raised_error_name(naive_path.read_text)}")

zipped: catalog.__file__ ends with 'bundle.zip/catalog/__init__.py'
naive open(): NotADirectoryError


The stubs:

In [190]:
def load_tax_rates(package: str) -> dict[str, float]:
    """data/tax_rates.json from the package."""
    raise NotImplementedError("Lab 3.3: implement load_tax_rates")


def product_names(package: str) -> list[str]:
    """The name column of data/products.csv, in file order."""
    raise NotImplementedError("Lab 3.3: implement product_names")


def data_file_names(package: str) -> list[str]:
    """Sorted file names in data/."""
    raise NotImplementedError("Lab 3.3: implement data_file_names")

Run the checks, from a folder and from a zip.

In [191]:
def read_catalog(zipped: bool) -> Callable[[], object]:
    def read() -> object:
        with temporary_modules(CATALOG_FILES, zipped=zipped):
            return load_tax_rates("catalog"), product_names("catalog"), data_file_names("catalog")

    return read


EXPECTED_CATALOG = (
    {"food": 0.05, "electronics": 0.18},
    ["Green tea", "Basmati rice", "Turmeric"],
    ["products.csv", "tax_rates.json"],
)
run_checks(
    [
        ("from a folder", read_catalog(zipped=False), EXPECTED_CATALOG),
        ("from a zip", read_catalog(zipped=True), EXPECTED_CATALOG),
    ]
)

FAIL  from a folder: raised NotImplementedError: Lab 3.3: implement load_tax_rates
FAIL  from a zip: raised NotImplementedError: Lab 3.3: implement load_tax_rates
0/2 checks passed


#### Exercise 6 — measure and reduce CLI startup with lazy imports

`REPORT_CLI` (given) imports six modules at the top but needs only `decimal`, and
only for its `total` command; `--version` needs none of them.

1. `parse_importtime(stderr_text)` → `{module: cumulative_microseconds}` from
   `python -X importtime` output: keep only lines starting with `"import time:"`,
   skip the header, strip the indentation from the module name.
2. `make_lazy(source)` → the source with every **module-level** `import` /
   `from … import` statement turned into `lazy import` / `lazy from … import`.
   Leave alone imports inside functions, `if` blocks or `try`, star imports and
   `from __future__` imports. (Hint: `ast.parse(source).body` holds the
   module-level statements and their line numbers.)

The checks run both versions with `-X importtime` and compare what they load.

In [192]:
REPORT_CLI = """
import asyncio
import email.parser
import http.client
import json
import xml.etree.ElementTree as ElementTree
from decimal import Decimal
import sys


def main(argv):
    if argv[:1] == ["--version"]:
        print("report 2.0")
    elif argv[:1] == ["total"]:
        print(sum(Decimal(value) for value in argv[1:]))
    else:
        print("usage: report [--version | total VALUE...]")


if __name__ == "__main__":
    main(sys.argv[1:])
"""


def parse_importtime(stderr_text: str) -> dict[str, int]:
    """{module: cumulative microseconds} from `python -X importtime` output."""
    raise NotImplementedError("Lab 3.3: implement parse_importtime")


def make_lazy(source: str) -> str:
    """Turn module-level imports into lazy imports."""
    raise NotImplementedError("Lab 3.3: implement make_lazy")

Run the checks. They need Python 3.15 for the `lazy` keyword.

In [193]:
SAMPLE_IMPORTTIME = """
import time: self [us] | cumulative | imported package
import time:       120 |        120 |   _json
import time:       850 |       1300 | json
import time:        90 |         90 |     _decimal
import time:       400 |        490 |   decimal
garbage line
""".strip()


def run_cli(source: str, *arguments: str) -> tuple[str, set[str]]:
    """Run a CLI source with -X importtime; return (stdout's last line, imported modules)."""
    with tempfile.TemporaryDirectory() as folder:
        script = Path(folder) / "report.py"
        script.write_text(source)
        completed = subprocess.run(
            [sys.executable, "-X", "importtime", str(script), *arguments],
            capture_output=True, text=True, env=CHILD_ENV, timeout=60,
        )  # fmt: skip
    return completed.stdout.strip(), set(parse_importtime(completed.stderr))


def startup_comparison() -> tuple[bool, bool, bool, bool]:
    eager_output, eager_modules = run_cli(REPORT_CLI, "--version")
    lazy_output, lazy_modules = run_cli(make_lazy(REPORT_CLI), "--version")
    print(f"    modules loaded for --version: eager {len(eager_modules)}, lazy {len(lazy_modules)}")
    return (
        eager_output == lazy_output == "report 2.0",
        "asyncio" in eager_modules,
        "asyncio" in lazy_modules,
        len(lazy_modules) * 2 < len(eager_modules),
    )


KEEP_SAMPLE = (
    "from __future__ import annotations\nimport os\n\n"
    "def f():\n    import json\n\n"
    "try:\n    import tomllib\nexcept ImportError:\n    pass\n"
)
run_checks(
    [
        (
            "parse sample",
            lambda: parse_importtime(SAMPLE_IMPORTTIME),
            {"_json": 120, "json": 1300, "_decimal": 90, "decimal": 490},
        ),
        (
            "make_lazy rewrites top-level imports",
            lambda: [line for line in make_lazy(REPORT_CLI).splitlines() if "import" in line][:2],
            ["lazy import asyncio", "lazy import email.parser"],
        ),
        (
            "make_lazy leaves the rest",
            lambda: make_lazy(KEEP_SAMPLE),
            KEEP_SAMPLE.replace("\nimport os", "\nlazy import os"),
        ),
        (
            "--version: same output, asyncio skipped, half the modules",
            startup_comparison,
            (True, True, False, True),
        ),
        (
            "total still works lazily",
            lambda: run_cli(make_lazy(REPORT_CLI), "total", "1.10", "2.20")[0],
            "3.30",
        ),
    ]
)

FAIL  parse sample: raised NotImplementedError: Lab 3.3: implement parse_importtime
FAIL  make_lazy rewrites top-level imports: raised NotImplementedError: Lab 3.3: implement make_lazy
FAIL  make_lazy leaves the rest: raised NotImplementedError: Lab 3.3: implement make_lazy


FAIL  --version: same output, asyncio skipped, half the modules: raised NotImplementedError: Lab 3.3: implement parse_importtime
FAIL  total still works lazily: raised NotImplementedError: Lab 3.3: implement make_lazy
0/5 checks passed


### Recap — Unit 3.3

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Modules as objects | `ModuleType` + `__spec__` / `__loader__` / `__path__`; the module dict is the globals | module records |
| What `import` does | cache → finders → loader → insert in `sys.modules` → execute → bind | resolve → instantiate → evaluate |
| Import forms | `from m import x` copies a binding; `__all__` controls `*` | ESM live bindings |
| Run once | Top-level code runs on the first import; module = singleton; `reload` is a REPL tool | module caching |
| `sys.path` | script folder or cwd, `PYTHONPATH`, stdlib, `site-packages` + `.pth`; `-P` drops the first | `node_modules` walk |
| Packages | `__init__.py` runs and re-exports; namespace packages merge folders | `package.json` `exports` |
| Relative imports | Need a parent package: use `python -m` | `./` specifiers |
| Entry points | `__name__ == "__main__"`, `__main__.py`, `[project.scripts]` | `require.main`, `bin` |
| Circular imports | Third module, module import, local import, `TYPE_CHECKING` | partial `exports` |
| `importlib` | `import_module`, `resources.files`, `metadata`, `pkgutil.iter_modules`, entry points | `import()`, `require.resolve` |
| Import hooks | Finders + loaders on `sys.meta_path` (and `sys.path_hooks`) | loaders / custom resolvers |
| Lazy imports | `lazy import` (3.15), `LazyLoader`; measure with `-X importtime` | dynamic `import()` |

**Further reading**

- [The import system](https://docs.python.org/3/reference/import.html),
  [`importlib`](https://docs.python.org/3/library/importlib.html),
  [`importlib.resources`](https://docs.python.org/3/library/importlib.resources.html),
  [`importlib.metadata`](https://docs.python.org/3/library/importlib.metadata.html),
  [`pkgutil`](https://docs.python.org/3/library/pkgutil.html)
- [`sys.path` initialization](https://docs.python.org/3/library/sys_path_init.html),
  [`site`](https://docs.python.org/3/library/site.html),
  [`__main__`](https://docs.python.org/3/library/__main__.html),
  [`-X importtime`](https://docs.python.org/3/using/cmdline.html#cmdoption-X)
- [PEP 302 — New import hooks](https://peps.python.org/pep-0302/),
  [PEP 420 — Implicit namespace packages](https://peps.python.org/pep-0420/),
  [PEP 451 — ModuleSpec](https://peps.python.org/pep-0451/),
  [PEP 649 — Deferred evaluation of annotations](https://peps.python.org/pep-0649/),
  [PEP 810 — Explicit lazy imports](https://peps.python.org/pep-0810/)

## Phase 3 Project — Internals Toolkit

Build `pyinspect`, a small command-line toolkit that turns this phase's
instruments into reusable tools, run with `python -m pyinspect`:

| Requirement | Command | Built on |
| --- | --- | --- |
| A memory-profiler wrapper around `tracemalloc` reporting the top allocating lines of any callable | `memory module:function` | 3.1.11–3.1.12, Lab 3.1 exercise 6 |
| An AST-based code scanner | `scan PATH...` | 3.2.2, 3.2.12, Lab 3.2 exercises 3 and 7 |
| An import-time report | `imports MODULE` | 3.3.12, Lab 3.3 exercise 6 |
| A package run with `python -m` | `__main__.py` | 3.3.6, 3.3.8 |

The code outgrows a notebook, so it lives in a real package in the repository's
`projects/` folder:

```text
projects/pyinspect/
├── __init__.py     version + lazy re-exports of the public API (3.3.12)
├── __main__.py     `python -m pyinspect` entry point (3.3.8)
├── cli.py          argparse subcommands and exit codes
├── memory.py       profile_allocations, format_memory_report
├── scanner.py      rule registry, scan_source, scan_paths, format_findings
├── importtime.py   parse_importtime, import_subtree, measure_import, format_import_report
└── examples.py     sample workloads for the memory command
```

Each step reads the relevant code with `inspect.getsource` (so the notebook always
shows the real file), uses it from the notebook, then runs the same feature from
the command line in a child process.

> **Note:** memory sizes and import times vary by machine and Python version; the
> self-check in Step 6 compares relationships, never exact numbers. `pyinspect`
> needs Python 3.15 because it uses `lazy` imports.

### Step 1 — the package and `python -m pyinspect`

The setup cell puts `projects/` on `sys.path` so the notebook can import the
package, and defines `run_pyinspect(*arguments, cwd=PROJECTS_DIR)` to run the CLI in a
child process (as you would in a terminal), plus the Unit 3.3 helpers the review
questions need.

In [194]:
import ast
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
import sysconfig
import textwrap
import tracemalloc
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from pathlib import Path

CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0"}
PROJECTS_DIR = find_repo_root() / "projects"
scratch_dir = find_repo_root() / "notebooks" / "_scratch"  # git-ignored
if str(PROJECTS_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECTS_DIR))  # so the notebook can import pyinspect

import pyinspect  # noqa: E402
from pyinspect import examples, scanner  # noqa: E402


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour of the running interpreter."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"


def run_pyinspect(*arguments: str, cwd: Path = PROJECTS_DIR) -> tuple[int, str]:
    """Run `python -m pyinspect ...` in a child process; return (exit status, output)."""
    completed = subprocess.run(
        [sys.executable, "-m", "pyinspect", *arguments],
        capture_output=True, text=True, cwd=cwd, timeout=120,
        env={**CHILD_ENV, "PYTHONPATH": str(PROJECTS_DIR)},
    )  # fmt: skip
    return completed.returncode, (completed.stdout + completed.stderr).strip()


def run_python(*arguments: str, cwd: Path | None = None) -> str:
    """Run the kernel's interpreter in a child process; return its output (cwd as <project>)."""
    completed = subprocess.run(
        [sys.executable, *arguments],
        capture_output=True,
        text=True,
        cwd=cwd,
        env=CHILD_ENV,
        timeout=120,
    )
    output = (completed.stdout + completed.stderr).strip()
    return output.replace(str(cwd), "<project>") if cwd is not None else output


@contextmanager
def scratch_project(name: str, files: dict[str, str]) -> Iterator[Path]:
    """Write files under notebooks/_scratch/<name>/ (Unit 3.3); delete them afterwards."""
    root = scratch_dir / name
    shutil.rmtree(root, ignore_errors=True)
    root.mkdir(parents=True)
    for relative_path, text in files.items():
        target = root / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(textwrap.dedent(text).lstrip("\n"))
    try:
        yield root
    finally:
        shutil.rmtree(root, ignore_errors=True)


def run_script(root: Path, script: str) -> str:
    """Save script as root/main.py and run it in a child interpreter from root (Unit 3.3)."""
    (root / "main.py").write_text(textwrap.dedent(script).lstrip("\n"))
    return run_python("main.py", cwd=root)


print(interpreter_label())
print(f"pyinspect {pyinspect.__version__} from {Path(pyinspect.__file__).parent.parent.name}/")

CPython 3.15.1, default (GIL) build, x86_64
pyinspect 0.1.0 from projects/


The files on disk:

In [195]:
package_dir = PROJECTS_DIR / "pyinspect"
for path in sorted(package_dir.glob("*.py")):
    print(f"{path.name:<14} {len(path.read_text().splitlines()):>4} lines")

__init__.py      47 lines
__main__.py       5 lines
cli.py           81 lines
examples.py      25 lines
importtime.py   101 lines
memory.py       116 lines
scanner.py      157 lines


`__main__.py` is a two-line shim that calls `cli.main()` and turns its return value
into the exit status, so the logic stays importable and testable (3.3.8). The CLI
uses `argparse` subcommands and three exit codes: 0 success, 1 findings (for
`scan`, so CI can fail a build), 2 usage errors.

In [196]:
print((package_dir / "__main__.py").read_text())
status, output = run_pyinspect("--version")
print(f"python -m pyinspect --version -> {output!r} (exit {status})")
status, output = run_pyinspect("--help")
print("\n".join(output.splitlines()[:9]))

"""Entry point for ``python -m pyinspect``."""

from pyinspect.cli import main

raise SystemExit(main())



python -m pyinspect --version -> 'pyinspect 0.1.0' (exit 0)


usage: pyinspect [-h] [--version] {memory,scan,imports} ...

CPython internals toolkit

positional arguments:
  {memory,scan,imports}
    memory              top allocating lines of a callable
    scan                AST-based code scanner
    imports             import-time report for a module


**Lazy re-exports.** `pyinspect/__init__.py` re-exports the public API with
`lazy from ... import ...` (3.3.12), so `import pyinspect` loads only the package
itself; each submodule loads on first use. The kernel imported `examples` and
`scanner` explicitly; the other two are still pending:

In [197]:
lazy_lines = [line for line in inspect.getsource(pyinspect).splitlines() if line.startswith("lazy")]
print("\n".join(lazy_lines))
loaded = sorted(name for name in sys.modules if name.startswith("pyinspect."))
print(f"submodules loaded so far: {loaded}")
print(f"{type(vars(pyinspect)['measure_import']).__name__=}  (still a lazy proxy)")

lazy from pyinspect.importtime import (
lazy from pyinspect.memory import (
lazy from pyinspect.scanner import RULES, Finding, format_findings, scan_paths, scan_source
submodules loaded so far: ['pyinspect.examples', 'pyinspect.scanner']
type(vars(pyinspect)['measure_import']).__name__='lazy_import'  (still a lazy proxy)


### Step 2 — a memory profiler around `tracemalloc`

`profile_allocations(function, *args, top=10, min_bytes=1024, **kwargs)` returns a
`MemoryReport` with:

- `peak_bytes`: the highest extra memory **during** the call (temporaries included),
  from `tracemalloc.reset_peak()` and `get_traced_memory()`,
- `retained_bytes` and `sites`: memory **still held** when the call returned
  (its return value, caches, leaks), from a snapshot diff grouped by line,
- the callable's `result`.

Design points from Unit 3.1: tracing is started only if it was off and stopped
again in `finally`; frames from `tracemalloc` itself are filtered out; a warm-up
`filter_traces` call compiles the filter patterns **before** the first snapshot
(otherwise the regex cache's own allocations would show up as "allocating
lines"), and lines below `min_bytes` are dropped as noise.

In [198]:
print(inspect.getsource(pyinspect.profile_allocations))

def profile_allocations(
    function: Callable[..., object],
    /,
    *args: object,
    top: int = 10,
    min_bytes: int = 1024,
    **kwargs: object,
) -> MemoryReport:
    """Call ``function(*args, **kwargs)`` under tracemalloc.

    ``peak_bytes`` is the highest extra memory during the call (temporaries included);
    ``retained_bytes`` and ``sites`` describe memory still held when it returned, including
    its return value. Lines holding less than ``min_bytes`` are left out (allocator and
    regex-cache noise). Tracing is left as it was found.
    """
    if top < 1:
        raise ValueError("top must be at least 1")
    started_here = not tracemalloc.is_tracing()
    if started_here:
        tracemalloc.start()
    ignore = [tracemalloc.Filter(False, pattern) for pattern in _IGNORED_FILES]
    try:
        tracemalloc.take_snapshot().filter_traces(ignore)  # warm-up: compiles the filter patterns
        before = tracemalloc.take_snapshot().filter_traces(ignore)
        trac

**From the notebook.** Any callable works, including one defined in a cell (its
source lines come from `linecache`, 3.1.11; the "file" of a notebook cell is a
generated number). The returned index is what stays in memory.

In [199]:
def build_order_index(order_count: int) -> dict[int, dict[str, object]]:
    """Index of fake orders by id."""
    return {order_id: {"id": order_id, "items": ["TEA-01"] * 3} for order_id in range(order_count)}


order_report = pyinspect.profile_allocations(build_order_index, 30_000, top=3)
print(pyinspect.format_memory_report(order_report))
print(f"result: {len(order_report.result):,} orders; tracing left on: {tracemalloc.is_tracing()}")

memory report for build_order_index
  peak during the call:   9.7 MiB
  held after the call:    9.7 MiB
  top allocating lines (memory still held):
       9.7 MiB   148,673 blocks  36043318.py:3      return {order_id: {"id": order_id, "items": ["TEA-01"] * 3} for order_id in range(order_count)}
result: 30,000 orders; tracing left on: False


**Spotting a leak.** `examples.fill_price_cache` adds entries to a module-level dict
that is never emptied (3.1.12). Profiling it twice in the same process shows memory
retained **every** time at the same line: the signature of a leak. The cache is
emptied afterwards.

In [200]:
for attempt in (1, 2):
    leak_report = pyinspect.profile_allocations(examples.fill_price_cache, top=1)
    retained = pyinspect.memory.format_bytes(leak_report.retained_bytes)
    top_site = leak_report.sites[0]
    print(f"call {attempt}: retained {retained} at {top_site.location}: {top_site.source}")
examples._PRICE_CACHE.clear()

call 1: retained 924.4 KiB at examples.py:15: _PRICE_CACHE[f"SKU-{number:05d}"] = number * 1.5


call 2: retained 1.1 MiB at examples.py:15: _PRICE_CACHE[f"SKU-{number:05d}"] = number * 1.5


**From the command line.** `memory module:function` imports the target in a fresh
interpreter and calls it with no arguments. A function that only churns temporaries
has a high peak and retains nothing. A bad target exits with status 2.

In [201]:
for target in ("pyinspect.examples:build_price_table", "pyinspect.examples:churn_temporaries"):
    status, output = run_pyinspect("memory", target, "--top", "2")
    print(f"$ python -m pyinspect memory {target}   (exit {status})\n{output}\n")
print(run_pyinspect("memory", "pyinspect.examples:no_such_function"))

$ python -m pyinspect memory pyinspect.examples:build_price_table   (exit 0)
memory report for build_price_table
  peak during the call:   5.1 MiB
  held after the call:    5.1 MiB
  top allocating lines (memory still held):
       5.1 MiB    79,905 blocks  examples.py:8      return [{"sku": f"SKU-{number:05d}", "price": number * 1.5} for number in range(20_000)]



$ python -m pyinspect memory pyinspect.examples:churn_temporaries   (exit 0)
memory report for churn_temporaries
  peak during the call:   851.1 KiB
  held after the call:    0 B
  top allocating lines (memory still held):
    (nothing retained)

(2, "pyinspect: error: module 'pyinspect.examples' has no attribute 'no_such_function'")


### Step 3 — an AST-based code scanner

Rules are plain functions registered with a decorator (the registry pattern from
Units 1.12 and 2.6). Each takes the parsed module and yields `(node, message)` pairs;
`scan_source` turns them into sortable `Finding` dataclasses (`path`, `line`,
`column`, `code`, `message`). A file that does not parse yields one `E999`
finding instead of crashing the scan. The registered rules, and one rule's code:

In [202]:
for registered in pyinspect.RULES.values():
    print(f"{registered.code}  {registered.description:<38} ({registered.check.__name__})")
print()
print(inspect.getsource(scanner.missing_docstrings))

B006  mutable default argument               (mutable_defaults)
E722  bare except                            (bare_except)
S307  eval() or exec() call                  (eval_exec)
D103  public function without a docstring    (missing_docstrings)

@rule("D103", "public function without a docstring")
def missing_docstrings(tree: ast.Module) -> Iterator[tuple[ast.AST, str]]:
    """Module-level and class-level public functions should say what they do."""
    containers: list[ast.AST] = [tree, *(n for n in tree.body if isinstance(n, ast.ClassDef))]
    for container in containers:
        for node in container.body:
            if isinstance(node, ast.FunctionDef | ast.AsyncFunctionDef):
                if not node.name.startswith("_") and ast.get_docstring(node) is None:
                    yield node, f"public function {node.name}() has no docstring"



**Scanning a sample project.** `scan_paths` accepts files and folders, skips
`__pycache__` and dot-folders, sorts findings by path, line and column, and can
report paths relative to a folder. The same scan runs through the CLI from inside
the project folder; it exits with status 1 because there are findings, and
`--json` gives machine-readable output.

In [203]:
SAMPLE_PROJECT = {
    "app/handlers.py": """
        def handle(event, seen=[]):
            try:
                return eval(event["expression"])
            except:
                return None
        """,
    "app/models.py": """
        class Order:
            def total(self):
                return 0

            def _private(self):
                return 1
        """,
    "app/clean.py": (
        '"""Clean module."""\n\n\ndef ok(value=None):\n    """Return value."""\n    return value\n'
    ),
    "app/broken.py": "def broken(:\n    pass\n",
    "app/.cache/ignored.py": "def ignored(x=[]):\n    pass\n",
}
with scratch_project("project3_sample", SAMPLE_PROJECT) as folder:
    sample_findings = pyinspect.scan_paths([folder / "app"], relative_to=folder)
    cli_status, cli_text = run_pyinspect("scan", "app", cwd=folder)
    json_status, json_text = run_pyinspect("scan", "app", "--json", cwd=folder)
print(pyinspect.format_findings(sample_findings))
same_text = cli_text == pyinspect.format_findings(sample_findings)
print(f"\nCLI exit status: {cli_status}; same text as in-process: {same_text}")
print(f"--json: {len(json.loads(json_text))} findings, first = {json.loads(json_text)[0]}")

app/broken.py:1:12: E999 syntax error: invalid syntax
app/handlers.py:1:1: D103 public function handle() has no docstring
app/handlers.py:1:24: B006 mutable default argument in handle()
app/handlers.py:3:16: S307 use of eval(): use ast.literal_eval or a parser
app/handlers.py:4:5: E722 bare except: catch a specific exception
app/models.py:2:5: D103 public function total() has no docstring
6 finding(s) in 3 file(s)

CLI exit status: 1; same text as in-process: True
--json: 6 findings, first = {'path': 'app/broken.py', 'line': 1, 'column': 12, 'code': 'E999', 'message': 'syntax error: invalid syntax'}


**Extending the scanner.** A new rule is one decorated function. The cell
registers a rule against `global` statements, scans a snippet, and removes the rule
again so later cells see the original rule set.

In [204]:
@scanner.rule("W001", "global statement")
def global_statements(tree: ast.Module) -> Iterator[tuple[ast.AST, str]]:
    """`global` makes functions depend on hidden module state (1.11)."""
    for node in ast.walk(tree):
        if isinstance(node, ast.Global):
            yield node, f"global statement for {', '.join(node.names)}"


try:
    snippet = 'COUNT = 0\n\n\ndef bump():\n    """Add one."""\n    global COUNT\n    COUNT += 1\n'
    for finding in pyinspect.scan_source(snippet, "counter.py"):
        print(finding)
finally:
    del pyinspect.RULES["W001"]
print(f"rules now: {list(pyinspect.RULES)}")

counter.py:6:5: W001 global statement for COUNT
rules now: ['B006', 'E722', 'S307', 'D103']


### Step 4 — an import-time report

`measure_import(module)` runs `python -X importtime -c "import module"` in a fresh
interpreter (the child gets the caller's `sys.path`), and `parse_importtime` turns
each line into an `ImportTiming(module, self_us, cumulative_us, depth)`; the module
name is validated first, because it is pasted into `-c` code.

The report must not count the interpreter's own start-up imports (`site`,
`codecs`, ...), which `-X importtime` also prints. CPython prints a module **after**
its dependencies, indented deeper, so `import_subtree` takes the target's line and
the run of deeper lines just above it. On a fixed sample:

In [205]:
SAMPLE_REPORT = """
import time: self [us] | cumulative | imported package
import time:       300 |        300 |   encodings.utf_8
import time:       500 |        800 | site
import time:       120 |        120 |     _json
import time:       400 |        520 |   json.decoder
import time:       210 |        210 |   json.encoder
import time:       650 |       1380 | json
"""
sample_timings = pyinspect.parse_importtime(SAMPLE_REPORT)
print(f"parsed {len(sample_timings)} lines; first: {sample_timings[0]}")
json_subtree = pyinspect.import_subtree(sample_timings, "json")
print(f"subtree of json: {[timing.module for timing in json_subtree]}")
print(pyinspect.format_import_report("json", sample_timings, top=2))
print(inspect.getsource(pyinspect.importtime.import_subtree).split('"""')[-1].rstrip())

parsed 6 lines; first: ImportTiming(module='encodings.utf_8', self_us=300, cumulative_us=300, depth=1)
subtree of json: ['_json', 'json.decoder', 'json.encoder', 'json']
import report for json
  modules imported:   4
  total (cumulative): 1.4 ms
  top 2 by self time:
       0.65 ms  json (depth 0)
       0.40 ms  json.decoder (depth 1)

    for index in range(len(timings) - 1, -1, -1):
        if timings[index].module == module_name and timings[index].depth == 0:
            first = index
            while first > 0 and timings[first - 1].depth > 0:
                first -= 1
            return timings[first : index + 1]
    return []


**Real measurements** through the CLI (times vary by machine and run). `os` is
already loaded at start-up, so there is nothing to report; an unknown module exits
with status 2.

In [206]:
for module_name in ("json", "asyncio", "os", "no_such_module"):
    status, output = run_pyinspect("imports", module_name, "--top", "3")
    print(f"$ python -m pyinspect imports {module_name}   (exit {status})\n{output}\n")

$ python -m pyinspect imports json   (exit 0)
import report for json
  modules imported:   24
  total (cumulative): 19.1 ms
  top 3 by self time:
       3.44 ms  enum (depth 3)
       2.77 ms  collections (depth 4)
       1.63 ms  functools (depth 3)



$ python -m pyinspect imports asyncio   (exit 0)
import report for asyncio
  modules imported:   121
  total (cumulative): 50.0 ms
  top 3 by self time:
       2.51 ms  logging (depth 4)
       2.22 ms  ssl (depth 2)
       2.15 ms  inspect (depth 3)



$ python -m pyinspect imports os   (exit 0)
import report for os: already loaded at interpreter start-up



$ python -m pyinspect imports no_such_module   (exit 2)
pyinspect: error: cannot import no_such_module: ModuleNotFoundError: No module named 'no_such_module'



### Step 5 — the toolkit inspects itself

Three checks on `pyinspect` with its own tools:

1. **scan**: the package passes its own rules (exit status 0).
2. **imports**: thanks to the lazy re-exports, importing `pyinspect` loads one
   module.
3. **memory**: what does scanning a real package (the standard library's `json`)
   cost? `profile_allocations` forwards extra keyword arguments to the callable.

In [207]:
status, output = run_pyinspect("scan", "pyinspect")
print(f"scan pyinspect -> {output!r} (exit {status})")
status, output = run_pyinspect("imports", "pyinspect")
print(output)
json_package = Path(json.__file__).parent
scan_report = pyinspect.profile_allocations(
    pyinspect.scan_paths, [json_package], relative_to=json_package.parent, top=2
)
print(f"\nscanning the json package: {len(scan_report.result)} findings")
print(pyinspect.format_memory_report(scan_report))

scan pyinspect -> 'no findings' (exit 0)
import report for pyinspect
  modules imported:   1
  total (cumulative): 0.2 ms
  top 1 by self time:
       0.17 ms  pyinspect (depth 0)



scanning the json package: 5 findings
memory report for scan_paths
  peak during the call:   1.0 MiB
  held after the call:    16.3 KiB
  top allocating lines (memory still held):
      14.3 KiB       206 blocks  ast.py:46          return compile(source, filename, mode, flags,


### Step 6 — self-check

Twelve checks against the requirements. They compare relationships (bigger than,
same line, exit status), never exact byte counts or timings.

In [208]:
def price_table_report() -> pyinspect.MemoryReport:
    return pyinspect.profile_allocations(examples.build_price_table)


def churn_is_peak_only() -> bool:
    report = pyinspect.profile_allocations(examples.churn_temporaries)
    return report.retained_bytes * 10 < report.peak_bytes


def sample_scan() -> list[tuple[str, int, str]]:
    with scratch_project("project3_sample", SAMPLE_PROJECT) as folder:
        findings = pyinspect.scan_paths([folder / "app"], relative_to=folder)
    return [(finding.path, finding.line, finding.code) for finding in findings]


EXPECTED_SAMPLE = [
    ("app/broken.py", 1, "E999"),
    ("app/handlers.py", 1, "D103"),
    ("app/handlers.py", 1, "B006"),
    ("app/handlers.py", 3, "S307"),
    ("app/handlers.py", 4, "E722"),
    ("app/models.py", 2, "D103"),
]
run_checks(
    [
        (
            "CLI --version",
            lambda: run_pyinspect("--version"),
            (0, f"pyinspect {pyinspect.__version__}"),
        ),
        (
            "memory: top line is the list",
            lambda: "return [" in price_table_report().sites[0].source,
            True,
        ),
        (
            "memory: retains over 1 MB",
            lambda: price_table_report().retained_bytes > 1_000_000,
            True,
        ),
        ("memory: temporaries are peak only", churn_is_peak_only, True),
        (
            "memory: tracing left off",
            lambda: (pyinspect.profile_allocations(len, "abc"), tracemalloc.is_tracing())[1],
            False,
        ),
        (
            "memory: top must be >= 1",
            lambda: raised_error_name(lambda: pyinspect.profile_allocations(len, "a", top=0)),
            "ValueError",
        ),
        ("scan: sample findings", sample_scan, EXPECTED_SAMPLE),
        ("scan: CLI exit status on pyinspect", lambda: run_pyinspect("scan", "pyinspect")[0], 0),
        (
            "imports: subtree excludes start-up",
            lambda: [t.module for t in pyinspect.import_subtree(sample_timings, "json")],
            ["_json", "json.decoder", "json.encoder", "json"],
        ),
        (
            "imports: lazy pyinspect loads one module",
            lambda: len(
                pyinspect.import_subtree(pyinspect.measure_import("pyinspect"), "pyinspect")
            ),
            1,
        ),
        (
            "imports: code injection refused",
            lambda: raised_error_name(lambda: pyinspect.measure_import("os; print(1)")),
            "ValueError",
        ),
        ("CLI: bad target exits 2", lambda: run_pyinspect("memory", "nope")[0], 2),
    ]
)

PASS  CLI --version


PASS  memory: top line is the list


PASS  memory: retains over 1 MB


PASS  memory: temporaries are peak only
PASS  memory: tracing left off
PASS  memory: top must be >= 1
PASS  scan: sample findings
PASS  scan: CLI exit status on pyinspect
PASS  imports: subtree excludes start-up
PASS  imports: lazy pyinspect loads one module
PASS  imports: code injection refused
PASS  CLI: bad target exits 2
12/12 checks passed


### Phase 3 review — 15 predict-the-output questions

Fifteen short snippets, five from each unit. For each one, decide what it prints
**before** opening the answer, then run the proof cell. If a question surprises
you, go back to the section it names. Snippets leave out type hints to stay short;
the proof cells add them. Snippets that need module files run in a child
interpreter from `notebooks/_scratch/` (as in Unit 3.3), so the kernel stays clean.

| Unit | Questions |
| --- | --- |
| 3.1 | 5 |
| 3.2 | 5 |
| 3.3 | 5 |

#### Question 1 — Unit 3.1

**Predict the output.**

```python
import weakref

try:
    weakref.ref([1, 2])
except TypeError as error:
    print(type(error).__name__)
```

<details><summary>Answer</summary>

`TypeError`. Built-in `list`, `dict`, `tuple`, `int` and `str` objects have no slot
for weak references (3.1.8). Instances of your own classes (even a `list` subclass)
can be weakly referenced.

</details>

Run the next cell to check your prediction.

In [209]:
import weakref

try:
    weakref.ref([1, 2])
except TypeError as error:
    print(type(error).__name__)

TypeError


#### Question 2 — Unit 3.1

**Predict the output.**

```python
import weakref


class Box:
    pass


log = []
box = Box()
weakref.finalize(box, log.append, "freed")
alias = box
del box
log.append("after del")
del alias
print(log)
```

<details><summary>Answer</summary>

`['after del', 'freed']`. `del box` removes one of two references, so the object
survives; the finalizer runs at `del alias`, when the count reaches zero (3.1.3).

</details>

Run the next cell to check your prediction.

In [210]:
import weakref


class Box:
    pass


log = []
box = Box()
weakref.finalize(box, log.append, "freed")
alias = box
del box
log.append("after del")
del alias
print(log)

['after del', 'freed']


#### Question 3 — Unit 3.1

**Predict the output.**

```python
import functools, gc, weakref

class Engine:
    @functools.cache
    def quote(self, sku):
        return len(sku)

engine = Engine()
engine.quote("TEA-01")
probe = weakref.ref(engine)
del engine
gc.collect()
print(probe() is not None)
```

<details><summary>Answer</summary>

`True`. The cache lives on the class and its keys include `self`, so it keeps the
instance alive; not even `gc.collect()` can free it (3.1.12).

</details>

Run the next cell to check your prediction.

In [211]:
import functools
import gc
import weakref


class Engine:
    @functools.cache  # noqa: B019 (the leak being asked about)
    def quote(self, sku: str) -> int:
        return len(sku)


engine = Engine()
engine.quote("TEA-01")
probe = weakref.ref(engine)
del engine
gc.collect()
print(probe() is not None)
Engine.quote.cache_clear()  # release it again

True


#### Question 4 — Unit 3.1

**Predict the output.**

```python
print(int("1000") is int("1000"))
```

<details><summary>Answer</summary>

`True` on 3.15, `False` on 3.14 and earlier. The small-int cache covers -5 to 1024
since 3.15 (it ended at 256 before), so both calls return the same immortal object
(3.1.10). Never rely on `is` for numbers.

</details>

Run the next cell to check your prediction.

In [212]:
print(int("1000") is int("1000"))

True


#### Question 5 — Unit 3.1

**Predict the output.**

```python
import gc

saved = []

class Phoenix:
    calls = 0

    def __del__(self):
        Phoenix.calls += 1
        saved.append(self)

bird = Phoenix()
del bird
saved.clear()
gc.collect()
print(Phoenix.calls)
```

<details><summary>Answer</summary>

`1`. The first `__del__` resurrects the object by storing it in `saved`. When it dies
again, PEP 442 guarantees the finalizer is not called a second time (3.1.7).

</details>

Run the next cell to check your prediction.

In [213]:
import gc

saved = []


class Phoenix:
    calls = 0

    def __del__(self) -> None:
        Phoenix.calls += 1
        saved.append(self)


bird = Phoenix()
del bird
saved.clear()
gc.collect()
print(Phoenix.calls)

1


#### Question 6 — Unit 3.2

**Predict the output.**

```python
def kib():
    return 2 ** 10

print(1024 in kib.__code__.co_consts)
```

<details><summary>Answer</summary>

`True`. The compiler folds `2 ** 10` into the constant `1024` (3.2.6); the function
just loads it.

</details>

Run the next cell to check your prediction.

In [214]:
def kib() -> int:
    return 2**10


print(1024 in kib.__code__.co_consts)

True


#### Question 7 — Unit 3.2

**Predict the output.**

```python
import ast

for text in ("1 + 2j", "1 + 2"):
    try:
        print(ast.literal_eval(text))
    except ValueError:
        print("ValueError")
```

<details><summary>Answer</summary>

`(1+2j)` then `ValueError`. `literal_eval` accepts `+` and `-` only to write complex
numbers; `1 + 2` is arithmetic, which it refuses (3.2.11). Use an AST evaluator like
Lab 3.2's `safe_eval` for arithmetic.

</details>

Run the next cell to check your prediction.

In [215]:
import ast

for text in ("1 + 2j", "1 + 2"):
    try:
        print(ast.literal_eval(text))
    except ValueError:
        print("ValueError")

(1+2j)
ValueError


#### Question 8 — Unit 3.2

**Predict the output.**

```python
def make():
    def inner():
        pass
    return inner

first, second = make(), make()
print(first is second, first.__code__ is second.__code__)
```

<details><summary>Answer</summary>

`False True`. Each call runs `def inner` again and builds a **new function object**,
but around the **same code object**, compiled once with `make` (3.2.4). That is also
why specialization state is shared between them (Lab 3.2, exercise 6).

</details>

Run the next cell to check your prediction.

In [216]:
def make() -> object:
    def inner() -> None:
        pass

    return inner


first, second = make(), make()
print(first is second, first.__code__ is second.__code__)

False True


#### Question 9 — Unit 3.2

**Predict the output.**

```python
try:
    eval("len('abc')", {"__builtins__": {}})
except NameError as error:
    print(error)
```

<details><summary>Answer</summary>

`name 'len' is not defined`. With an empty `__builtins__`, names like `len` are not
found. This is **not** a sandbox: `().__class__.__base__.__subclasses__()` still
reaches dangerous classes (3.2.11).

</details>

Run the next cell to check your prediction.

In [217]:
try:
    eval("len('abc')", {"__builtins__": {}})
except NameError as error:
    print(error)

name 'len' is not defined


#### Question 10 — Unit 3.2

**Predict the output.**

```python
def is_small(size):
    return size in [1, 2, 3]

print((1, 2, 3) in is_small.__code__.co_consts)
```

<details><summary>Answer</summary>

`True`. A list literal used only for `in` becomes a constant **tuple** built once at
compile time (3.2.6); a set literal would become a `frozenset`.

</details>

Run the next cell to check your prediction.

In [218]:
def is_small(size: int) -> bool:
    return size in [1, 2, 3]


print((1, 2, 3) in is_small.__code__.co_consts)

True


#### Question 11 — Unit 3.3

**Predict the output.**

```python
# counter_mod.py contains:
#     print("loading")
#     hits = 0
import counter_mod
import counter_mod
counter_mod.hits += 1
import counter_mod as again
print(again.hits)
```

<details><summary>Answer</summary>

```text
loading
1
```

The module's code runs once; later imports find it in `sys.modules` (3.3.4). `again`
is the same module object, so it sees the incremented `hits`.

</details>

Run the next cell to check your prediction.

In [219]:
COUNTER_SCRIPT = """
import counter_mod
import counter_mod

counter_mod.hits += 1
import counter_mod as again

print(again.hits)
"""
with scratch_project(
    "project3_counter", {"counter_mod.py": "print('loading')\nhits = 0\n"}
) as root:
    print(run_script(root, COUNTER_SCRIPT))

loading
1


#### Question 12 — Unit 3.3

**Predict the output.**

```python
# shapes.py contains:
#     __all__ = ["circle"]
#     def circle(): ...
#     def square(): ...
#     _scale = 2
from shapes import *
print("circle" in dir(), "square" in dir(), "_scale" in dir())
```

<details><summary>Answer</summary>

`True False False`. With `__all__` defined, `import *` takes exactly those names;
`square` is public but not listed, and `_scale` would be skipped even without
`__all__` (3.3.3).

</details>

Run the next cell to check your prediction.

In [220]:
SHAPES_FILES = {
    "shapes.py": """
        __all__ = ["circle"]

        def circle(): ...

        def square(): ...

        _scale = 2
        """,
}
SHAPES_SCRIPT = """
from shapes import *

print("circle" in dir(), "square" in dir(), "_scale" in dir())
"""
with scratch_project("project3_shapes", SHAPES_FILES) as root:
    print(run_script(root, SHAPES_SCRIPT))

True False False


#### Question 13 — Unit 3.3

**Predict the output.**

```python
# cycle_a.py:  import cycle_b
#              VALUE = 42
# cycle_b.py:  import cycle_a
#              print(hasattr(cycle_a, "VALUE"))
import cycle_a
print(cycle_a.VALUE)
```

<details><summary>Answer</summary>

```text
False
42
```

`cycle_a` is in `sys.modules` before its code runs. Its first line imports `cycle_b`,
whose `import cycle_a` returns the **partially initialized** module, which has no
`VALUE` yet (3.3.9). Plain `import` survives the cycle; `from cycle_a import VALUE`
would have raised `ImportError`.

</details>

Run the next cell to check your prediction.

In [221]:
cycle_files = {
    "cycle_a.py": "import cycle_b\n\nVALUE = 42\n",
    "cycle_b.py": "import cycle_a\n\nprint(hasattr(cycle_a, 'VALUE'))\n",
}
with scratch_project("project3_cycle", cycle_files) as root:
    print(run_script(root, "import cycle_a\n\nprint(cycle_a.VALUE)\n"))

False
42


#### Question 14 — Unit 3.3

**Predict the output.**

```python
import sys

sys.modules["config"] = 42
import config
print(config + 1)
```

<details><summary>Answer</summary>

`43`. `import` checks `sys.modules` first and returns whatever is cached there, even an
int (3.3.2). Test doubles and some plugin systems use this on purpose.

</details>

Run the next cell to check your prediction.

In [222]:
CONFIG_SCRIPT = """
import sys

sys.modules["config"] = 42
import config

print(config + 1)
"""
print(run_python("-c", textwrap.dedent(CONFIG_SCRIPT)))

43


#### Question 15 — Unit 3.3

**Predict the output.**

```python
# tools/report.py contains:  print(__name__)
# In a terminal:   python -m tools.report
# In Python:       import tools.report
```

<details><summary>Answer</summary>

```text
__main__
tools.report
```

Run as the program (with `-m` or as a file), a module's `__name__` is `"__main__"`;
imported, it is its qualified name (3.3.8). That is what the
`if __name__ == "__main__":` guard tests.

</details>

Run the next cell to check your prediction.

In [223]:
tools_files = {"tools/__init__.py": "", "tools/report.py": "print(__name__)\n"}
with scratch_project("project3_tools", tools_files) as root:
    print(run_python("-m", "tools.report", cwd=root))
    print(run_script(root, "import tools.report"))

__main__


tools.report


### Phase 3 complete

You can now explain where an object's memory comes from (frames referencing heap
objects, pymalloc and free lists) and when it is freed (reference counting, the
cyclic GC, immortality), find leaks with `tracemalloc`, read `dis` output down to
specialized instructions and the JIT's micro-ops, transform code with `ast`, and
trace exactly what happens between `import x` and the first line of your code. The
`pyinspect` toolkit packages those skills.

**Next: Phase 4 — Professional Python: Packaging, Typing, Testing and the Standard
Library.** `pyinspect` is a natural first candidate for what Phase 4 teaches: a
`pyproject.toml` with a `[project.scripts]` entry point, type checking with mypy,
and a pytest suite to replace the notebook self-check.